In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

print(os.listdir("/content/drive/MyDrive"))

In [ ]:
zip_path = "/content/drive/MyDrive/air+quality.zip"

In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/air+quality.zip"
extract_path = "/content/air_quality_data"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(os.listdir(extract_path))

In [ ]:
print(os.listdir("/content/air_quality_data"))

In [ ]:
import pandas as pd

df_org = pd.read_excel("/content/air_quality_data/AirQualityUCI.xlsx")
df = df_org.copy()

df.head()

In [ ]:
df.columns

In [ ]:
df.shape

In [ ]:
for i in  df.columns :
  print(i)
  print(df[i].unique())


In [ ]:
df.describe()

In [ ]:
import numpy as np

df = df.replace(-200, np.nan)

In [ ]:
df.isnull().sum()

In [ ]:
import missingno as msno
msno.matrix(df)

In [ ]:
msno.dendrogram(df)

# treating missing values

In [ ]:
df["ds"] = pd.to_datetime(
    df["Date"].astype(str) + " " + df["Time"].astype(str),
    errors="coerce"
)

df = df.drop(columns=["Date", "Time"])
df = df.set_index("ds")
df = df.sort_index()

df.head()

In [ ]:
df.info()

In [ ]:
for col in df.columns:
    if df[col].isna().sum() > 0:
        df[col + "_missing"] = df[col].isna().astype(int)

In [ ]:
df

In [ ]:
print(df.index.min())
print(df.index.max())
print(df.shape)

In [ ]:
missing_percent = df.isna().mean().sort_values(ascending=False) * 100
missing_percent

In [ ]:
for col in df.columns:
    if df[col].isna().sum() > 0:
        df[col + "_missing"] = df[col].isna().astype(int)

df.head()

In [ ]:
df = df.asfreq("h")

df.head()

In [ ]:
print(df.index.min())
print(df.index.max())
print(df.shape)

df.isna().sum()

In [ ]:
missing_indicator_cols = [col for col in df.columns if col.endswith("_missing")]

df[missing_indicator_cols] = df[missing_indicator_cols].fillna(1).astype(int)

In [ ]:
split_idx = int(len(df) * 0.8)

train = df.iloc[:split_idx].copy()
test = df.iloc[split_idx:].copy()

print("Train period:")
print(train.index.min(), "to", train.index.max())

print("\nTest period:")
print(test.index.min(), "to", test.index.max())

print("\nTrain shape:", train.shape)
print("Test shape:", test.shape)

In [ ]:
print("Missing percentage in train:")
print((train.isna().mean() * 100).sort_values(ascending=False))

print("\nMissing percentage in test:")
print((test.isna().mean() * 100).sort_values(ascending=False))

In [ ]:
save_path = "/content/drive/MyDrive/air_quality_checkpoint"

os.makedirs(save_path, exist_ok=True)

df.to_pickle(save_path + "/df_clean.pkl")
train.to_pickle(save_path + "/train_clean.pkl")
test.to_pickle(save_path + "/test_clean.pkl")

print("Checkpoint saved.")

First part of imputation


In [ ]:
# Missingness indicator columns
missing_indicator_cols = [col for col in train.columns if col.endswith("_missing")]

# All real variables, excluding missingness indicators
simple_cols = [
    col for col in train.columns
    if col not in missing_indicator_cols
]

print("Simple imputation columns:")
print(simple_cols)

In [ ]:
advanced_cols = [
    col for col in ["CO(GT)", "T", "NO2(GT)"]
    if col in train.columns
]

print("Advanced imputation columns:")
print(advanced_cols)

In [ ]:
def ffill_impute_train_test(train, test, cols):
    train_imp = train.copy()
    test_imp = test.copy()

    train_medians = train[cols].median()

    # Train: use only previous values inside train
    train_imp[cols] = train_imp[cols].ffill()
    train_imp[cols] = train_imp[cols].fillna(train_medians)

    # Test: allow test to use the last known value from train
    combined = pd.concat([train_imp.tail(1), test_imp])
    combined[cols] = combined[cols].ffill()
    combined[cols] = combined[cols].fillna(train_medians)

    test_imp = combined.iloc[1:].copy()

    return train_imp, test_imp


train_ffill, test_ffill = ffill_impute_train_test(train, test, simple_cols)

print("Forward fill done.")
print("Train missing:", train_ffill[simple_cols].isna().sum().sum())
print("Test missing:", test_ffill[simple_cols].isna().sum().sum())

In [ ]:
def ffill_impute_train_test(train, test, cols):
    train_imp = train.copy()
    test_imp = test.copy()

    train_medians = train[cols].median()

    # Train: use only previous values inside train
    train_imp[cols] = train_imp[cols].ffill()
    train_imp[cols] = train_imp[cols].fillna(train_medians)

    # Test: allow test to use the last known value from train
    combined = pd.concat([train_imp.tail(1), test_imp])
    combined[cols] = combined[cols].ffill()
    combined[cols] = combined[cols].fillna(train_medians)

    test_imp = combined.iloc[1:].copy()

    return train_imp, test_imp


train_ffill, test_ffill = ffill_impute_train_test(train, test, simple_cols)

print("Forward fill done.")
print("Train missing:", train_ffill[simple_cols].isna().sum().sum())
print("Test missing:", test_ffill[simple_cols].isna().sum().sum())

In [ ]:
def time_interpolation_train_test(train, test, cols):
    train_imp = train.copy()
    test_imp = test.copy()

    train_medians = train[cols].median()

    # Train: interpolation inside train only
    train_imp[cols] = train_imp[cols].interpolate(
        method="time",
        limit_direction="both"
    )

    train_imp[cols] = train_imp[cols].fillna(train_medians)

    # Test: no future interpolation, only past values
    combined = pd.concat([train_imp.tail(1), test_imp])
    combined[cols] = combined[cols].ffill()
    combined[cols] = combined[cols].fillna(train_medians)

    test_imp = combined.iloc[1:].copy()

    return train_imp, test_imp


train_interp, test_interp = time_interpolation_train_test(train, test, simple_cols)

print("Time interpolation done.")
print("Train missing:", train_interp[simple_cols].isna().sum().sum())
print("Test missing:", test_interp[simple_cols].isna().sum().sum())

In [ ]:
def fit_gma_statistics(train, cols):
    stats = {}

    helper = pd.DataFrame(index=train.index)
    helper["hour"] = train.index.hour
    helper["dayofweek"] = train.index.dayofweek

    for col in cols:
        temp = pd.concat([train[[col]], helper], axis=1)

        dow_hour_median = temp.groupby(["dayofweek", "hour"])[col].median()
        hour_median = temp.groupby("hour")[col].median()
        global_median = train[col].median()

        stats[col] = {
            "dow_hour_median": dow_hour_median,
            "hour_median": hour_median,
            "global_median": global_median
        }

    return stats


def apply_gma_imputation(data, cols, stats):
    data_imp = data.copy()

    helper = pd.DataFrame(index=data_imp.index)
    helper["hour"] = data_imp.index.hour
    helper["dayofweek"] = data_imp.index.dayofweek

    for col in cols:
        s = stats[col]

        # Same day of week + same hour
        keys = list(zip(helper["dayofweek"], helper["hour"]))

        dow_hour_values = pd.Series(
            [s["dow_hour_median"].get(key, np.nan) for key in keys],
            index=data_imp.index
        )

        data_imp[col] = data_imp[col].fillna(dow_hour_values)

        # Same hour
        hour_values = helper["hour"].map(s["hour_median"])
        data_imp[col] = data_imp[col].fillna(hour_values)

        # Global train median
        data_imp[col] = data_imp[col].fillna(s["global_median"])

    return data_imp


gma_stats = fit_gma_statistics(train, simple_cols)

train_gma = apply_gma_imputation(train, simple_cols, gma_stats)
test_gma = apply_gma_imputation(test, simple_cols, gma_stats)

print("GMA imputation done.")
print("Train missing:", train_gma[simple_cols].isna().sum().sum())
print("Test missing:", test_gma[simple_cols].isna().sum().sum())

In [ ]:
base_train = train_ffill.copy()
base_test = test_ffill.copy()

In [ ]:
from statsmodels.tsa.statespace.structural import UnobservedComponents

def kalman_impute_train_test(train_raw, test_raw, base_train, base_test, cols, seasonal_period=24):
    train_imp = base_train.copy()
    test_imp = base_test.copy()

    train_medians = train_raw[cols].median()

    for col in cols:
        print(f"Kalman imputation for: {col}")

        y_train = train_raw[col].astype(float)

        if y_train.notna().sum() < seasonal_period * 2:
            print(f"Too few observations for {col}. Keeping ffill base.")
            continue

        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")

                model = UnobservedComponents(
                    y_train,
                    level="local level",
                    seasonal=seasonal_period
                )

                result = model.fit(disp=False)

            # Train predictions
            train_pred = result.predict(start=0, end=len(train_raw) - 1)
            train_pred = pd.Series(train_pred, index=train_raw.index)

            # Keep real observed values, replace only missing values
            train_imp[col] = train_raw[col].copy()
            train_imp.loc[train_imp[col].isna(), col] = train_pred[train_imp[col].isna()]
            train_imp[col] = train_imp[col].ffill().fillna(train_medians[col])

            # Test forecast from train-fitted model
            test_forecast = result.forecast(steps=len(test_raw))
            test_forecast = pd.Series(test_forecast, index=test_raw.index)

            test_imp[col] = test_raw[col].copy()
            test_imp.loc[test_imp[col].isna(), col] = test_forecast[test_imp[col].isna()]

            combined = pd.concat([train_imp[[col]].tail(1), test_imp[[col]]])
            combined[col] = combined[col].ffill().fillna(train_medians[col])
            test_imp[col] = combined.iloc[1:][col]

        except Exception as e:
            print(f"Kalman failed for {col}: {e}")
            print("Keeping ffill base.")

    return train_imp, test_imp


train_kalman, test_kalman = kalman_impute_train_test(
    train,
    test,
    base_train,
    base_test,
    advanced_cols,
    seasonal_period=24
)

print("Kalman imputation done.")
print("Train missing:", train_kalman[simple_cols].isna().sum().sum())
print("Test missing:", test_kalman[simple_cols].isna().sum().sum())

In [ ]:
from statsmodels.tsa.statespace.structural import UnobservedComponents

def kalman_impute_train_test(train_raw, test_raw, base_train, base_test, cols, seasonal_period=24):
    train_imp = base_train.copy()
    test_imp = base_test.copy()

    train_medians = train_raw[cols].median()

    for col in cols:
        print(f"Kalman imputation for: {col}")

        y_train = train_raw[col].astype(float)

        if y_train.notna().sum() < seasonal_period * 2:
            print(f"Too few observations for {col}. Keeping ffill base.")
            continue

        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")

                model = UnobservedComponents(
                    y_train,
                    level="local level",
                    seasonal=seasonal_period
                )

                result = model.fit(disp=False)

            # Train predictions
            train_pred = result.predict(start=0, end=len(train_raw) - 1)
            train_pred = pd.Series(train_pred, index=train_raw.index)

            # Keep real observed values, replace only missing values
            train_imp[col] = train_raw[col].copy()
            train_imp.loc[train_imp[col].isna(), col] = train_pred[train_imp[col].isna()]
            train_imp[col] = train_imp[col].ffill().fillna(train_medians[col])

            # Test forecast from train-fitted model
            test_forecast = result.forecast(steps=len(test_raw))
            test_forecast = pd.Series(test_forecast, index=test_raw.index)

            test_imp[col] = test_raw[col].copy()
            test_imp.loc[test_imp[col].isna(), col] = test_forecast[test_imp[col].isna()]

            combined = pd.concat([train_imp[[col]].tail(1), test_imp[[col]]])
            combined[col] = combined[col].ffill().fillna(train_medians[col])
            test_imp[col] = combined.iloc[1:][col]

        except Exception as e:
            print(f"Kalman failed for {col}: {e}")
            print("Keeping ffill base.")

    return train_imp, test_imp


train_kalman, test_kalman = kalman_impute_train_test(
    train,
    test,
    base_train,
    base_test,
    advanced_cols,
    seasonal_period=24
)

print("Kalman imputation done.")
print("Train missing:", train_kalman[simple_cols].isna().sum().sum())
print("Test missing:", test_kalman[simple_cols].isna().sum().sum())

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

def sarima_impute_train_test(
    train_raw,
    test_raw,
    base_train,
    base_test,
    cols,
    order=(1, 0, 1),
    seasonal_order=(1, 0, 1, 24)
):
    train_imp = base_train.copy()
    test_imp = base_test.copy()

    train_medians = train_raw[cols].median()

    for col in cols:
        print(f"SARIMA imputation for: {col}")

        y_train = train_raw[col].astype(float)

        if y_train.notna().sum() < 50:
            print(f"Too few observations for {col}. Keeping ffill base.")
            continue

        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")

                model = SARIMAX(
                    y_train,
                    order=order,
                    seasonal_order=seasonal_order,
                    enforce_stationarity=False,
                    enforce_invertibility=False
                )

                result = model.fit(disp=False, maxiter=50)

            # Train prediction
            train_pred = result.predict(start=0, end=len(train_raw) - 1)
            train_pred = pd.Series(train_pred, index=train_raw.index)

            train_imp[col] = train_raw[col].copy()
            train_imp.loc[train_imp[col].isna(), col] = train_pred[train_imp[col].isna()]
            train_imp[col] = train_imp[col].ffill().fillna(train_medians[col])

            # Test forecast
            test_forecast = result.forecast(steps=len(test_raw))
            test_forecast = pd.Series(test_forecast, index=test_raw.index)

            test_imp[col] = test_raw[col].copy()
            test_imp.loc[test_imp[col].isna(), col] = test_forecast[test_imp[col].isna()]

            combined = pd.concat([train_imp[[col]].tail(1), test_imp[[col]]])
            combined[col] = combined[col].ffill().fillna(train_medians[col])
            test_imp[col] = combined.iloc[1:][col]

        except Exception as e:
            print(f"SARIMA failed for {col}: {e}")
            print("Keeping ffill base.")

    return train_imp, test_imp


train_sarima, test_sarima = sarima_impute_train_test(
    train,
    test,
    base_train,
    base_test,
    advanced_cols,
    order=(1, 0, 1),
    seasonal_order=(1, 0, 1, 24)
)

print("SARIMA imputation done.")
print("Train missing:", train_sarima[simple_cols].isna().sum().sum())
print("Test missing:", test_sarima[simple_cols].isna().sum().sum())

In [ ]:
from prophet import Prophet

def prophet_impute_train_test(train_raw, test_raw, base_train, base_test, cols):
    train_imp = base_train.copy()
    test_imp = base_test.copy()

    train_medians = train_raw[cols].median()

    for col in cols:
        print(f"Prophet imputation for: {col}")

        prophet_train = train_raw[[col]].copy()
        prophet_train = prophet_train.reset_index()
        prophet_train.columns = ["ds", "y"]
        prophet_train = prophet_train.dropna(subset=["y"])

        if len(prophet_train) < 50:
            print(f"Too few observations for {col}. Keeping ffill base.")
            continue

        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")

                model = Prophet(
                    daily_seasonality=True,
                    weekly_seasonality=True,
                    yearly_seasonality=False
                )

                model.fit(prophet_train)

            # Train prediction
            future_train = pd.DataFrame({"ds": train_raw.index})
            forecast_train = model.predict(future_train)

            train_pred = pd.Series(
                forecast_train["yhat"].values,
                index=train_raw.index
            )

            train_imp[col] = train_raw[col].copy()
            train_imp.loc[train_imp[col].isna(), col] = train_pred[train_imp[col].isna()]
            train_imp[col] = train_imp[col].ffill().fillna(train_medians[col])

            # Test prediction from train-fitted model
            future_test = pd.DataFrame({"ds": test_raw.index})
            forecast_test = model.predict(future_test)

            test_pred = pd.Series(
                forecast_test["yhat"].values,
                index=test_raw.index
            )

            test_imp[col] = test_raw[col].copy()
            test_imp.loc[test_imp[col].isna(), col] = test_pred[test_imp[col].isna()]

            combined = pd.concat([train_imp[[col]].tail(1), test_imp[[col]]])
            combined[col] = combined[col].ffill().fillna(train_medians[col])
            test_imp[col] = combined.iloc[1:][col]

        except Exception as e:
            print(f"Prophet failed for {col}: {e}")
            print("Keeping ffill base.")

    return train_imp, test_imp


train_prophet, test_prophet = prophet_impute_train_test(
    train,
    test,
    base_train,
    base_test,
    advanced_cols
)

print("Prophet imputation done.")
print("Train missing:", train_prophet[simple_cols].isna().sum().sum())
print("Test missing:", test_prophet[simple_cols].isna().sum().sum())

In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

def create_iterative_matrix(data, cols):
    X = data[cols].copy()

    # Calendar features
    X["hour"] = data.index.hour
    X["dayofweek"] = data.index.dayofweek
    X["month"] = data.index.month
    X["is_weekend"] = (data.index.dayofweek >= 5).astype(int)

    # Add missingness indicators if they exist
    for col in cols:
        miss_col = col + "_missing"
        if miss_col in data.columns:
            X[miss_col] = data[miss_col]

    X = X.replace([np.inf, -np.inf], np.nan)

    return X


def iterative_impute_train_test(train_raw, test_raw, base_train, base_test, cols):
    train_imp = base_train.copy()
    test_imp = base_test.copy()

    X_train = create_iterative_matrix(train_raw, cols)
    X_test = create_iterative_matrix(test_raw, cols)

    X_test = X_test.reindex(columns=X_train.columns)

    imputer = IterativeImputer(
        estimator=BayesianRidge(),
        max_iter=20,
        initial_strategy="median",
        random_state=42
    )

    X_train_imp = imputer.fit_transform(X_train)
    X_test_imp = imputer.transform(X_test)

    X_train_imp = pd.DataFrame(
        X_train_imp,
        index=train_raw.index,
        columns=X_train.columns
    )

    X_test_imp = pd.DataFrame(
        X_test_imp,
        index=test_raw.index,
        columns=X_test.columns
    )

    # Replace only selected advanced columns
    for col in cols:
        train_imp[col] = X_train_imp[col]
        test_imp[col] = X_test_imp[col]

    return train_imp, test_imp


train_iterative, test_iterative = iterative_impute_train_test(
    train,
    test,
    base_train,
    base_test,
    advanced_cols
)

print("Iterative imputation done.")
print("Train missing:", train_iterative[simple_cols].isna().sum().sum())
print("Test missing:", test_iterative[simple_cols].isna().sum().sum())

In [ ]:
imputed_datasets = {
    "ffill": (train_ffill, test_ffill),
    "interp": (train_interp, test_interp),
    "gma": (train_gma, test_gma),
    "kalman": (train_kalman, test_kalman),
    "sarima": (train_sarima, test_sarima),
    "prophet": (train_prophet, test_prophet),
    "iterative": (train_iterative, test_iterative)
}

for name, (tr, te) in imputed_datasets.items():
    print("\n" + "="*60)
    print(name.upper())
    print("="*60)
    print("Train shape:", tr.shape)
    print("Test shape:", te.shape)
    print("Train missing:", tr[simple_cols].isna().sum().sum())
    print("Test missing:", te[simple_cols].isna().sum().sum())

In [ ]:
import os

save_path = "/content/drive/MyDrive/air_quality_checkpoint"
os.makedirs(save_path, exist_ok=True)

for name, (tr, te) in imputed_datasets.items():
    tr.to_pickle(f"{save_path}/train_{name}.pkl")
    te.to_pickle(f"{save_path}/test_{name}.pkl")

print("All imputed datasets saved.")

# feature enginering

In [ ]:
import numpy as np
import pandas as pd

target = "CO(GT)"

exog_cols = [
    col for col in ["T", "NO2(GT)"]
    if col in train.columns
]

print("Exogenous variables:", exog_cols)


def create_forecasting_features(train_imp, test_imp, target, exog_cols):
    """
    Create forecasting features for time series prediction.

    Important:
    - We concatenate train and test only to calculate lags continuously.
    - All lag/rolling features use shift(1), so they use only past values.
    - The target itself at time t is NOT used as a feature.
    """

    combined = pd.concat([train_imp, test_imp]).sort_index()
    data = combined.copy()

    # ======================================================
    # 1. Calendar features
    # ======================================================
    data["hour"] = data.index.hour
    data["dayofweek"] = data.index.dayofweek
    data["month"] = data.index.month
    data["is_weekend"] = (data.index.dayofweek >= 5).astype(int)

    # ======================================================
    # 2. Cyclical encoding
    # ======================================================
    data["hour_sin"] = np.sin(2 * np.pi * data["hour"] / 24)
    data["hour_cos"] = np.cos(2 * np.pi * data["hour"] / 24)

    data["dow_sin"] = np.sin(2 * np.pi * data["dayofweek"] / 7)
    data["dow_cos"] = np.cos(2 * np.pi * data["dayofweek"] / 7)

    # ======================================================
    # 3. Target lag features
    # ======================================================
    target_lags = [1, 2, 3, 6, 12, 24, 48, 168]

    for lag in target_lags:
        data[f"{target}_lag_{lag}"] = data[target].shift(lag)

    # ======================================================
    # 4. Target rolling features
    # shift(1) avoids using the current target value
    # ======================================================
    data[f"{target}_roll_3_mean"] = data[target].shift(1).rolling(3).mean()
    data[f"{target}_roll_6_mean"] = data[target].shift(1).rolling(6).mean()
    data[f"{target}_roll_24_mean"] = data[target].shift(1).rolling(24).mean()
    data[f"{target}_roll_168_mean"] = data[target].shift(1).rolling(168).mean()

    data[f"{target}_roll_24_std"] = data[target].shift(1).rolling(24).std()
    data[f"{target}_roll_168_std"] = data[target].shift(1).rolling(168).std()

    # ======================================================
    # 5. Exogenous variable lag features
    # ======================================================
    for col in exog_cols:
        for lag in [1, 2, 3, 24, 168]:
            data[f"{col}_lag_{lag}"] = data[col].shift(lag)

        data[f"{col}_roll_24_mean"] = data[col].shift(1).rolling(24).mean()
        data[f"{col}_roll_168_mean"] = data[col].shift(1).rolling(168).mean()

    # ======================================================
    # 6. Split back into train and test
    # ======================================================
    train_feat = data.loc[train_imp.index].copy()
    test_feat = data.loc[test_imp.index].copy()

    return train_feat, test_feat

In [ ]:
featured_datasets = {}

for name, (tr, te) in imputed_datasets.items():
    print(f"Creating features for: {name}")

    train_feat, test_feat = create_forecasting_features(
        train_imp=tr,
        test_imp=te,
        target=target,
        exog_cols=exog_cols
    )

    featured_datasets[name] = (train_feat, test_feat)

print("Feature engineering completed.")

In [ ]:
def prepare_model_data(train_feat, test_feat, target):
    """
    Prepare X_train, y_train, X_test, y_test.

    We train and evaluate only on rows where the target was originally observed.
    """

    target_missing_col = target + "_missing"

    # Use only rows where target was originally real
    train_model = train_feat[train_feat[target_missing_col] == 0].copy()
    test_model = test_feat[test_feat[target_missing_col] == 0].copy()

    # Columns that should not be used as predictors
    forbidden_cols = [
        target,
        target_missing_col
    ]

    feature_cols = [
        col for col in train_feat.columns
        if col not in forbidden_cols
    ]

    # Drop rows where lag/rolling features are still missing
    train_model = train_model.dropna(subset=feature_cols + [target])
    test_model = test_model.dropna(subset=feature_cols + [target])

    X_train = train_model[feature_cols]
    y_train = train_model[target]

    X_test = test_model[feature_cols]
    y_test = test_model[target]

    return X_train, y_train, X_test, y_test, feature_cols

In [ ]:
model_ready_datasets = {}

for name, (train_feat, test_feat) in featured_datasets.items():
    X_train, y_train, X_test, y_test, feature_cols = prepare_model_data(
        train_feat=train_feat,
        test_feat=test_feat,
        target=target
    )

    model_ready_datasets[name] = {
        "X_train": X_train,
        "y_train": y_train,
        "X_test": X_test,
        "y_test": y_test,
        "feature_cols": feature_cols
    }

    print("\n" + "="*60)
    print(name.upper())
    print("="*60)
    print("X_train:", X_train.shape)
    print("y_train:", y_train.shape)
    print("X_test:", X_test.shape)
    print("y_test:", y_test.shape)

In [ ]:
example_name = "ffill"

X_train_example = model_ready_datasets[example_name]["X_train"]

print("Number of features:", X_train_example.shape[1])
print("\nFeatures:")
print(X_train_example.columns.tolist())

In [ ]:
import os

save_path = "/content/drive/MyDrive/air_quality_checkpoint"
os.makedirs(save_path, exist_ok=True)

for name, (train_feat, test_feat) in featured_datasets.items():
    train_feat.to_pickle(f"{save_path}/train_features_{name}.pkl")
    test_feat.to_pickle(f"{save_path}/test_features_{name}.pkl")

print("Feature datasets saved.")

# baseline

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

def smape(y_true, y_pred):
    """
    Symmetric MAPE.
    Better than MAPE when values can be close to zero.
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    denominator = np.where(denominator == 0, 1e-8, denominator)

    return np.mean(np.abs(y_true - y_pred) / denominator) * 100


def evaluate_forecast(y_true, y_pred):
    """
    Return common forecasting metrics.
    """
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    s_mape = smape(y_true, y_pred)

    return {
        "MAE": mae,
        "RMSE": rmse,
        "sMAPE": s_mape
    }

In [ ]:
target = "CO(GT)"

baseline_results = []

for name, data in model_ready_datasets.items():

    X_train = data["X_train"]
    y_train = data["y_train"]
    X_test = data["X_test"]
    y_test = data["y_test"]

    print(f"Running baselines for: {name}")

    # Dictionary of baseline predictions
    baselines = {}

    # 1. Mean baseline
    baselines["mean_train"] = np.repeat(y_train.mean(), len(y_test))

    # 2. Naive lag 1
    lag_1_col = f"{target}_lag_1"
    if lag_1_col in X_test.columns:
        baselines["naive_lag_1"] = X_test[lag_1_col].values

    # 3. Daily seasonal naive
    lag_24_col = f"{target}_lag_24"
    if lag_24_col in X_test.columns:
        baselines["daily_naive_lag_24"] = X_test[lag_24_col].values

    # 4. Weekly seasonal naive
    lag_168_col = f"{target}_lag_168"
    if lag_168_col in X_test.columns:
        baselines["weekly_naive_lag_168"] = X_test[lag_168_col].values

    # 5. Rolling mean 24h
    roll_24_col = f"{target}_roll_24_mean"
    if roll_24_col in X_test.columns:
        baselines["rolling_mean_24"] = X_test[roll_24_col].values

    # 6. Rolling mean 168h
    roll_168_col = f"{target}_roll_168_mean"
    if roll_168_col in X_test.columns:
        baselines["rolling_mean_168"] = X_test[roll_168_col].values

    # Evaluate each baseline
    for baseline_name, y_pred in baselines.items():
        metrics = evaluate_forecast(y_test, y_pred)

        baseline_results.append({
            "imputation": name,
            "model": baseline_name,
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"],
            "sMAPE": metrics["sMAPE"]
        })

baseline_results_df = pd.DataFrame(baseline_results)

baseline_results_df

In [ ]:
baseline_results_df.sort_values("RMSE")

In [ ]:
baseline_results_df.sort_values("MAE")

In [ ]:
best_baseline_by_imputation = (
    baseline_results_df
    .sort_values("RMSE")
    .groupby("imputation")
    .first()
    .reset_index()
)

best_baseline_by_imputation

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(14, 6))

sns.barplot(
    data=baseline_results_df.sort_values("RMSE"),
    x="RMSE",
    y="model",
    hue="imputation"
)

plt.title("Baseline models comparison by RMSE")
plt.xlabel("RMSE")
plt.ylabel("Baseline model")
plt.legend(title="Imputation method", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.show()

start with ml models

In [ ]:
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

def smape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    denominator = np.where(denominator == 0, 1e-8, denominator)

    return np.mean(np.abs(y_true - y_pred) / denominator) * 100


def evaluate_forecast(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    s_mape = smape(y_true, y_pred)

    return mae, rmse, s_mape

In [ ]:
try:
    from xgboost import XGBRegressor
    xgboost_available = True
except:
    xgboost_available = False
    print("XGBoost not available. Install with: !pip install xgboost")

In [ ]:
ml_results = []

for name, data in model_ready_datasets.items():

    print(f"\nTraining ML models for imputation method: {name}")

    X_train = data["X_train"]
    y_train = data["y_train"]
    X_test = data["X_test"]
    y_test = data["y_test"]

    models = {
        "RandomForest": RandomForestRegressor(
            n_estimators=200,
            max_depth=12,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1
        ),

        "HistGradientBoosting": HistGradientBoostingRegressor(
            max_iter=300,
            learning_rate=0.05,
            max_leaf_nodes=31,
            random_state=42
        )
    }

    if xgboost_available:
        models["XGBoost"] = XGBRegressor(
            n_estimators=500,
            learning_rate=0.03,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        )

    for model_name, model in models.items():

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        mae, rmse, s_mape = evaluate_forecast(y_test, y_pred)

        ml_results.append({
            "imputation": name,
            "model": model_name,
            "MAE": mae,
            "RMSE": rmse,
            "sMAPE": s_mape
        })

ml_results_df = pd.DataFrame(ml_results)

ml_results_df.sort_values("RMSE")

In [ ]:
best_baseline_rmse = baseline_results_df["RMSE"].min()
best_baseline = baseline_results_df.sort_values("RMSE").iloc[0]

print("Best baseline:")
print(best_baseline)

print("\nBest ML model:")
print(ml_results_df.sort_values("RMSE").iloc[0])

print("\nBaseline RMSE to beat:", best_baseline_rmse)

are we forcasting or undercasting


In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import matplotlib.pyplot as plt

best_imputation = "gma"
best_model_name = "HistGradientBoosting"

X_train = model_ready_datasets[best_imputation]["X_train"]
y_train = model_ready_datasets[best_imputation]["y_train"]
X_test = model_ready_datasets[best_imputation]["X_test"]
y_test = model_ready_datasets[best_imputation]["y_test"]

best_model = HistGradientBoostingRegressor(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    random_state=42
)

best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("Best model:", best_model_name)
print("Best imputation:", best_imputation)
print("MAE:", mae)
print("RMSE:", rmse)

In [ ]:
plt.figure(figsize=(16, 5))

plt.plot(y_test.index, y_test.values, label="Actual CO(GT)", alpha=0.8)
plt.plot(y_test.index, y_pred, label="Predicted CO(GT)", alpha=0.8)

plt.title("Actual vs Predicted CO(GT) — GMA + HistGradientBoosting")
plt.xlabel("Date")
plt.ylabel("CO(GT)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(16, 4))
plt.plot(y_test.index, residuals)
plt.axhline(0, color="black", linestyle="--")
plt.title("Residuals over time — Best model")
plt.xlabel("Date")
plt.ylabel("Error")
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(residuals, bins=40)
plt.title("Distribution of residuals")
plt.xlabel("Prediction error")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring="neg_root_mean_squared_error"
)

feature_importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance": perm.importances_mean
}).sort_values("importance", ascending=False)

feature_importance.head(20)

# try some variables


In [ ]:
def prepare_strict_forecasting_data(train_feat, test_feat, target):
    target_missing_col = target + "_missing"

    # Keep only rows where target was originally observed
    train_model = train_feat[train_feat[target_missing_col] == 0].copy()
    test_model = test_feat[test_feat[target_missing_col] == 0].copy()

    forbidden_cols = [
        target,
        target_missing_col
    ]

    feature_cols = []

    for col in train_feat.columns:
        if col in forbidden_cols:
            continue

        # Keep calendar features
        if col in ["hour", "dayofweek", "month", "is_weekend", "hour_sin", "hour_cos", "dow_sin", "dow_cos"]:
            feature_cols.append(col)

        # Keep lag and rolling features
        elif "_lag_" in col or "_roll_" in col:
            feature_cols.append(col)

        # Keep missingness indicators for explanatory variables
        elif col.endswith("_missing") and col != target_missing_col:
            feature_cols.append(col)

        # Remove current same-time variables like C6H6(GT), NOx(GT), NO2(GT), T
        else:
            pass

    train_model = train_model.dropna(subset=feature_cols + [target])
    test_model = test_model.dropna(subset=feature_cols + [target])

    X_train = train_model[feature_cols]
    y_train = train_model[target]

    X_test = test_model[feature_cols]
    y_test = test_model[target]

    return X_train, y_train, X_test, y_test, feature_cols

In [ ]:
strict_model_ready_datasets = {}

for name, (train_feat, test_feat) in featured_datasets.items():
    X_train, y_train, X_test, y_test, feature_cols = prepare_strict_forecasting_data(
        train_feat,
        test_feat,
        target
    )

    strict_model_ready_datasets[name] = {
        "X_train": X_train,
        "y_train": y_train,
        "X_test": X_test,
        "y_test": y_test,
        "feature_cols": feature_cols
    }

    print("\n" + "="*60)
    print(name.upper())
    print("="*60)
    print("X_train:", X_train.shape)
    print("X_test:", X_test.shape)
    print("Number of features:", len(feature_cols))

other models

In [ ]:
# ============================================================
# 1. CHOOSE DATASET VERSION
# ============================================================

try:
    DATASETS = strict_model_ready_datasets
    print("Using strict forecasting datasets.")
except NameError:
    DATASETS = model_ready_datasets
    print("Using standard model-ready datasets.")

target = "CO(GT)"

In [ ]:
# ============================================================
# 2. METRICS
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def smape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    denominator = np.where(denominator == 0, 1e-8, denominator)

    return np.mean(np.abs(y_true - y_pred) / denominator) * 100


def evaluate_forecast(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    s_mape = smape(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return mae, rmse, s_mape, r2

In [ ]:
# ============================================================
# 3. IMPORT MODELS
# ============================================================

import warnings
warnings.filterwarnings("ignore")

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    AdaBoostRegressor
)
from sklearn.neural_network import MLPRegressor

# Optional models
try:
    from xgboost import XGBRegressor
    xgboost_available = True
except:
    xgboost_available = False
    print("XGBoost not available.")

try:
    from lightgbm import LGBMRegressor
    lightgbm_available = True
except:
    lightgbm_available = False
    print("LightGBM not available. Install with: !pip install lightgbm")

try:
    from catboost import CatBoostRegressor
    catboost_available = True
except:
    catboost_available = False
    print("CatBoost not available. Install with: !pip install catboost")

In [ ]:
!pip install catboost

In [ ]:
# ============================================================
# 4. MODEL LIST
# ============================================================

def get_models():
    models = {}

    # Linear models
    models["LinearRegression"] = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ])

    models["Ridge"] = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0))
    ])

    models["Lasso"] = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Lasso(alpha=0.001, max_iter=5000))
    ])

    models["ElasticNet"] = Pipeline([
        ("scaler", StandardScaler()),
        ("model", ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=5000))
    ])

    # Distance / kernel models
    models["KNN"] = Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsRegressor(n_neighbors=10, weights="distance"))
    ])

    models["SVR_RBF"] = Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVR(kernel="rbf", C=10, epsilon=0.1))
    ])

    # Tree models
    models["DecisionTree"] = DecisionTreeRegressor(
        max_depth=10,
        min_samples_leaf=10,
        random_state=42
    )

    models["RandomForest"] = RandomForestRegressor(
        n_estimators=300,
        max_depth=14,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )

    models["ExtraTrees"] = ExtraTreesRegressor(
        n_estimators=300,
        max_depth=14,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )

    models["GradientBoosting"] = GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )

    models["HistGradientBoosting"] = HistGradientBoostingRegressor(
        max_iter=400,
        learning_rate=0.04,
        max_leaf_nodes=31,
        random_state=42
    )

    models["AdaBoost"] = AdaBoostRegressor(
        n_estimators=200,
        learning_rate=0.05,
        random_state=42
    )

    # Neural network
    models["MLP"] = Pipeline([
        ("scaler", StandardScaler()),
        ("model", MLPRegressor(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            learning_rate_init=0.001,
            max_iter=300,
            random_state=42
        ))
    ])

    # Optional boosting models
    if xgboost_available:
        models["XGBoost"] = XGBRegressor(
            n_estimators=600,
            learning_rate=0.03,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        )

    if lightgbm_available:
        models["LightGBM"] = LGBMRegressor(
            n_estimators=600,
            learning_rate=0.03,
            max_depth=-1,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1
        )

    if catboost_available:
        models["CatBoost"] = CatBoostRegressor(
            iterations=600,
            learning_rate=0.03,
            depth=6,
            loss_function="RMSE",
            random_seed=42,
            verbose=False
        )

    return models

In [ ]:
# ============================================================
# 5. TRAIN ALL MODELS
# ============================================================

ml_results = []
trained_models = {}

for imputation_name, data in DATASETS.items():

    print("\n" + "="*80)
    print(f"Imputation method: {imputation_name}")
    print("="*80)

    X_train = data["X_train"]
    y_train = data["y_train"]
    X_test = data["X_test"]
    y_test = data["y_test"]

    models = get_models()

    for model_name, model in models.items():
        print(f"Training {model_name}...")

        try:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            mae, rmse, s_mape, r2 = evaluate_forecast(y_test, y_pred)

            ml_results.append({
                "imputation": imputation_name,
                "model": model_name,
                "MAE": mae,
                "RMSE": rmse,
                "sMAPE": s_mape,
                "R2": r2
            })

            trained_models[(imputation_name, model_name)] = model

        except Exception as e:
            print(f"{model_name} failed on {imputation_name}. Error: {e}")

ml_results_df = pd.DataFrame(ml_results)

ml_results_df.sort_values("RMSE")

# Now casting


In [ ]:
target = "CO(GT)"

def prepare_nowcasting_data(train_feat, test_feat, target):
    """
    Prepare data for nowcasting.

    Nowcasting allows current-time explanatory variables:
    - C6H6(GT)
    - NOx(GT)
    - NO2(GT)
    - T
    - RH
    - AH
    - sensor variables
    - lags
    - rolling features
    - calendar features

    But it does NOT allow CO(GT) itself as a feature.
    """

    target_missing_col = target + "_missing"

    # Keep only rows where target was originally observed
    train_model = train_feat[train_feat[target_missing_col] == 0].copy()
    test_model = test_feat[test_feat[target_missing_col] == 0].copy()

    # Do not use the target itself or its missingness indicator
    forbidden_cols = [
        target,
        target_missing_col
    ]

    feature_cols = [
        col for col in train_feat.columns
        if col not in forbidden_cols
    ]

    # Drop rows with remaining missing values
    train_model = train_model.dropna(subset=feature_cols + [target])
    test_model = test_model.dropna(subset=feature_cols + [target])

    X_train = train_model[feature_cols]
    y_train = train_model[target]

    X_test = test_model[feature_cols]
    y_test = test_model[target]

    return X_train, y_train, X_test, y_test, feature_cols

In [ ]:
nowcasting_datasets = {}

for name, (train_feat, test_feat) in featured_datasets.items():

    X_train, y_train, X_test, y_test, feature_cols = prepare_nowcasting_data(
        train_feat=train_feat,
        test_feat=test_feat,
        target=target
    )

    nowcasting_datasets[name] = {
        "X_train": X_train,
        "y_train": y_train,
        "X_test": X_test,
        "y_test": y_test,
        "feature_cols": feature_cols
    }

    print("\n" + "="*60)
    print(name.upper())
    print("="*60)
    print("X_train:", X_train.shape)
    print("X_test:", X_test.shape)
    print("Number of features:", len(feature_cols))

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
    GradientBoostingRegressor
)
import numpy as np
import pandas as pd

def smape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    denominator = np.where(denominator == 0, 1e-8, denominator)

    return np.mean(np.abs(y_true - y_pred) / denominator) * 100


def evaluate_forecast(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    s_mape = smape(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return mae, rmse, s_mape, r2

In [ ]:
try:
    from xgboost import XGBRegressor
    xgboost_available = True
except:
    xgboost_available = False
    print("XGBoost not available.")

try:
    from lightgbm import LGBMRegressor
    lightgbm_available = True
except:
    lightgbm_available = False
    print("LightGBM not available.")

In [ ]:
nowcasting_results = []
nowcasting_trained_models = {}

for imputation_name, data in nowcasting_datasets.items():

    print("\n" + "="*80)
    print("Nowcasting with imputation:", imputation_name)
    print("="*80)

    X_train = data["X_train"]
    y_train = data["y_train"]
    X_test = data["X_test"]
    y_test = data["y_test"]

    models = {
        "RandomForest": RandomForestRegressor(
            n_estimators=300,
            max_depth=14,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1
        ),

        "ExtraTrees": ExtraTreesRegressor(
            n_estimators=300,
            max_depth=14,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1
        ),

        "HistGradientBoosting": HistGradientBoostingRegressor(
            max_iter=400,
            learning_rate=0.04,
            max_leaf_nodes=31,
            random_state=42
        ),

        "GradientBoosting": GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=3,
            random_state=42
        )
    }

    if xgboost_available:
        models["XGBoost"] = XGBRegressor(
            n_estimators=600,
            learning_rate=0.03,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        )

    if lightgbm_available:
        models["LightGBM"] = LGBMRegressor(
            n_estimators=600,
            learning_rate=0.03,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1
        )

    for model_name, model in models.items():

        print("Training:", model_name)

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        mae, rmse, s_mape, r2 = evaluate_forecast(y_test, y_pred)

        nowcasting_results.append({
            "imputation": imputation_name,
            "model": model_name,
            "MAE": mae,
            "RMSE": rmse,
            "sMAPE": s_mape,
            "R2": r2
        })

        nowcasting_trained_models[(imputation_name, model_name)] = model

nowcasting_results_df = pd.DataFrame(nowcasting_results)

nowcasting_results_df.sort_values("RMSE").head(20)

In [ ]:
best_imp = "gma"

X_train_full = nowcasting_datasets[best_imp]["X_train"]
y_train_full = nowcasting_datasets[best_imp]["y_train"]

X_test = nowcasting_datasets[best_imp]["X_test"]
y_test = nowcasting_datasets[best_imp]["y_test"]

print(X_train_full.shape, X_test.shape)

In [ ]:
split_idx = int(len(X_train_full) * 0.8)

X_train_tune = X_train_full.iloc[:split_idx].copy()
y_train_tune = y_train_full.iloc[:split_idx].copy()

X_val = X_train_full.iloc[split_idx:].copy()
y_val = y_train_full.iloc[split_idx:].copy()

print("Train tuning:", X_train_tune.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterGrid
import numpy as np
import pandas as pd

def smape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    denominator = np.where(denominator == 0, 1e-8, denominator)
    return np.mean(np.abs(y_true - y_pred) / denominator) * 100

def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    s_mape = smape(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, s_mape, r2

In [ ]:
hgb_grid = {
    "max_iter": [300, 500, 800],
    "learning_rate": [0.02, 0.03, 0.04, 0.05],
    "max_leaf_nodes": [15, 31, 63],
    "min_samples_leaf": [10, 20, 30, 50],
    "l2_regularization": [0.0, 0.01, 0.1, 1.0]
}

hgb_results = []
best_hgb_model = None
best_hgb_rmse = np.inf
best_hgb_params = None

for params in ParameterGrid(hgb_grid):
    model = HistGradientBoostingRegressor(
        **params,
        random_state=42
    )

    model.fit(X_train_tune, y_train_tune)
    val_pred = model.predict(X_val)

    mae, rmse, s_mape, r2 = evaluate(y_val, val_pred)

    hgb_results.append({
        **params,
        "MAE": mae,
        "RMSE": rmse,
        "sMAPE": s_mape,
        "R2": r2
    })

    if rmse < best_hgb_rmse:
        best_hgb_rmse = rmse
        best_hgb_model = model
        best_hgb_params = params

hgb_results_df = pd.DataFrame(hgb_results).sort_values("RMSE")

print("Best validation RMSE:", best_hgb_rmse)
print("Best params:", best_hgb_params)

hgb_results_df.head(10)

# deep learning model

In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

target = "CO(GT)"
best_imp = "gma"

X_train_full = nowcasting_datasets[best_imp]["X_train"].copy()
y_train_full = nowcasting_datasets[best_imp]["y_train"].copy()

X_test = nowcasting_datasets[best_imp]["X_test"].copy()
y_test = nowcasting_datasets[best_imp]["y_test"].copy()

# Safety cleaning
X_train_full = X_train_full.replace([np.inf, -np.inf], np.nan).ffill().bfill().fillna(0)
X_test = X_test.replace([np.inf, -np.inf], np.nan).ffill().bfill().fillna(0)

print("Train:", X_train_full.shape)
print("Test:", X_test.shape)

In [ ]:
def smape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    denominator = np.where(denominator == 0, 1e-8, denominator)

    return np.mean(np.abs(y_true - y_pred) / denominator) * 100


def evaluate_model(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    s_mape = smape(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return {
        "MAE": mae,
        "RMSE": rmse,
        "sMAPE": s_mape,
        "R2": r2
    }

In [ ]:
split_idx = int(len(X_train_full) * 0.8)

X_train = X_train_full.iloc[:split_idx].copy()
y_train = y_train_full.iloc[:split_idx].copy()

X_val = X_train_full.iloc[split_idx:].copy()
y_val = y_train_full.iloc[split_idx:].copy()

print("Train tune:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train_scaled = x_scaler.fit_transform(X_train)
X_val_scaled = x_scaler.transform(X_val)
X_test_scaled = x_scaler.transform(X_test)

y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).ravel()
y_val_scaled = y_scaler.transform(y_val.values.reshape(-1, 1)).ravel()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

batch_size = 128

train_ds = TensorDataset(
    torch.tensor(X_train_scaled, dtype=torch.float32),
    torch.tensor(y_train_scaled, dtype=torch.float32)
)

val_ds = TensorDataset(
    torch.tensor(X_val_scaled, dtype=torch.float32),
    torch.tensor(y_val_scaled, dtype=torch.float32)
)

test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

In [ ]:
class FTTransformerRegressor(nn.Module):
    def __init__(
        self,
        n_features,
        d_token=64,
        n_heads=4,
        n_layers=3,
        dropout=0.1
    ):
        super().__init__()

        self.n_features = n_features
        self.d_token = d_token

        # Each numerical feature becomes one token
        self.weight = nn.Parameter(torch.randn(n_features, d_token) * 0.02)
        self.bias = nn.Parameter(torch.zeros(n_features, d_token))

        # CLS token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_token))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_token,
            nhead=n_heads,
            dim_feedforward=d_token * 4,
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_layers
        )

        self.head = nn.Sequential(
            nn.LayerNorm(d_token),
            nn.Linear(d_token, d_token),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_token, 1)
        )

    def forward(self, x):
        # x shape: batch_size × n_features
        tokens = x.unsqueeze(-1) * self.weight.unsqueeze(0) + self.bias.unsqueeze(0)

        cls = self.cls_token.expand(x.size(0), -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)

        encoded = self.transformer(tokens)

        cls_output = encoded[:, 0, :]

        return self.head(cls_output).squeeze(-1)

In [ ]:
def train_ft_transformer(
    model,
    train_loader,
    val_loader,
    epochs=80,
    lr=1e-3,
    patience=10
):
    model = model.to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=1e-4
    )

    loss_fn = nn.MSELoss()

    best_val_loss = np.inf
    best_state = None
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        train_losses = []

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        model.eval()
        val_losses = []

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)

                pred = model(xb)
                loss = loss_fn(pred, yb)
                val_losses.append(loss.item())

        train_loss = np.mean(train_losses)
        val_loss = np.mean(val_losses)

        if epoch % 5 == 0:
            print(f"Epoch {epoch} | Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print("Early stopping.")
            break

    model.load_state_dict(best_state)

    return model

In [ ]:
ft_model = FTTransformerRegressor(
    n_features=X_train.shape[1],
    d_token=64,
    n_heads=4,
    n_layers=3,
    dropout=0.1
)

ft_model = train_ft_transformer(
    ft_model,
    train_loader,
    val_loader,
    epochs=80,
    lr=1e-3,
    patience=10
)

In [ ]:
ft_model.eval()

with torch.no_grad():
    pred_scaled = ft_model(test_tensor.to(device)).cpu().numpy()

ft_pred = y_scaler.inverse_transform(pred_scaled.reshape(-1, 1)).ravel()

ft_metrics = evaluate_model(y_test, ft_pred)

print("FT-Transformer test results:")
print(ft_metrics)

Hybrid model: BiLSTM embeddings + GBDT

In [ ]:
seq_len = 48  # last 48 hours

# Combine train and test GMA-imputed data
seq_source = pd.concat([train_gma, test_gma]).sort_index().copy()

# Safe target history: previous CO only, not current CO
seq_source[f"{target}_lag_safe"] = seq_source[target].shift(1)

# Use real variables, excluding missingness indicators and excluding current target
seq_cols = [
    col for col in seq_source.columns
    if (not col.endswith("_missing")) and (col != target)
]

# Add safe target lag
if f"{target}_lag_safe" not in seq_cols:
    seq_cols.append(f"{target}_lag_safe")

# Clean
seq_source[seq_cols] = (
    seq_source[seq_cols]
    .replace([np.inf, -np.inf], np.nan)
    .ffill()
    .bfill()
    .fillna(0)
)

print("Sequence columns:")
print(seq_cols)
print("Number of sequence variables:", len(seq_cols))

In [ ]:
train_end = train_gma.index.max()

seq_mean = seq_source.loc[seq_source.index <= train_end, seq_cols].mean()
seq_std = seq_source.loc[seq_source.index <= train_end, seq_cols].std().replace(0, 1)

seq_source_scaled = seq_source.copy()
seq_source_scaled[seq_cols] = (seq_source_scaled[seq_cols] - seq_mean) / seq_std

In [ ]:
position_map = pd.Series(
    np.arange(len(seq_source_scaled)),
    index=seq_source_scaled.index
)

seq_array = seq_source_scaled[seq_cols].values


def make_sequence_windows(indices, seq_len):
    X_seq = []
    valid_indices = []

    for idx in indices:
        if idx not in position_map.index:
            continue

        pos = position_map.loc[idx]

        if pos < seq_len - 1:
            continue

        window = seq_array[pos - seq_len + 1 : pos + 1]

        X_seq.append(window)
        valid_indices.append(idx)

    return np.array(X_seq), pd.Index(valid_indices)

In [ ]:
Xseq_train_full, idx_train_valid = make_sequence_windows(X_train_full.index, seq_len)
Xseq_test, idx_test_valid = make_sequence_windows(X_test.index, seq_len)

Xtab_train_full = X_train_full.loc[idx_train_valid].copy()
ytab_train_full = y_train_full.loc[idx_train_valid].copy()

Xtab_test = X_test.loc[idx_test_valid].copy()
ytab_test = y_test.loc[idx_test_valid].copy()

print("Sequence train:", Xseq_train_full.shape)
print("Tabular train:", Xtab_train_full.shape)
print("Sequence test:", Xseq_test.shape)
print("Tabular test:", Xtab_test.shape)

In [ ]:
split_idx = int(len(Xseq_train_full) * 0.8)

Xseq_train = Xseq_train_full[:split_idx]
Xseq_val = Xseq_train_full[split_idx:]

yseq_train = ytab_train_full.iloc[:split_idx]
yseq_val = ytab_train_full.iloc[split_idx:]

In [ ]:
y_seq_scaler = StandardScaler()

yseq_train_scaled = y_seq_scaler.fit_transform(yseq_train.values.reshape(-1, 1)).ravel()
yseq_val_scaled = y_seq_scaler.transform(yseq_val.values.reshape(-1, 1)).ravel()

In [ ]:
train_seq_ds = TensorDataset(
    torch.tensor(Xseq_train, dtype=torch.float32),
    torch.tensor(yseq_train_scaled, dtype=torch.float32)
)

val_seq_ds = TensorDataset(
    torch.tensor(Xseq_val, dtype=torch.float32),
    torch.tensor(yseq_val_scaled, dtype=torch.float32)
)

train_seq_loader = DataLoader(train_seq_ds, batch_size=128, shuffle=True)
val_seq_loader = DataLoader(val_seq_ds, batch_size=128, shuffle=False)

In [ ]:
class BiLSTMRegressor(nn.Module):
    def __init__(self, n_features, hidden_size=64, num_layers=1, dropout=0.1):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0 if num_layers == 1 else dropout
        )

        self.head = nn.Sequential(
            nn.Linear(hidden_size * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x, return_embedding=False):
        output, (h, c) = self.lstm(x)

        # Last layer forward and backward hidden states
        embedding = torch.cat([h[-2], h[-1]], dim=1)

        pred = self.head(embedding).squeeze(-1)

        if return_embedding:
            return pred, embedding

        return pred

In [ ]:
def train_bilstm(model, train_loader, val_loader, epochs=60, lr=1e-3, patience=8):
    model = model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.MSELoss()

    best_val_loss = np.inf
    best_state = None
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        train_losses = []

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        model.eval()
        val_losses = []

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)

                pred = model(xb)
                loss = loss_fn(pred, yb)
                val_losses.append(loss.item())

        train_loss = np.mean(train_losses)
        val_loss = np.mean(val_losses)

        if epoch % 5 == 0:
            print(f"Epoch {epoch} | Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print("Early stopping.")
            break

    model.load_state_dict(best_state)

    return model

In [ ]:
bilstm_model = BiLSTMRegressor(
    n_features=Xseq_train_full.shape[2],
    hidden_size=64,
    num_layers=1,
    dropout=0.1
)

bilstm_model = train_bilstm(
    bilstm_model,
    train_seq_loader,
    val_seq_loader,
    epochs=60,
    lr=1e-3,
    patience=8
)

In [ ]:
def extract_embeddings(model, Xseq, batch_size=256):
    model.eval()

    loader = DataLoader(
        torch.tensor(Xseq, dtype=torch.float32),
        batch_size=batch_size,
        shuffle=False
    )

    embeddings = []

    with torch.no_grad():
        for xb in loader:
            xb = xb.to(device)
            _, emb = model(xb, return_embedding=True)
            embeddings.append(emb.cpu().numpy())

    return np.vstack(embeddings)

In [ ]:
emb_train_full = extract_embeddings(bilstm_model, Xseq_train_full)
emb_test = extract_embeddings(bilstm_model, Xseq_test)

print("Embeddings train:", emb_train_full.shape)
print("Embeddings test:", emb_test.shape)

In [ ]:
emb_cols = [f"bilstm_emb_{i}" for i in range(emb_train_full.shape[1])]

emb_train_df = pd.DataFrame(
    emb_train_full,
    index=Xtab_train_full.index,
    columns=emb_cols
)

emb_test_df = pd.DataFrame(
    emb_test,
    index=Xtab_test.index,
    columns=emb_cols
)

X_hybrid_train = pd.concat([Xtab_train_full, emb_train_df], axis=1)
X_hybrid_test = pd.concat([Xtab_test, emb_test_df], axis=1)

y_hybrid_train = ytab_train_full.copy()
y_hybrid_test = ytab_test.copy()

print("Hybrid train:", X_hybrid_train.shape)
print("Hybrid test:", X_hybrid_test.shape)

In [ ]:
try:
    from lightgbm import LGBMRegressor
    hybrid_model = LGBMRegressor(
        n_estimators=800,
        learning_rate=0.03,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )
    print("Using LightGBM for hybrid model.")
except:
    from sklearn.ensemble import ExtraTreesRegressor
    hybrid_model = ExtraTreesRegressor(
        n_estimators=600,
        max_depth=None,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
    print("Using ExtraTrees for hybrid model.")

In [ ]:
hybrid_model.fit(X_hybrid_train, y_hybrid_train)

hybrid_pred = hybrid_model.predict(X_hybrid_test)

hybrid_metrics = evaluate_model(y_hybrid_test, hybrid_pred)

print("Hybrid BiLSTM + GBDT test results:")
print(hybrid_metrics)

Temporal Fusion Transformer

In [ ]:
!pip -q install pytorch-forecasting lightning

In [ ]:
import torch
import pandas as pd
import numpy as np

from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.metrics import RMSE
from pytorch_forecasting.data import GroupNormalizer

from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import EarlyStopping

In [ ]:
train_feat_gma, test_feat_gma = featured_datasets["gma"]

data_tft = pd.concat([train_feat_gma, test_feat_gma]).sort_index().copy()

data_tft["time_idx"] = np.arange(len(data_tft))
data_tft["series_id"] = "air_quality"

# Clean numeric columns
for col in data_tft.columns:
    if col != "series_id":
        data_tft[col] = pd.to_numeric(data_tft[col], errors="coerce")

data_tft = data_tft.replace([np.inf, -np.inf], np.nan)
data_tft = data_tft.ffill().bfill().fillna(0)

train_cutoff = len(train_feat_gma) - 1

print(data_tft.shape)
print("Train cutoff:", train_cutoff)

In [ ]:
target_missing_col = target + "_missing"

feature_cols = [
    col for col in nowcasting_datasets["gma"]["feature_cols"]
    if col in data_tft.columns
]

# TFT known real variables:
# These are features available at prediction time in nowcasting.
known_reals = feature_cols + ["time_idx"]

# Remove duplicates
known_reals = list(dict.fromkeys(known_reals))

print("Number of TFT known real features:", len(known_reals))

In [ ]:
max_encoder_length = 48
max_prediction_length = 1

# Rename columns to remove '.' characters
data_tft.columns = [col.replace('.', '_') for col in data_tft.columns]
known_reals = [col.replace('.', '_') for col in known_reals]
target = target.replace('.', '_')

training_tft = TimeSeriesDataSet(
    data_tft[data_tft["time_idx"] <= train_cutoff],
    time_idx="time_idx",
    target=target,
    group_ids=["series_id"],

    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,

    time_varying_known_reals=known_reals,
    time_varying_unknown_reals=[target],

    target_normalizer=GroupNormalizer(groups=["series_id"]),

    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True
)

validation_tft = TimeSeriesDataSet.from_dataset(
    training_tft,
    data_tft,
    min_prediction_idx=train_cutoff + 1,
    stop_randomization=True
)

batch_size = 64

train_loader_tft = training_tft.to_dataloader(
    train=True,
    batch_size=batch_size,
    num_workers=0
)

val_loader_tft = validation_tft.to_dataloader(
    train=False,
    batch_size=batch_size,
    num_workers=0
)

In [ ]:
tft = TemporalFusionTransformer.from_dataset(
    training_tft,
    learning_rate=0.03,
    hidden_size=32,
    attention_head_size=4,
    dropout=0.1,
    hidden_continuous_size=16,
    loss=RMSE(),
    log_interval=-1,
    reduce_on_plateau_patience=4
)

early_stop_callback = EarlyStopping(
    monitor="val_loss",
    min_delta=1e-4,
    patience=5,
    verbose=True,
    mode="min"
)

trainer = Trainer(
    max_epochs=25,
    accelerator="auto",
    gradient_clip_val=0.1,
    callbacks=[early_stop_callback],
    enable_checkpointing=False,
    logger=False
)

trainer.fit(
    tft,
    train_dataloaders=train_loader_tft,
    val_dataloaders=val_loader_tft
)

In [ ]:
tft_pred = tft.predict(val_loader_tft)
tft_pred = np.array(tft_pred).reshape(-1)

# Align with test part
test_part = data_tft[data_tft["time_idx"] > train_cutoff].copy()

# In case prediction length differs slightly
test_part = test_part.iloc[-len(tft_pred):].copy()

mask_real_target = test_part[target_missing_col].values == 0

y_true_tft = test_part[target].values
y_pred_tft = tft_pred

tft_metrics = evaluate_model(
    y_true_tft[mask_real_target],
    y_pred_tft[mask_real_target]
)

print("TFT test results:")
print(tft_metrics)

In [ ]:
advanced_results = []

advanced_results.append({
    "model": "FT-Transformer",
    **ft_metrics
})

advanced_results.append({
    "model": "BiLSTM embeddings + GBDT",
    **hybrid_metrics
})

advanced_results.append({
    "model": "TFT",
    **tft_metrics
})

advanced_results_df = pd.DataFrame(advanced_results).sort_values("RMSE")

advanced_results_df

In [ ]:
current_best_rmse = 0.485091  # your current GMA + HistGradientBoosting result

advanced_results_df["beats_current_best"] = advanced_results_df["RMSE"] < current_best_rmse

advanced_results_df

In [ ]:
advanced_results_df = pd.DataFrame(advanced_results).sort_values("R2", ascending=False)

advanced_results_df

In [ ]:
best_advanced_by_r2 = advanced_results_df.sort_values("R2", ascending=False).iloc[0]

print("Best advanced model by R2:")
print(best_advanced_by_r2)

In [ ]:
advanced_predictions = {
    "FT-Transformer": {
        "y_true": y_test,
        "y_pred": ft_pred
    },

    "BiLSTM embeddings + GBDT": {
        "y_true": y_hybrid_test,
        "y_pred": hybrid_pred
    },

    "TFT": {
        "y_true": pd.Series(y_true_tft, index=test_part.index[-len(y_true_tft):]),
        "y_pred": y_pred_tft
    }
}

In [ ]:
advanced_predictions = {
    "FT-Transformer": {
        "y_true": y_test,
        "y_pred": ft_pred
    },

    "BiLSTM embeddings + GBDT": {
        "y_true": y_hybrid_test,
        "y_pred": hybrid_pred
    },

    "TFT": {
        "y_true": pd.Series(
            y_true_tft[mask_real_target],
            index=test_part.index[-len(y_true_tft):][mask_real_target]
        ),
        "y_pred": y_pred_tft[mask_real_target]
    }
}

In [ ]:
best_model_name = best_advanced_by_r2["model"]

print("Best model selected:", best_model_name)

best_y_true = advanced_predictions[best_model_name]["y_true"]
best_y_pred = advanced_predictions[best_model_name]["y_pred"]

In [ ]:
min_len = min(len(best_y_true), len(best_y_pred))

best_y_true = best_y_true.iloc[-min_len:] if hasattr(best_y_true, "iloc") else best_y_true[-min_len:]
best_y_pred = best_y_pred[-min_len:]

print("y_true length:", len(best_y_true))
print("y_pred length:", len(best_y_pred))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(16, 5))

if hasattr(best_y_true, "index"):
    x_axis = best_y_true.index
else:
    x_axis = range(len(best_y_true))

plt.plot(x_axis, best_y_true, label="Actual CO(GT)", alpha=0.8)
plt.plot(x_axis, best_y_pred, label=f"Predicted CO(GT) - {best_model_name}", alpha=0.8)

plt.title(f"Actual vs Predicted CO(GT) — Best Advanced Model by R²: {best_model_name}")
plt.xlabel("Date")
plt.ylabel("CO(GT)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
residuals = np.array(best_y_true) - np.array(best_y_pred)

plt.figure(figsize=(16, 4))

if hasattr(best_y_true, "index"):
    x_axis = best_y_true.index
else:
    x_axis = range(len(best_y_true))

plt.plot(x_axis, residuals)
plt.axhline(0, color="black", linestyle="--")

plt.title(f"Residuals over time — {best_model_name}")
plt.xlabel("Date")
plt.ylabel("Error")
plt.grid(True)
plt.show()

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(best_y_true, best_y_pred)
rmse = np.sqrt(mean_squared_error(best_y_true, best_y_pred))
r2 = r2_score(best_y_true, best_y_pred)

print("Best advanced model:", best_model_name)
print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

{'MAE': 0.2521207688654478, 'RMSE': np.float64(0.38130829922583537), 'sMAPE': np.float64(16.763423562102197), 'R2': 0.9229409069651028}

# new bidlstm

In [ ]:
class BiLSTMGlobalPoolingRegressor(nn.Module):
    def __init__(self, n_features, hidden_size=64, num_layers=1, dropout=0.1):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0 if num_layers == 1 else dropout
        )

        # BiLSTM output dimension = hidden_size * 2
        lstm_output_dim = hidden_size * 2

        # We concatenate mean + max + min pooling
        embedding_dim = lstm_output_dim * 3

        self.head = nn.Sequential(
            nn.Linear(embedding_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(64, 1)
        )

    def forward(self, x, return_embedding=False):
        # output shape: batch_size × seq_len × hidden_size*2
        output, (h, c) = self.lstm(x)

        # Global pooling over the full 48-hour window
        mean_pool = output.mean(dim=1)
        max_pool = output.max(dim=1).values
        min_pool = output.min(dim=1).values

        # Final embedding
        embedding = torch.cat(
            [mean_pool, max_pool, min_pool],
            dim=1
        )

        pred = self.head(embedding).squeeze(-1)

        if return_embedding:
            return pred, embedding

        return pred

In [ ]:
bilstm_model = BiLSTMRegressor(
    n_features=Xseq_train_full.shape[2],
    hidden_size=64,
    num_layers=1,
    dropout=0.1
)

In [ ]:
bilstm_model = BiLSTMGlobalPoolingRegressor(
    n_features=Xseq_train_full.shape[2],
    hidden_size=64,
    num_layers=1,
    dropout=0.1
)

In [ ]:
bilstm_model = train_bilstm(
    bilstm_model,
    train_seq_loader,
    val_seq_loader,
    epochs=80,
    lr=1e-3,
    patience=10
)

In [ ]:
emb_train_full = extract_embeddings(bilstm_model, Xseq_train_full)
emb_test = extract_embeddings(bilstm_model, Xseq_test)

print("Embeddings train:", emb_train_full.shape)
print("Embeddings test:", emb_test.shape)

In [ ]:
emb_cols = [f"bilstm_pool_emb_{i}" for i in range(emb_train_full.shape[1])]

emb_train_df = pd.DataFrame(
    emb_train_full,
    index=Xtab_train_full.index,
    columns=emb_cols
)

emb_test_df = pd.DataFrame(
    emb_test,
    index=Xtab_test.index,
    columns=emb_cols
)

X_hybrid_train = pd.concat([Xtab_train_full, emb_train_df], axis=1)
X_hybrid_test = pd.concat([Xtab_test, emb_test_df], axis=1)

y_hybrid_train = ytab_train_full.copy()
y_hybrid_test = ytab_test.copy()

print("Hybrid train:", X_hybrid_train.shape)
print("Hybrid test:", X_hybrid_test.shape)

In [ ]:
hybrid_model.fit(X_hybrid_train, y_hybrid_train)

hybrid_pred = hybrid_model.predict(X_hybrid_test)

hybrid_metrics = evaluate_model(y_hybrid_test, hybrid_pred)

print("Hybrid BiLSTM Global Pooling + GBDT test results:")
print(hybrid_metrics)

# stacking errors

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


def smape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    denominator = np.where(denominator == 0, 1e-8, denominator)

    return np.mean(np.abs(y_true - y_pred) / denominator) * 100


def evaluate_model(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    s_mape = smape(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return {
        "MAE": mae,
        "RMSE": rmse,
        "sMAPE": s_mape,
        "R2": r2
    }

In [ ]:
class BiLSTMGlobalPoolingRegressor(nn.Module):
    def __init__(self, n_features, hidden_size=64, num_layers=1, dropout=0.1):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0 if num_layers == 1 else dropout
        )

        lstm_output_dim = hidden_size * 2
        embedding_dim = lstm_output_dim * 3  # mean + max + min pooling

        self.head = nn.Sequential(
            nn.Linear(embedding_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        output, (h, c) = self.lstm(x)

        mean_pool = output.mean(dim=1)
        max_pool = output.max(dim=1).values
        min_pool = output.min(dim=1).values

        embedding = torch.cat(
            [mean_pool, max_pool, min_pool],
            dim=1
        )

        pred = self.head(embedding).squeeze(-1)

        return pred

In [ ]:
def train_bilstm_model(
    X_train_seq,
    y_train,
    X_val_seq,
    y_val,
    hidden_size=64,
    epochs=60,
    lr=1e-3,
    batch_size=128,
    patience=8
):
    y_scaler = StandardScaler()

    y_train_scaled = y_scaler.fit_transform(
        np.array(y_train).reshape(-1, 1)
    ).ravel()

    y_val_scaled = y_scaler.transform(
        np.array(y_val).reshape(-1, 1)
    ).ravel()

    train_ds = TensorDataset(
        torch.tensor(X_train_seq, dtype=torch.float32),
        torch.tensor(y_train_scaled, dtype=torch.float32)
    )

    val_ds = TensorDataset(
        torch.tensor(X_val_seq, dtype=torch.float32),
        torch.tensor(y_val_scaled, dtype=torch.float32)
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False
    )

    model = BiLSTMGlobalPoolingRegressor(
        n_features=X_train_seq.shape[2],
        hidden_size=hidden_size,
        num_layers=1,
        dropout=0.1
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=1e-4
    )

    loss_fn = nn.MSELoss()

    best_val_loss = np.inf
    best_state = None
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        train_losses = []

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        model.eval()
        val_losses = []

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)

                pred = model(xb)
                loss = loss_fn(pred, yb)
                val_losses.append(loss.item())

        train_loss = np.mean(train_losses)
        val_loss = np.mean(val_losses)

        if epoch % 5 == 0:
            print(f"Epoch {epoch} | Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print("Early stopping.")
            break

    model.load_state_dict(best_state)

    return model, y_scaler


def predict_bilstm(model, y_scaler, X_seq, batch_size=256):
    model.eval()

    loader = DataLoader(
        torch.tensor(X_seq, dtype=torch.float32),
        batch_size=batch_size,
        shuffle=False
    )

    preds_scaled = []

    with torch.no_grad():
        for xb in loader:
            xb = xb.to(device)
            pred = model(xb)
            preds_scaled.append(pred.cpu().numpy())

    preds_scaled = np.concatenate(preds_scaled)

    preds = y_scaler.inverse_transform(
        preds_scaled.reshape(-1, 1)
    ).ravel()

    return preds

In [ ]:
def make_time_folds(n, start_frac=0.60, n_folds=4):
    """
    Expanding-window folds.

    Example:
    Fold 1: train 0%→60%, validate 60%→70%
    Fold 2: train 0%→70%, validate 70%→80%
    Fold 3: train 0%→80%, validate 80%→90%
    Fold 4: train 0%→90%, validate 90%→100%
    """

    start = int(n * start_frac)
    boundaries = np.linspace(start, n, n_folds + 1, dtype=int)

    folds = []

    for i in range(n_folds):
        train_end = boundaries[i]
        val_start = boundaries[i]
        val_end = boundaries[i + 1]

        folds.append((0, train_end, val_start, val_end))

    return folds

In [ ]:
n = len(Xseq_train_full)

oof_bilstm_pred = np.full(n, np.nan)

folds = make_time_folds(
    n=n,
    start_frac=0.60,
    n_folds=4
)

for fold_id, (tr_start, tr_end, val_start, val_end) in enumerate(folds, 1):
    print("\n" + "="*80)
    print(f"Fold {fold_id}")
    print(f"Train: {tr_start} → {tr_end}")
    print(f"Valid: {val_start} → {val_end}")
    print("="*80)

    X_seq_tr = Xseq_train_full[tr_start:tr_end]
    y_tr = ytab_train_full.iloc[tr_start:tr_end]

    X_seq_val = Xseq_train_full[val_start:val_end]
    y_val = ytab_train_full.iloc[val_start:val_end]

    fold_model, fold_scaler = train_bilstm_model(
        X_train_seq=X_seq_tr,
        y_train=y_tr,
        X_val_seq=X_seq_val,
        y_val=y_val,
        hidden_size=64,
        epochs=60,
        lr=1e-3,
        batch_size=128,
        patience=8
    )

    val_pred = predict_bilstm(
        fold_model,
        fold_scaler,
        X_seq_val
    )

    oof_bilstm_pred[val_start:val_end] = val_pred

In [ ]:
oof_mask = ~np.isnan(oof_bilstm_pred)

print("OOF observations:", oof_mask.sum())
print("Total train observations:", len(oof_bilstm_pred))

In [ ]:
residual_train = ytab_train_full.iloc[oof_mask] - oof_bilstm_pred[oof_mask]

X_residual_train = Xtab_train_full.iloc[oof_mask].copy()

X_residual_train = (
    X_residual_train
    .replace([np.inf, -np.inf], np.nan)
    .ffill()
    .bfill()
    .fillna(0)
)

print("Residual training data:", X_residual_train.shape)
print("Residual target:", residual_train.shape)

In [ ]:
try:
    from lightgbm import LGBMRegressor

    residual_model = LGBMRegressor(
        n_estimators=800,
        learning_rate=0.03,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )

    print("Using LightGBM residual model.")

except:
    from sklearn.ensemble import HistGradientBoostingRegressor

    residual_model = HistGradientBoostingRegressor(
        max_iter=800,
        learning_rate=0.03,
        max_leaf_nodes=31,
        l2_regularization=0.01,
        random_state=42
    )

    print("Using HistGradientBoosting residual model.")

In [ ]:
residual_model.fit(X_residual_train, residual_train)

In [ ]:
final_split = int(len(Xseq_train_full) * 0.90)

X_seq_final_train = Xseq_train_full[:final_split]
y_final_train = ytab_train_full.iloc[:final_split]

X_seq_final_val = Xseq_train_full[final_split:]
y_final_val = ytab_train_full.iloc[final_split:]

final_bilstm_model, final_y_scaler = train_bilstm_model(
    X_train_seq=X_seq_final_train,
    y_train=y_final_train,
    X_val_seq=X_seq_final_val,
    y_val=y_final_val,
    hidden_size=64,
    epochs=80,
    lr=1e-3,
    batch_size=128,
    patience=10
)

In [ ]:
bilstm_test_pred = predict_bilstm(
    final_bilstm_model,
    final_y_scaler,
    Xseq_test
)

bilstm_metrics = evaluate_model(
    ytab_test,
    bilstm_test_pred
)

print("BiLSTM baseline test metrics:")
print(bilstm_metrics)

In [ ]:
X_residual_test = Xtab_test.copy()

X_residual_test = (
    X_residual_test
    .replace([np.inf, -np.inf], np.nan)
    .ffill()
    .bfill()
    .fillna(0)
)

residual_test_pred = residual_model.predict(X_residual_test)

In [ ]:
residual_stacked_pred = bilstm_test_pred + residual_test_pred

residual_stacking_metrics = evaluate_model(
    ytab_test,
    residual_stacked_pred
)

print("Residual stacking test metrics:")
print(residual_stacking_metrics)

In [ ]:
comparison_residual_stacking = pd.DataFrame([
    {
        "model": "BiLSTM alone",
        **bilstm_metrics
    },
    {
        "model": "BiLSTM + GBDT residual stacking",
        **residual_stacking_metrics
    },
    {
        "model": "Previous hybrid BiLSTM embeddings + GBDT",
        "MAE": 0.2521207688654478,
        "RMSE": 0.38130829922583537,
        "sMAPE": np.nan,
        "R2": 0.9229409069651028
    }
]).sort_values("RMSE")

comparison_residual_stacking

# RFE

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
def smape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    denominator = np.where(denominator == 0, 1e-8, denominator)

    return np.mean(np.abs(y_true - y_pred) / denominator) * 100


def evaluate_model(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    s_mape = smape(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return {
        "MAE": mae,
        "RMSE": rmse,
        "sMAPE": s_mape,
        "R2": r2
    }

In [ ]:
# Temporal validation inside training data
split_idx = int(len(X_hybrid_train) * 0.8)

X_train_rfe = X_hybrid_train.iloc[:split_idx].copy()
y_train_rfe = y_hybrid_train.iloc[:split_idx].copy()

X_val_rfe = X_hybrid_train.iloc[split_idx:].copy()
y_val_rfe = y_hybrid_train.iloc[split_idx:].copy()

print("RFE train:", X_train_rfe.shape)
print("RFE validation:", X_val_rfe.shape)
print("Final test:", X_hybrid_test.shape)

In [ ]:
def recursive_feature_elimination_gbdt(
    base_model,
    X_train,
    y_train,
    X_val,
    y_val,
    min_features=20,
    patience=20,
    metric_to_optimize="R2"
):
    """
    Recursive Feature Elimination for GBDT models.

    Starts with all features.
    At each step:
    1. Train model
    2. Evaluate on validation set
    3. Read feature importances
    4. Drop the least important feature
    5. Repeat

    Keeps the feature subset with the best validation performance.
    """

    current_features = list(X_train.columns)

    history = []

    best_score = -np.inf if metric_to_optimize == "R2" else np.inf
    best_features = current_features.copy()
    best_metrics = None

    no_improve_count = 0

    iteration = 0

    while len(current_features) > min_features:
        iteration += 1

        print(f"\nIteration {iteration}")
        print(f"Number of features: {len(current_features)}")

        model = clone(base_model)

        model.fit(
            X_train[current_features],
            y_train
        )

        val_pred = model.predict(X_val[current_features])

        metrics = evaluate_model(y_val, val_pred)

        print(
            f"Validation RMSE: {metrics['RMSE']:.5f} | "
            f"R2: {metrics['R2']:.5f} | "
            f"MAE: {metrics['MAE']:.5f}"
        )

        history.append({
            "iteration": iteration,
            "n_features": len(current_features),
            "RMSE": metrics["RMSE"],
            "MAE": metrics["MAE"],
            "sMAPE": metrics["sMAPE"],
            "R2": metrics["R2"],
            "features": current_features.copy()
        })

        # Check improvement
        if metric_to_optimize == "R2":
            improved = metrics["R2"] > best_score
            current_score = metrics["R2"]
        else:
            improved = metrics["RMSE"] < best_score
            current_score = metrics["RMSE"]

        if improved:
            best_score = current_score
            best_features = current_features.copy()
            best_metrics = metrics.copy()
            no_improve_count = 0

            print("New best feature subset found.")
        else:
            no_improve_count += 1

        if no_improve_count >= patience:
            print("\nEarly stopping: no improvement.")
            break

        # Get feature importances
        if not hasattr(model, "feature_importances_"):
            raise ValueError("The model does not have feature_importances_. Use a tree/GBDT model.")

        importances = pd.Series(
            model.feature_importances_,
            index=current_features
        )

        least_important_feature = importances.sort_values().index[0]
        least_importance_value = importances.sort_values().iloc[0]

        print(
            f"Dropping least important feature: {least_important_feature} "
            f"(importance={least_importance_value})"
        )

        current_features.remove(least_important_feature)

    history_df = pd.DataFrame(history)

    return best_features, best_metrics, history_df

In [ ]:
best_features_rfe, best_metrics_rfe, rfe_history_df = recursive_feature_elimination_gbdt(
    base_model=hybrid_model,
    X_train=X_train_rfe,
    y_train=y_train_rfe,
    X_val=X_val_rfe,
    y_val=y_val_rfe,
    min_features=30,
    patience=25,
    metric_to_optimize="R2"
)

In [ ]:
print("Best validation metrics from RFE:")
print(best_metrics_rfe)

print("\nNumber of selected features:")
print(len(best_features_rfe))

print("\nSelected features:")
print(best_features_rfe)

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    rfe_history_df["n_features"],
    rfe_history_df["R2"],
    marker="o"
)

plt.gca().invert_xaxis()

plt.title("RFE performance — Validation R²")
plt.xlabel("Number of features")
plt.ylabel("Validation R²")
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    rfe_history_df["n_features"],
    rfe_history_df["RMSE"],
    marker="o"
)

plt.gca().invert_xaxis()

plt.title("RFE performance — Validation RMSE")
plt.xlabel("Number of features")
plt.ylabel("Validation RMSE")
plt.grid(True)
plt.show()

In [ ]:
final_rfe_model = clone(hybrid_model)

final_rfe_model.fit(
    X_hybrid_train[best_features_rfe],
    y_hybrid_train
)

rfe_test_pred = final_rfe_model.predict(
    X_hybrid_test[best_features_rfe]
)

rfe_test_metrics = evaluate_model(
    y_hybrid_test,
    rfe_test_pred
)

print("Final RFE model test metrics:")
print(rfe_test_metrics)

In [ ]:
df

# k flod stratefy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import copy
import warnings

warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


def smape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    denominator = np.where(denominator == 0, 1e-8, denominator)

    return np.mean(np.abs(y_true - y_pred) / denominator) * 100


def evaluate_model(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    s_mape = smape(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return {
        "MAE": mae,
        "RMSE": rmse,
        "sMAPE": s_mape,
        "R2": r2
    }

In [ ]:
class BiLSTMGlobalPoolingEmbedder(nn.Module):
    def __init__(self, n_features, hidden_size=64, num_layers=1, dropout=0.1):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0 if num_layers == 1 else dropout
        )

        lstm_output_dim = hidden_size * 2
        embedding_dim = lstm_output_dim * 3

        self.head = nn.Sequential(
            nn.Linear(embedding_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(64, 1)
        )

    def forward(self, x, return_embedding=False):
        output, (h, c) = self.lstm(x)

        mean_pool = output.mean(dim=1)
        max_pool = output.max(dim=1).values
        min_pool = output.min(dim=1).values

        embedding = torch.cat(
            [mean_pool, max_pool, min_pool],
            dim=1
        )

        pred = self.head(embedding).squeeze(-1)

        if return_embedding:
            return pred, embedding

        return pred

In [ ]:
def train_bilstm_for_embeddings(
    X_train_seq,
    y_train,
    X_val_seq,
    y_val,
    hidden_size=64,
    epochs=60,
    lr=1e-3,
    batch_size=128,
    patience=8
):
    y_scaler = StandardScaler()

    y_train_scaled = y_scaler.fit_transform(
        np.array(y_train).reshape(-1, 1)
    ).ravel()

    y_val_scaled = y_scaler.transform(
        np.array(y_val).reshape(-1, 1)
    ).ravel()

    train_ds = TensorDataset(
        torch.tensor(X_train_seq, dtype=torch.float32),
        torch.tensor(y_train_scaled, dtype=torch.float32)
    )

    val_ds = TensorDataset(
        torch.tensor(X_val_seq, dtype=torch.float32),
        torch.tensor(y_val_scaled, dtype=torch.float32)
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False
    )

    model = BiLSTMGlobalPoolingEmbedder(
        n_features=X_train_seq.shape[2],
        hidden_size=hidden_size,
        num_layers=1,
        dropout=0.1
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=1e-4
    )

    loss_fn = nn.MSELoss()

    best_val_loss = np.inf
    best_state = None
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        train_losses = []

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        model.eval()
        val_losses = []

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)

                pred = model(xb)
                loss = loss_fn(pred, yb)
                val_losses.append(loss.item())

        train_loss = np.mean(train_losses)
        val_loss = np.mean(val_losses)

        if epoch % 5 == 0:
            print(f"Epoch {epoch} | Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print("Early stopping.")
            break

    model.load_state_dict(best_state)

    return model

In [ ]:
def extract_bilstm_embeddings(model, X_seq, batch_size=256):
    model.eval()

    loader = DataLoader(
        torch.tensor(X_seq, dtype=torch.float32),
        batch_size=batch_size,
        shuffle=False
    )

    embeddings = []

    with torch.no_grad():
        for xb in loader:
            xb = xb.to(device)
            _, emb = model(xb, return_embedding=True)
            embeddings.append(emb.cpu().numpy())

    return np.vstack(embeddings)

In [ ]:
def make_blocked_folds(n_samples, n_splits=5):
    indices = np.arange(n_samples)
    folds = np.array_split(indices, n_splits)
    return folds

In [ ]:
n_splits = 5
n_train = len(Xseq_train_full)

folds = make_blocked_folds(n_train, n_splits=n_splits)

oof_embeddings = None
test_embeddings_from_folds = []

all_indices = np.arange(n_train)

for fold_id, val_idx in enumerate(folds, 1):
    print("\n" + "="*80)
    print(f"OOF Fold {fold_id}/{n_splits}")
    print("="*80)

    train_idx = np.setdiff1d(all_indices, val_idx)

    # Inner validation for early stopping
    inner_cut = int(len(train_idx) * 0.85)

    inner_train_idx = train_idx[:inner_cut]
    inner_val_idx = train_idx[inner_cut:]

    X_seq_tr = Xseq_train_full[inner_train_idx]
    y_tr = ytab_train_full.iloc[inner_train_idx]

    X_seq_inner_val = Xseq_train_full[inner_val_idx]
    y_inner_val = ytab_train_full.iloc[inner_val_idx]

    X_seq_oof = Xseq_train_full[val_idx]

    print("BiLSTM train:", X_seq_tr.shape)
    print("BiLSTM inner val:", X_seq_inner_val.shape)
    print("OOF block:", X_seq_oof.shape)

    fold_model = train_bilstm_for_embeddings(
        X_train_seq=X_seq_tr,
        y_train=y_tr,
        X_val_seq=X_seq_inner_val,
        y_val=y_inner_val,
        hidden_size=64,
        epochs=60,
        lr=1e-3,
        batch_size=128,
        patience=8
    )

    # Embeddings for left-out training block
    val_emb = extract_bilstm_embeddings(
        fold_model,
        X_seq_oof
    )

    if oof_embeddings is None:
        embedding_dim = val_emb.shape[1]
        oof_embeddings = np.full(
            shape=(n_train, embedding_dim),
            fill_value=np.nan
        )

    oof_embeddings[val_idx] = val_emb

    # Test embeddings from this fold model
    test_emb = extract_bilstm_embeddings(
        fold_model,
        Xseq_test
    )

    test_embeddings_from_folds.append(test_emb)

    # Clean memory
    del fold_model
    torch.cuda.empty_cache()

In [ ]:
print("OOF embeddings shape:", oof_embeddings.shape)
print("Missing OOF values:", np.isnan(oof_embeddings).sum())

In [ ]:
test_embeddings_mean = np.mean(
    np.stack(test_embeddings_from_folds, axis=0),
    axis=0
)

print("Test embeddings shape:", test_embeddings_mean.shape)

In [ ]:
emb_cols = [
    f"oof_bilstm_emb_{i}"
    for i in range(oof_embeddings.shape[1])
]

oof_emb_train_df = pd.DataFrame(
    oof_embeddings,
    index=Xtab_train_full.index,
    columns=emb_cols
)

oof_emb_test_df = pd.DataFrame(
    test_embeddings_mean,
    index=Xtab_test.index,
    columns=emb_cols
)

X_oof_hybrid_train = pd.concat(
    [Xtab_train_full, oof_emb_train_df],
    axis=1
)

X_oof_hybrid_test = pd.concat(
    [Xtab_test, oof_emb_test_df],
    axis=1
)

y_oof_hybrid_train = ytab_train_full.copy()
y_oof_hybrid_test = ytab_test.copy()

# Safety cleaning
X_oof_hybrid_train = (
    X_oof_hybrid_train
    .replace([np.inf, -np.inf], np.nan)
    .ffill()
    .bfill()
    .fillna(0)
)

X_oof_hybrid_test = (
    X_oof_hybrid_test
    .replace([np.inf, -np.inf], np.nan)
    .ffill()
    .bfill()
    .fillna(0)
)

print("OOF hybrid train:", X_oof_hybrid_train.shape)
print("OOF hybrid test:", X_oof_hybrid_test.shape)

In [ ]:
try:
    from lightgbm import LGBMRegressor

    oof_hybrid_model = LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.025,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=20,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.05,
        reg_lambda=0.10,
        random_state=42,
        n_jobs=-1,
        force_col_wise=True,
        verbose=-1
    )

    print("Using LightGBM.")

except:
    from sklearn.ensemble import ExtraTreesRegressor

    oof_hybrid_model = ExtraTreesRegressor(
        n_estimators=800,
        max_depth=None,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )

    print("Using ExtraTrees.")

In [ ]:
oof_hybrid_model.fit(
    X_oof_hybrid_train,
    y_oof_hybrid_train
)

In [ ]:
oof_hybrid_pred = oof_hybrid_model.predict(
    X_oof_hybrid_test
)

oof_hybrid_metrics = evaluate_model(
    y_oof_hybrid_test,
    oof_hybrid_pred
)

print("OOF BiLSTM embeddings + GBDT test metrics:")
print(oof_hybrid_metrics)

In [ ]:
previous_best_metrics = {
    "MAE": 0.2521207688654478,
    "RMSE": 0.38130829922583537,
    "sMAPE": np.nan,
    "R2": 0.9229409069651028
}

comparison_oof = pd.DataFrame([
    {
        "model": "Previous BiLSTM embeddings + GBDT",
        **previous_best_metrics
    },
    {
        "model": "OOF BiLSTM embeddings + GBDT",
        **oof_hybrid_metrics
    }
]).sort_values("RMSE")

comparison_oof

In [ ]:
rmse_improvement = (
    previous_best_metrics["RMSE"] - oof_hybrid_metrics["RMSE"]
) / previous_best_metrics["RMSE"] * 100

r2_improvement = (
    oof_hybrid_metrics["R2"] - previous_best_metrics["R2"]
)

print("RMSE improvement (%):", rmse_improvement)
print("R2 improvement:", r2_improvement)

# sencor

In [ ]:
def add_nowcasting_interactions(X):
    X = X.copy()
    eps = 1e-6

    # Log transforms for skewed pollution variables
    for col in [
        "C6H6(GT)", "NOx(GT)", "NO2(GT)",
        "PT08.S1(CO)", "PT08.S2(NMHC)",
        "PT08.S3(NOx)", "PT08.S4(NO2)",
        "PT08.S5(O3)"
    ]:
        if col in X.columns:
            X[f"log_{col}"] = np.log1p(np.maximum(X[col], 0))

    # Important ratios
    if "NO2(GT)" in X.columns and "NOx(GT)" in X.columns:
        X["NO2_over_NOx"] = X["NO2(GT)"] / (X["NOx(GT)"] + eps)
        X["NOx_minus_NO2"] = X["NOx(GT)"] - X["NO2(GT)"]

    if "C6H6(GT)" in X.columns and "NOx(GT)" in X.columns:
        X["C6H6_over_NOx"] = X["C6H6(GT)"] / (X["NOx(GT)"] + eps)

    if "C6H6(GT)" in X.columns and "NO2(GT)" in X.columns:
        X["C6H6_over_NO2"] = X["C6H6(GT)"] / (X["NO2(GT)"] + eps)

    # Sensor interactions
    if "C6H6(GT)" in X.columns and "NOx(GT)" in X.columns:
        X["C6H6_x_NOx"] = X["C6H6(GT)"] * X["NOx(GT)"]

    if "C6H6(GT)" in X.columns and "NO2(GT)" in X.columns:
        X["C6H6_x_NO2"] = X["C6H6(GT)"] * X["NO2(GT)"]

    if "PT08.S1(CO)" in X.columns and "PT08.S5(O3)" in X.columns:
        X["PT08S1_x_PT08S5"] = X["PT08.S1(CO)"] * X["PT08.S5(O3)"]

    # Weather interactions
    if "T" in X.columns and "RH" in X.columns:
        X["T_x_RH"] = X["T"] * X["RH"]

    if "AH" in X.columns and "T" in X.columns:
        X["AH_over_T"] = X["AH"] / (X["T"].abs() + eps)

    # Time × sensor interactions
    for sensor in ["C6H6(GT)", "NOx(GT)", "NO2(GT)", "PT08.S1(CO)", "PT08.S5(O3)"]:
        if sensor in X.columns and "hour_sin" in X.columns:
            X[f"{sensor}_x_hour_sin"] = X[sensor] * X["hour_sin"]

        if sensor in X.columns and "hour_cos" in X.columns:
            X[f"{sensor}_x_hour_cos"] = X[sensor] * X["hour_cos"]

        if sensor in X.columns and "is_weekend" in X.columns:
            X[f"{sensor}_x_weekend"] = X[sensor] * X["is_weekend"]

    # Peak indicators based on current sensors
    for col in ["C6H6(GT)", "NOx(GT)", "NO2(GT)", "PT08.S1(CO)", "PT08.S5(O3)"]:
        if col in X.columns:
            q90 = X[col].quantile(0.90)
            q95 = X[col].quantile(0.95)

            X[f"{col}_high_90"] = (X[col] > q90).astype(int)
            X[f"{col}_high_95"] = (X[col] > q95).astype(int)

    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.ffill().bfill().fillna(0)

    return X

In [ ]:
Xtab_train_fe = add_nowcasting_interactions(Xtab_train_full)
Xtab_test_fe = add_nowcasting_interactions(Xtab_test)

# Align columns
Xtab_test_fe = Xtab_test_fe.reindex(columns=Xtab_train_fe.columns, fill_value=0)

print(Xtab_train_fe.shape)
print(Xtab_test_fe.shape)

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

def evaluate_model(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    denominator = np.where(denominator == 0, 1e-8, denominator)
    smape = np.mean(np.abs(y_true - y_pred) / denominator) * 100

    return {
        "MAE": mae,
        "RMSE": rmse,
        "sMAPE": smape,
        "R2": r2
    }

In [ ]:
models = {}

models["HistGradientBoosting_FE"] = HistGradientBoostingRegressor(
    max_iter=800,
    learning_rate=0.03,
    max_leaf_nodes=31,
    l2_regularization=0.01,
    random_state=42
)

models["ExtraTrees_FE"] = ExtraTreesRegressor(
    n_estimators=800,
    max_depth=None,
    min_samples_leaf=2,
    max_features=0.8,
    random_state=42,
    n_jobs=-1
)

try:
    from lightgbm import LGBMRegressor

    models["LightGBM_FE"] = LGBMRegressor(
        n_estimators=1200,
        learning_rate=0.02,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=15,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.05,
        reg_lambda=0.10,
        random_state=42,
        n_jobs=-1,
        force_col_wise=True,
        verbose=-1
    )
except:
    print("LightGBM not available.")

try:
    from xgboost import XGBRegressor

    models["XGBoost_FE"] = XGBRegressor(
        n_estimators=1200,
        learning_rate=0.02,
        max_depth=4,
        min_child_weight=3,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )
except:
    print("XGBoost not available.")

In [ ]:
fe_results = []
fe_predictions = {}

for name, model in models.items():
    print("Training:", name)

    model.fit(Xtab_train_fe, ytab_train_full)
    pred = model.predict(Xtab_test_fe)

    metrics = evaluate_model(ytab_test, pred)

    fe_results.append({
        "model": name,
        **metrics
    })

    fe_predictions[name] = pred

fe_results_df = pd.DataFrame(fe_results).sort_values("RMSE")
fe_results_df

In [ ]:
pred_matrix = np.column_stack([
    fe_predictions[name]
    for name in fe_predictions.keys()
])

ensemble_pred = pred_matrix.mean(axis=1)

ensemble_metrics = evaluate_model(ytab_test, ensemble_pred)

print("Simple FE ensemble:")
print(ensemble_metrics)

In [ ]:
peak_threshold = ytab_train_full.quantile(0.85)

print("Peak threshold:", peak_threshold)

train_peak_mask = ytab_train_full >= peak_threshold
test_peak_mask = ytab_test >= peak_threshold

print("Train peak observations:", train_peak_mask.sum())
print("Test peak observations:", test_peak_mask.sum())

In [ ]:
peak_model = LGBMRegressor(
    n_estimators=800,
    learning_rate=0.02,
    num_leaves=31,
    min_child_samples=10,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.03,
    reg_lambda=0.05,
    random_state=42,
    n_jobs=-1,
    force_col_wise=True,
    verbose=-1
)

peak_model.fit(
    Xtab_train_fe.loc[train_peak_mask],
    ytab_train_full.loc[train_peak_mask]
)

In [ ]:
best_fe_model_name = fe_results_df.iloc[0]["model"]
best_fe_model = models[best_fe_model_name]

normal_pred = fe_predictions[best_fe_model_name]
peak_pred = peak_model.predict(Xtab_test_fe)

In [ ]:
likely_peak_test = (
    (Xtab_test_fe["C6H6(GT)_high_90"] == 1) |
    (Xtab_test_fe["NOx(GT)_high_90"] == 1) |
    (Xtab_test_fe["PT08.S5(O3)_high_90"] == 1)
)

peak_aware_pred = normal_pred.copy()
peak_aware_pred[likely_peak_test.values] = peak_pred[likely_peak_test.values]

peak_aware_metrics = evaluate_model(ytab_test, peak_aware_pred)

print("Peak-aware model:")
print(peak_aware_metrics)

In [ ]:
comparison_peak = pd.DataFrame([
    {
        "model": "Best FE GBDT",
        **evaluate_model(ytab_test, normal_pred)
    },
    {
        "model": "Peak-aware FE GBDT",
        **peak_aware_metrics
    },
    {
        "model": "Previous BiLSTM embeddings + GBDT",
        "MAE": 0.2521207688654478,
        "RMSE": 0.38130829922583537,
        "sMAPE": np.nan,
        "R2": 0.9229409069651028
    }
]).sort_values("RMSE")

comparison_peak

# bilstm

In [ ]:
import numpy as np
import pandas as pd
import copy
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


def smape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    denominator = np.where(denominator == 0, 1e-8, denominator)

    return np.mean(np.abs(y_true - y_pred) / denominator) * 100


def evaluate_model(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    s_mape = smape(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return {
        "MAE": mae,
        "RMSE": rmse,
        "sMAPE": s_mape,
        "R2": r2
    }

In [ ]:
class BiLSTMGlobalPoolingRegressor(nn.Module):
    def __init__(self, n_features, hidden_size=64, num_layers=1, dropout=0.1):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0 if num_layers == 1 else dropout
        )

        lstm_output_dim = hidden_size * 2
        embedding_dim = lstm_output_dim * 3

        self.head = nn.Sequential(
            nn.Linear(embedding_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        output, (h, c) = self.lstm(x)

        mean_pool = output.mean(dim=1)
        max_pool = output.max(dim=1).values
        min_pool = output.min(dim=1).values

        embedding = torch.cat([mean_pool, max_pool, min_pool], dim=1)

        pred = self.head(embedding).squeeze(-1)

        return pred

In [ ]:
def train_bilstm_predictor(
    X_train_seq,
    y_train,
    X_val_seq,
    y_val,
    hidden_size=64,
    epochs=60,
    lr=1e-3,
    batch_size=128,
    patience=8
):
    y_scaler = StandardScaler()

    y_train_scaled = y_scaler.fit_transform(
        np.array(y_train).reshape(-1, 1)
    ).ravel()

    y_val_scaled = y_scaler.transform(
        np.array(y_val).reshape(-1, 1)
    ).ravel()

    train_ds = TensorDataset(
        torch.tensor(X_train_seq, dtype=torch.float32),
        torch.tensor(y_train_scaled, dtype=torch.float32)
    )

    val_ds = TensorDataset(
        torch.tensor(X_val_seq, dtype=torch.float32),
        torch.tensor(y_val_scaled, dtype=torch.float32)
    )

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    model = BiLSTMGlobalPoolingRegressor(
        n_features=X_train_seq.shape[2],
        hidden_size=hidden_size,
        num_layers=1,
        dropout=0.1
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=1e-4
    )

    loss_fn = nn.MSELoss()

    best_val_loss = np.inf
    best_state = None
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        train_losses = []

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        model.eval()
        val_losses = []

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)

                pred = model(xb)
                loss = loss_fn(pred, yb)
                val_losses.append(loss.item())

        train_loss = np.mean(train_losses)
        val_loss = np.mean(val_losses)

        if epoch % 5 == 0:
            print(f"Epoch {epoch} | Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print("Early stopping.")
            break

    model.load_state_dict(best_state)

    return model, y_scaler

In [ ]:
def predict_bilstm(model, y_scaler, X_seq, batch_size=256):
    model.eval()

    loader = DataLoader(
        torch.tensor(X_seq, dtype=torch.float32),
        batch_size=batch_size,
        shuffle=False
    )

    preds_scaled = []

    with torch.no_grad():
        for xb in loader:
            xb = xb.to(device)
            pred = model(xb)
            preds_scaled.append(pred.cpu().numpy())

    preds_scaled = np.concatenate(preds_scaled)

    preds = y_scaler.inverse_transform(
        preds_scaled.reshape(-1, 1)
    ).ravel()

    return preds

In [ ]:
def make_blocked_folds(n_samples, n_splits=5):
    indices = np.arange(n_samples)
    folds = np.array_split(indices, n_splits)
    return folds

In [ ]:
n_splits = 5
n_train = len(Xseq_train_full)

folds = make_blocked_folds(n_train, n_splits=n_splits)
all_indices = np.arange(n_train)

oof_bilstm_pred = np.full(n_train, np.nan)
test_predictions_from_folds = []

for fold_id, val_idx in enumerate(folds, 1):

    print("\n" + "="*80)
    print(f"OOF prediction fold {fold_id}/{n_splits}")
    print("="*80)

    train_idx = np.setdiff1d(all_indices, val_idx)

    # Inner validation for early stopping
    inner_cut = int(len(train_idx) * 0.85)

    inner_train_idx = train_idx[:inner_cut]
    inner_val_idx = train_idx[inner_cut:]

    X_seq_tr = Xseq_train_full[inner_train_idx]
    y_tr = ytab_train_full.iloc[inner_train_idx]

    X_seq_val = Xseq_train_full[inner_val_idx]
    y_val = ytab_train_full.iloc[inner_val_idx]

    X_seq_oof = Xseq_train_full[val_idx]

    model_fold, scaler_fold = train_bilstm_predictor(
        X_train_seq=X_seq_tr,
        y_train=y_tr,
        X_val_seq=X_seq_val,
        y_val=y_val,
        hidden_size=64,
        epochs=60,
        lr=1e-3,
        batch_size=128,
        patience=8
    )

    # OOF prediction for left-out block
    val_pred = predict_bilstm(
        model_fold,
        scaler_fold,
        X_seq_oof
    )

    oof_bilstm_pred[val_idx] = val_pred

    # Test prediction from this fold model
    test_pred_fold = predict_bilstm(
        model_fold,
        scaler_fold,
        Xseq_test
    )

    test_predictions_from_folds.append(test_pred_fold)

    del model_fold
    torch.cuda.empty_cache()

In [ ]:
print("Missing OOF predictions:", np.isnan(oof_bilstm_pred).sum())

In [ ]:
bilstm_test_pred_avg = np.mean(
    np.column_stack(test_predictions_from_folds),
    axis=1
)

bilstm_oof_metrics = evaluate_model(
    ytab_train_full,
    oof_bilstm_pred
)

bilstm_test_metrics = evaluate_model(
    ytab_test,
    bilstm_test_pred_avg
)

print("BiLSTM OOF train metrics:")
print(bilstm_oof_metrics)

print("BiLSTM averaged test metrics:")
print(bilstm_test_metrics)

In [ ]:
residual_train = ytab_train_full.values - oof_bilstm_pred

print("Residual mean:", residual_train.mean())
print("Residual std:", residual_train.std())

In [ ]:
X_stack_train = Xtab_train_full.copy()
X_stack_test = Xtab_test.copy()

X_stack_train["bilstm_oof_pred"] = oof_bilstm_pred
X_stack_test["bilstm_oof_pred"] = bilstm_test_pred_avg

if "CO(GT)_lag_1" in X_stack_train.columns:
    X_stack_train["bilstm_minus_lag1"] = (
        X_stack_train["bilstm_oof_pred"] - X_stack_train["CO(GT)_lag_1"]
    )

    X_stack_test["bilstm_minus_lag1"] = (
        X_stack_test["bilstm_oof_pred"] - X_stack_test["CO(GT)_lag_1"]
    )

X_stack_train = (
    X_stack_train
    .replace([np.inf, -np.inf], np.nan)
    .ffill()
    .bfill()
    .fillna(0)
)

X_stack_test = (
    X_stack_test
    .replace([np.inf, -np.inf], np.nan)
    .ffill()
    .bfill()
    .fillna(0)
)

In [ ]:
try:
    from lightgbm import LGBMRegressor

    residual_corrector = LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.02,
        num_leaves=31,
        min_child_samples=15,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.05,
        reg_lambda=0.10,
        random_state=42,
        n_jobs=-1,
        force_col_wise=True,
        verbose=-1
    )

except:
    from sklearn.ensemble import HistGradientBoostingRegressor

    residual_corrector = HistGradientBoostingRegressor(
        max_iter=1000,
        learning_rate=0.02,
        max_leaf_nodes=31,
        l2_regularization=0.05,
        random_state=42
    )

In [ ]:
residual_corrector.fit(
    X_stack_train,
    residual_train
)

residual_test_pred = residual_corrector.predict(
    X_stack_test
)

final_residual_stack_pred = bilstm_test_pred_avg + residual_test_pred

residual_stack_metrics = evaluate_model(
    ytab_test,
    final_residual_stack_pred
)

print("OOF BiLSTM prediction + GBDT residual correction:")
print(residual_stack_metrics)

In [ ]:
try:
    from lightgbm import LGBMRegressor

    direct_stack_model = LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.02,
        num_leaves=31,
        min_child_samples=15,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.05,
        reg_lambda=0.10,
        random_state=42,
        n_jobs=-1,
        force_col_wise=True,
        verbose=-1
    )

except:
    from sklearn.ensemble import HistGradientBoostingRegressor

    direct_stack_model = HistGradientBoostingRegressor(
        max_iter=1000,
        learning_rate=0.02,
        max_leaf_nodes=31,
        l2_regularization=0.05,
        random_state=42
    )

In [ ]:
direct_stack_model.fit(
    X_stack_train,
    ytab_train_full
)

direct_stack_pred = direct_stack_model.predict(
    X_stack_test
)

direct_stack_metrics = evaluate_model(
    ytab_test,
    direct_stack_pred
)

print("OOF BiLSTM prediction + GBDT direct stacking:")
print(direct_stack_metrics)

In [ ]:
comparison_next_strategy = pd.DataFrame([
    {
        "model": "Previous BiLSTM embeddings + GBDT",
        "MAE": 0.252121,
        "RMSE": 0.381308,
        "sMAPE": np.nan,
        "R2": 0.922941
    },
    {
        "model": "OOF BiLSTM embeddings + GBDT",
        "MAE": 0.294440,
        "RMSE": 0.485921,
        "sMAPE": 18.138443,
        "R2": 0.874858
    },
    {
        "model": "Best FE GBDT",
        "MAE": 0.316506,
        "RMSE": 0.474606,
        "sMAPE": 21.006539,
        "R2": 0.880618
    },
    {
        "model": "OOF BiLSTM pred + residual GBDT",
        **residual_stack_metrics
    },
    {
        "model": "OOF BiLSTM pred + direct GBDT",
        **direct_stack_metrics
    }
]).sort_values("RMSE")

comparison_next_strategy

# TCN

In [ ]:
# ============================================================
# TCN + GBDT HYBRID MODEL FOR NOWCASTING CO(GT)
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import copy
import warnings

warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ------------------------------------------------------------
# 0. Safety check
# ------------------------------------------------------------

required_vars = [
    "Xseq_train_full", "Xseq_test",
    "Xtab_train_full", "Xtab_test",
    "ytab_train_full", "ytab_test"
]

for var in required_vars:
    if var not in globals():
        raise ValueError(f"{var} is missing. Run the sequence-building code first.")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

print("Sequence train:", Xseq_train_full.shape)
print("Sequence test:", Xseq_test.shape)
print("Tabular train:", Xtab_train_full.shape)
print("Tabular test:", Xtab_test.shape)


# ------------------------------------------------------------
# 1. Metrics
# ------------------------------------------------------------

def smape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    denominator = np.where(denominator == 0, 1e-8, denominator)

    return np.mean(np.abs(y_true - y_pred) / denominator) * 100


def evaluate_model(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    s_mape = smape(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return {
        "MAE": mae,
        "RMSE": rmse,
        "sMAPE": s_mape,
        "R2": r2
    }


# ------------------------------------------------------------
# 2. Train / validation split for TCN
# ------------------------------------------------------------

split_idx = int(len(Xseq_train_full) * 0.8)

Xseq_train = Xseq_train_full[:split_idx]
Xseq_val = Xseq_train_full[split_idx:]

yseq_train = ytab_train_full.iloc[:split_idx]
yseq_val = ytab_train_full.iloc[split_idx:]

print("TCN train:", Xseq_train.shape)
print("TCN validation:", Xseq_val.shape)


# ------------------------------------------------------------
# 3. Scale target for neural network
# ------------------------------------------------------------

y_scaler = StandardScaler()

yseq_train_scaled = y_scaler.fit_transform(
    yseq_train.values.reshape(-1, 1)
).ravel()

yseq_val_scaled = y_scaler.transform(
    yseq_val.values.reshape(-1, 1)
).ravel()


# ------------------------------------------------------------
# 4. DataLoaders
# ------------------------------------------------------------

batch_size = 128

train_ds = TensorDataset(
    torch.tensor(Xseq_train, dtype=torch.float32),
    torch.tensor(yseq_train_scaled, dtype=torch.float32)
)

val_ds = TensorDataset(
    torch.tensor(Xseq_val, dtype=torch.float32),
    torch.tensor(yseq_val_scaled, dtype=torch.float32)
)

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=batch_size,
    shuffle=False
)


# ------------------------------------------------------------
# 5. TCN building blocks
# ------------------------------------------------------------

class Chomp1d(nn.Module):
    """
    Removes extra timesteps created by padding.
    This keeps the convolution causal.
    """
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        if self.chomp_size == 0:
            return x
        return x[:, :, :-self.chomp_size].contiguous()


class TemporalBlock(nn.Module):
    """
    One residual TCN block:
    dilated causal convolution + ReLU + dropout + residual connection.
    """
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        dilation,
        dropout
    ):
        super().__init__()

        padding = (kernel_size - 1) * dilation

        self.conv1 = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size,
            padding=padding,
            dilation=dilation
        )
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(
            out_channels,
            out_channels,
            kernel_size,
            padding=padding,
            dilation=dilation
        )
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.downsample = (
            nn.Conv1d(in_channels, out_channels, kernel_size=1)
            if in_channels != out_channels
            else None
        )

        self.final_relu = nn.ReLU()

    def forward(self, x):
        out = self.conv1(x)
        out = self.chomp1(out)
        out = self.relu1(out)
        out = self.dropout1(out)

        out = self.conv2(out)
        out = self.chomp2(out)
        out = self.relu2(out)
        out = self.dropout2(out)

        residual = x if self.downsample is None else self.downsample(x)

        return self.final_relu(out + residual)


class TCNRegressor(nn.Module):
    """
    TCN model that returns:
    - prediction
    - temporal embedding if return_embedding=True

    Input shape:
    batch_size × seq_len × n_features

    Conv1D expects:
    batch_size × n_features × seq_len
    """
    def __init__(
        self,
        n_features,
        channels=[64, 64, 128],
        kernel_size=3,
        dropout=0.1
    ):
        super().__init__()

        layers = []
        in_channels = n_features

        for i, out_channels in enumerate(channels):
            dilation = 2 ** i

            layers.append(
                TemporalBlock(
                    in_channels=in_channels,
                    out_channels=out_channels,
                    kernel_size=kernel_size,
                    dilation=dilation,
                    dropout=dropout
                )
            )

            in_channels = out_channels

        self.tcn = nn.Sequential(*layers)

        final_channels = channels[-1]

        # Global pooling: remembers entire 48h window
        embedding_dim = final_channels * 3  # mean + max + last

        self.head = nn.Sequential(
            nn.Linear(embedding_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(64, 1)
        )

    def forward(self, x, return_embedding=False):
        # x: batch × seq_len × features
        x = x.transpose(1, 2)  # batch × features × seq_len

        out = self.tcn(x)      # batch × channels × seq_len

        mean_pool = out.mean(dim=2)
        max_pool = out.max(dim=2).values
        last_state = out[:, :, -1]

        embedding = torch.cat(
            [mean_pool, max_pool, last_state],
            dim=1
        )

        pred = self.head(embedding).squeeze(-1)

        if return_embedding:
            return pred, embedding

        return pred


# ------------------------------------------------------------
# 6. Train TCN
# ------------------------------------------------------------

def train_tcn(
    model,
    train_loader,
    val_loader,
    epochs=80,
    lr=1e-3,
    patience=10
):
    model = model.to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=1e-4
    )

    loss_fn = nn.MSELoss()

    best_val_loss = np.inf
    best_state = None
    patience_counter = 0

    history = []

    for epoch in range(epochs):
        model.train()
        train_losses = []

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()

            pred = model(xb)
            loss = loss_fn(pred, yb)

            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        model.eval()
        val_losses = []

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)

                pred = model(xb)
                loss = loss_fn(pred, yb)

                val_losses.append(loss.item())

        train_loss = np.mean(train_losses)
        val_loss = np.mean(val_losses)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss
        })

        if epoch % 5 == 0:
            print(
                f"Epoch {epoch} | "
                f"Train loss: {train_loss:.5f} | "
                f"Val loss: {val_loss:.5f}"
            )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print("Early stopping.")
            break

    model.load_state_dict(best_state)

    return model, pd.DataFrame(history)


# ------------------------------------------------------------
# 7. Initialize and train TCN
# ------------------------------------------------------------

tcn_model = TCNRegressor(
    n_features=Xseq_train_full.shape[2],
    channels=[64, 64, 128],
    kernel_size=3,
    dropout=0.1
)

tcn_model, tcn_history = train_tcn(
    model=tcn_model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=80,
    lr=1e-3,
    patience=10
)


# ------------------------------------------------------------
# 8. Plot TCN training curve
# ------------------------------------------------------------

plt.figure(figsize=(10, 4))
plt.plot(tcn_history["epoch"], tcn_history["train_loss"], label="Train loss")
plt.plot(tcn_history["epoch"], tcn_history["val_loss"], label="Validation loss")
plt.title("TCN training curve")
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.legend()
plt.grid(True)
plt.show()


# ------------------------------------------------------------
# 9. Extract TCN embeddings
# ------------------------------------------------------------

def extract_tcn_embeddings(model, Xseq, batch_size=256):
    model.eval()

    loader = DataLoader(
        torch.tensor(Xseq, dtype=torch.float32),
        batch_size=batch_size,
        shuffle=False
    )

    embeddings = []

    with torch.no_grad():
        for xb in loader:
            xb = xb.to(device)

            _, emb = model(
                xb,
                return_embedding=True
            )

            embeddings.append(emb.cpu().numpy())

    return np.vstack(embeddings)


emb_train_full = extract_tcn_embeddings(
    tcn_model,
    Xseq_train_full
)

emb_test = extract_tcn_embeddings(
    tcn_model,
    Xseq_test
)

print("TCN embeddings train:", emb_train_full.shape)
print("TCN embeddings test:", emb_test.shape)


# ------------------------------------------------------------
# 10. Build TCN + tabular hybrid dataset
# ------------------------------------------------------------

emb_cols = [
    f"tcn_emb_{i}"
    for i in range(emb_train_full.shape[1])
]

emb_train_df = pd.DataFrame(
    emb_train_full,
    index=Xtab_train_full.index,
    columns=emb_cols
)

emb_test_df = pd.DataFrame(
    emb_test,
    index=Xtab_test.index,
    columns=emb_cols
)

X_tcn_hybrid_train = pd.concat(
    [Xtab_train_full, emb_train_df],
    axis=1
)

X_tcn_hybrid_test = pd.concat(
    [Xtab_test, emb_test_df],
    axis=1
)

y_tcn_hybrid_train = ytab_train_full.copy()
y_tcn_hybrid_test = ytab_test.copy()

# Safety cleaning
X_tcn_hybrid_train = (
    X_tcn_hybrid_train
    .replace([np.inf, -np.inf], np.nan)
    .ffill()
    .bfill()
    .fillna(0)
)

X_tcn_hybrid_test = (
    X_tcn_hybrid_test
    .replace([np.inf, -np.inf], np.nan)
    .ffill()
    .bfill()
    .fillna(0)
)

print("TCN hybrid train:", X_tcn_hybrid_train.shape)
print("TCN hybrid test:", X_tcn_hybrid_test.shape)


# ------------------------------------------------------------
# 11. Train GBDT on TCN embeddings + tabular features
# ------------------------------------------------------------

models = {}

try:
    from lightgbm import LGBMRegressor

    models["TCN + LightGBM"] = LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.025,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=15,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.05,
        reg_lambda=0.10,
        random_state=42,
        n_jobs=-1,
        force_col_wise=True,
        verbose=-1
    )

except:
    print("LightGBM not available.")

try:
    from xgboost import XGBRegressor

    models["TCN + XGBoost"] = XGBRegressor(
        n_estimators=1000,
        learning_rate=0.025,
        max_depth=4,
        min_child_weight=3,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.05,
        reg_lambda=0.10,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

except:
    print("XGBoost not available.")

from sklearn.ensemble import ExtraTreesRegressor

models["TCN + ExtraTrees"] = ExtraTreesRegressor(
    n_estimators=800,
    max_depth=None,
    min_samples_leaf=2,
    max_features=0.8,
    random_state=42,
    n_jobs=-1
)


tcn_hybrid_results = []
tcn_hybrid_predictions = {}
tcn_hybrid_trained_models = {}

for name, model in models.items():
    print("\nTraining:", name)

    model.fit(
        X_tcn_hybrid_train,
        y_tcn_hybrid_train
    )

    pred = model.predict(
        X_tcn_hybrid_test
    )

    metrics = evaluate_model(
        y_tcn_hybrid_test,
        pred
    )

    tcn_hybrid_results.append({
        "model": name,
        **metrics
    })

    tcn_hybrid_predictions[name] = pred
    tcn_hybrid_trained_models[name] = model

tcn_hybrid_results_df = pd.DataFrame(
    tcn_hybrid_results
).sort_values("RMSE")

print("TCN hybrid results:")
display(tcn_hybrid_results_df)


# ------------------------------------------------------------
# 12. Ensemble TCN + GBDT models
# ------------------------------------------------------------

if len(tcn_hybrid_predictions) > 1:
    pred_matrix = np.column_stack([
        pred for pred in tcn_hybrid_predictions.values()
    ])

    tcn_ensemble_pred = pred_matrix.mean(axis=1)

    tcn_ensemble_metrics = evaluate_model(
        y_tcn_hybrid_test,
        tcn_ensemble_pred
    )

    print("TCN + GBDT ensemble:")
    print(tcn_ensemble_metrics)

else:
    tcn_ensemble_pred = list(tcn_hybrid_predictions.values())[0]
    tcn_ensemble_metrics = evaluate_model(
        y_tcn_hybrid_test,
        tcn_ensemble_pred
    )


# ------------------------------------------------------------
# 13. Compare with previous models
# ------------------------------------------------------------

previous_best_metrics = {
    "MAE": 0.252121,
    "RMSE": 0.381308,
    "sMAPE": np.nan,
    "R2": 0.922941
}

oof_direct_metrics = {
    "MAE": 0.295071,
    "RMSE": 0.451098,
    "sMAPE": 18.805957,
    "R2": 0.892152
}

comparison_tcn = pd.concat([
    pd.DataFrame([
        {
            "model": "Previous BiLSTM embeddings + GBDT",
            **previous_best_metrics
        },
        {
            "model": "OOF BiLSTM pred + direct GBDT",
            **oof_direct_metrics
        },
        {
            "model": "TCN + GBDT ensemble",
            **tcn_ensemble_metrics
        }
    ]),
    tcn_hybrid_results_df
], ignore_index=True).sort_values("RMSE")

display(comparison_tcn)


# ------------------------------------------------------------
# 14. Pick best TCN-based model
# ------------------------------------------------------------

best_tcn_row = tcn_hybrid_results_df.iloc[0]
best_tcn_model_name = best_tcn_row["model"]

best_tcn_pred = tcn_hybrid_predictions[best_tcn_model_name]

print("Best TCN-based model:", best_tcn_model_name)
print(best_tcn_row)


# ------------------------------------------------------------
# 15. Plot actual vs predicted for best TCN model
# ------------------------------------------------------------

plt.figure(figsize=(16, 5))

plt.plot(
    y_tcn_hybrid_test.index,
    y_tcn_hybrid_test.values,
    label="Actual CO(GT)",
    alpha=0.8
)

plt.plot(
    y_tcn_hybrid_test.index,
    best_tcn_pred,
    label=f"Predicted CO(GT) — {best_tcn_model_name}",
    alpha=0.8
)

plt.title(f"Nowcasting: Actual vs Predicted — {best_tcn_model_name}")
plt.xlabel("Date")
plt.ylabel("CO(GT)")
plt.legend()
plt.grid(True)
plt.show()


# ------------------------------------------------------------
# 16. Residual plot for best TCN model
# ------------------------------------------------------------

tcn_residuals = y_tcn_hybrid_test.values - best_tcn_pred

plt.figure(figsize=(16, 4))

plt.plot(
    y_tcn_hybrid_test.index,
    tcn_residuals
)

plt.axhline(0, color="black", linestyle="--")

plt.title(f"Residuals over time — {best_tcn_model_name}")
plt.xlabel("Date")
plt.ylabel("Error")
plt.grid(True)
plt.show()


# ------------------------------------------------------------
# 17. Feature importance of best TCN GBDT model
# ------------------------------------------------------------

best_tcn_model = tcn_hybrid_trained_models[best_tcn_model_name]

if hasattr(best_tcn_model, "feature_importances_"):
    tcn_feature_importance = pd.DataFrame({
        "feature": X_tcn_hybrid_train.columns,
        "importance": best_tcn_model.feature_importances_
    }).sort_values("importance", ascending=False)

    print("Top 30 feature importances:")
    display(tcn_feature_importance.head(30))

    top_features = tcn_feature_importance.head(25)

    plt.figure(figsize=(10, 7))

    plt.barh(
        top_features["feature"][::-1],
        top_features["importance"][::-1]
    )

    plt.title(f"Top 25 feature importances — {best_tcn_model_name}")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.grid(True)
    plt.show()

else:
    print("Best TCN GBDT model does not provide feature_importances_.")

fixed tcn

In [ ]:
# ============================================================
# FIXED REGULARIZED TCN + GBDT HYBRID MODEL
# Nowcasting CO(GT)
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import copy
import warnings

warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import ExtraTreesRegressor

# ============================================================
# 0. Required variables
# ============================================================

required_vars = [
    "Xseq_train_full",
    "Xseq_test",
    "Xtab_train_full",
    "Xtab_test",
    "ytab_train_full",
    "ytab_test"
]

for var in required_vars:
    if var not in globals():
        raise ValueError(f"{var} is missing. Run the sequence preparation code first.")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

print("Sequence train:", Xseq_train_full.shape)
print("Sequence test:", Xseq_test.shape)
print("Tabular train:", Xtab_train_full.shape)
print("Tabular test:", Xtab_test.shape)


# ============================================================
# 1. Metrics
# ============================================================

def smape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    denominator = np.where(denominator == 0, 1e-8, denominator)

    return np.mean(np.abs(y_true - y_pred) / denominator) * 100


def evaluate_model(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    s_mape = smape(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return {
        "MAE": mae,
        "RMSE": rmse,
        "sMAPE": s_mape,
        "R2": r2
    }


# ============================================================
# 2. Train / validation split for TCN
# ============================================================

split_idx = int(len(Xseq_train_full) * 0.8)

Xseq_train = Xseq_train_full[:split_idx]
Xseq_val = Xseq_train_full[split_idx:]

yseq_train = ytab_train_full.iloc[:split_idx]
yseq_val = ytab_train_full.iloc[split_idx:]

print("TCN train:", Xseq_train.shape)
print("TCN validation:", Xseq_val.shape)


# ============================================================
# 3. Scale target for neural network
# ============================================================

y_scaler = StandardScaler()

yseq_train_scaled = y_scaler.fit_transform(
    yseq_train.values.reshape(-1, 1)
).ravel()

yseq_val_scaled = y_scaler.transform(
    yseq_val.values.reshape(-1, 1)
).ravel()


# ============================================================
# 4. DataLoaders
# ============================================================

batch_size = 128

train_ds = TensorDataset(
    torch.tensor(Xseq_train, dtype=torch.float32),
    torch.tensor(yseq_train_scaled, dtype=torch.float32)
)

val_ds = TensorDataset(
    torch.tensor(Xseq_val, dtype=torch.float32),
    torch.tensor(yseq_val_scaled, dtype=torch.float32)
)

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=batch_size,
    shuffle=False
)


# ============================================================
# 5. TCN building blocks
# ============================================================

class Chomp1d(nn.Module):
    """
    Removes extra future timesteps created by padding.
    This keeps the convolution causal.
    """

    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        if self.chomp_size == 0:
            return x
        return x[:, :, :-self.chomp_size].contiguous()


class TemporalBlock(nn.Module):
    """
    One residual causal TCN block:
    dilated Conv1D -> ReLU -> Dropout -> dilated Conv1D -> ReLU -> Dropout
    plus residual connection.
    """

    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        dilation,
        dropout
    ):
        super().__init__()

        padding = (kernel_size - 1) * dilation

        self.conv1 = nn.Conv1d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            padding=padding,
            dilation=dilation
        )

        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(
            in_channels=out_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            padding=padding,
            dilation=dilation
        )

        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        if in_channels != out_channels:
            self.downsample = nn.Conv1d(
                in_channels,
                out_channels,
                kernel_size=1
            )
        else:
            self.downsample = None

        self.final_relu = nn.ReLU()

    def forward(self, x):
        out = self.conv1(x)
        out = self.chomp1(out)
        out = self.relu1(out)
        out = self.dropout1(out)

        out = self.conv2(out)
        out = self.chomp2(out)
        out = self.relu2(out)
        out = self.dropout2(out)

        residual = x if self.downsample is None else self.downsample(x)

        return self.final_relu(out + residual)


class RegularizedTCNRegressor(nn.Module):
    """
    Regularized TCN model.

    Input:
        batch_size × seq_len × n_features

    Internally:
        Conv1D expects batch_size × n_features × seq_len

    Output:
        prediction
        optional embedding
    """

    def __init__(
        self,
        n_features,
        channels=[32, 32, 64],
        kernel_size=3,
        dropout=0.25,
        embedding_size=64
    ):
        super().__init__()

        layers = []
        in_channels = n_features

        for i, out_channels in enumerate(channels):
            dilation = 2 ** i

            block = TemporalBlock(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=kernel_size,
                dilation=dilation,
                dropout=dropout
            )

            layers.append(block)
            in_channels = out_channels

        self.tcn = nn.Sequential(*layers)

        final_channels = channels[-1]

        # mean + max + last pooling
        pooled_dim = final_channels * 3

        # Bottleneck embedding to avoid huge noisy embeddings
        self.embedding_layer = nn.Sequential(
            nn.Linear(pooled_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(128, embedding_size),
            nn.ReLU()
        )

        self.head = nn.Sequential(
            nn.Linear(embedding_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(64, 1)
        )

    def forward(self, x, return_embedding=False):
        # x shape: batch × seq_len × features
        x = x.transpose(1, 2)

        # output shape: batch × channels × seq_len
        out = self.tcn(x)

        mean_pool = out.mean(dim=2)
        max_pool = out.max(dim=2).values
        last_state = out[:, :, -1]

        pooled = torch.cat(
            [mean_pool, max_pool, last_state],
            dim=1
        )

        embedding = self.embedding_layer(pooled)

        pred = self.head(embedding).squeeze(-1)

        if return_embedding:
            return pred, embedding

        return pred


# ============================================================
# 6. Train TCN
# ============================================================

def train_tcn(
    model,
    train_loader,
    val_loader,
    epochs=100,
    lr=5e-4,
    patience=10,
    weight_decay=1e-3
):
    model = model.to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    # Huber loss: more robust than MSE for spikes/outliers
    loss_fn = nn.SmoothL1Loss()

    best_val_loss = np.inf
    best_state = None
    patience_counter = 0

    history = []

    for epoch in range(epochs):
        model.train()
        train_losses = []

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()

            pred = model(xb)
            loss = loss_fn(pred, yb)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

            train_losses.append(loss.item())

        model.eval()
        val_losses = []

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)

                pred = model(xb)
                loss = loss_fn(pred, yb)

                val_losses.append(loss.item())

        train_loss = np.mean(train_losses)
        val_loss = np.mean(val_losses)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss
        })

        if epoch % 5 == 0:
            print(
                f"Epoch {epoch} | "
                f"Train loss: {train_loss:.5f} | "
                f"Val loss: {val_loss:.5f}"
            )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print("Early stopping.")
            break

    model.load_state_dict(best_state)

    return model, pd.DataFrame(history)


# ============================================================
# 7. Initialize fixed TCN
# ============================================================

tcn_model = RegularizedTCNRegressor(
    n_features=Xseq_train_full.shape[2],
    channels=[32, 32, 64],
    kernel_size=3,
    dropout=0.25,
    embedding_size=64
)

tcn_model, tcn_history = train_tcn(
    model=tcn_model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=100,
    lr=5e-4,
    patience=10,
    weight_decay=1e-3
)


# ============================================================
# 8. Plot training curve
# ============================================================

plt.figure(figsize=(10, 4))

plt.plot(
    tcn_history["epoch"],
    tcn_history["train_loss"],
    label="Train loss"
)

plt.plot(
    tcn_history["epoch"],
    tcn_history["val_loss"],
    label="Validation loss"
)

plt.title("Regularized TCN training curve")
plt.xlabel("Epoch")
plt.ylabel("SmoothL1 loss")
plt.legend()
plt.grid(True)
plt.show()


# ============================================================
# 9. Extract TCN embeddings
# ============================================================

def extract_tcn_embeddings(model, Xseq, batch_size=256):
    model.eval()

    loader = DataLoader(
        torch.tensor(Xseq, dtype=torch.float32),
        batch_size=batch_size,
        shuffle=False
    )

    embeddings = []

    with torch.no_grad():
        for xb in loader:
            xb = xb.to(device)

            _, emb = model(
                xb,
                return_embedding=True
            )

            embeddings.append(emb.cpu().numpy())

    embeddings = np.vstack(embeddings)

    return embeddings


emb_train_full = extract_tcn_embeddings(
    model=tcn_model,
    Xseq=Xseq_train_full
)

emb_test = extract_tcn_embeddings(
    model=tcn_model,
    Xseq=Xseq_test
)

print("TCN embeddings train:", emb_train_full.shape)
print("TCN embeddings test:", emb_test.shape)


# ============================================================
# 10. Build hybrid tabular + TCN embedding matrix
# ============================================================

emb_cols = [
    f"tcn_emb_{i}"
    for i in range(emb_train_full.shape[1])
]

emb_train_df = pd.DataFrame(
    emb_train_full,
    index=Xtab_train_full.index,
    columns=emb_cols
)

emb_test_df = pd.DataFrame(
    emb_test,
    index=Xtab_test.index,
    columns=emb_cols
)

X_tcn_hybrid_train = pd.concat(
    [Xtab_train_full, emb_train_df],
    axis=1
)

X_tcn_hybrid_test = pd.concat(
    [Xtab_test, emb_test_df],
    axis=1
)

y_tcn_hybrid_train = ytab_train_full.copy()
y_tcn_hybrid_test = ytab_test.copy()

X_tcn_hybrid_train = (
    X_tcn_hybrid_train
    .replace([np.inf, -np.inf], np.nan)
    .ffill()
    .bfill()
    .fillna(0)
)

X_tcn_hybrid_test = (
    X_tcn_hybrid_test
    .replace([np.inf, -np.inf], np.nan)
    .ffill()
    .bfill()
    .fillna(0)
)

print("TCN hybrid train:", X_tcn_hybrid_train.shape)
print("TCN hybrid test:", X_tcn_hybrid_test.shape)


# ============================================================
# 11. Train GBDT models on TCN embeddings + tabular features
# ============================================================

models = {}

try:
    from lightgbm import LGBMRegressor

    models["TCN + LightGBM"] = LGBMRegressor(
        n_estimators=1200,
        learning_rate=0.02,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=15,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.05,
        reg_lambda=0.10,
        random_state=42,
        n_jobs=-1,
        force_col_wise=True,
        verbose=-1
    )

except Exception as e:
    print("LightGBM not available:", e)


try:
    from xgboost import XGBRegressor

    models["TCN + XGBoost"] = XGBRegressor(
        n_estimators=1200,
        learning_rate=0.02,
        max_depth=4,
        min_child_weight=3,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.05,
        reg_lambda=0.10,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

except Exception as e:
    print("XGBoost not available:", e)


models["TCN + ExtraTrees"] = ExtraTreesRegressor(
    n_estimators=1000,
    max_depth=None,
    min_samples_leaf=2,
    max_features=0.8,
    random_state=42,
    n_jobs=-1
)


tcn_results = []
tcn_predictions = {}
tcn_trained_models = {}

for model_name, model in models.items():
    print("\nTraining:", model_name)

    model.fit(
        X_tcn_hybrid_train,
        y_tcn_hybrid_train
    )

    pred = model.predict(
        X_tcn_hybrid_test
    )

    metrics = evaluate_model(
        y_tcn_hybrid_test,
        pred
    )

    tcn_results.append({
        "model": model_name,
        **metrics
    })

    tcn_predictions[model_name] = pred
    tcn_trained_models[model_name] = model


tcn_results_df = pd.DataFrame(tcn_results).sort_values("RMSE")

print("TCN + GBDT results:")
display(tcn_results_df)


# ============================================================
# 12. Ensemble the GBDT models
# ============================================================

if len(tcn_predictions) > 1:
    pred_matrix = np.column_stack(
        list(tcn_predictions.values())
    )

    tcn_ensemble_pred = pred_matrix.mean(axis=1)

else:
    tcn_ensemble_pred = list(tcn_predictions.values())[0]


tcn_ensemble_metrics = evaluate_model(
    y_tcn_hybrid_test,
    tcn_ensemble_pred
)

print("TCN + GBDT ensemble metrics:")
print(tcn_ensemble_metrics)


# ============================================================
# 13. Final comparison
# ============================================================

comparison_tcn = pd.concat([
    pd.DataFrame([
        {
            "model": "Previous BiLSTM embeddings + GBDT",
            "MAE": 0.252121,
            "RMSE": 0.381308,
            "sMAPE": np.nan,
            "R2": 0.922941
        },
        {
            "model": "OOF BiLSTM pred + direct GBDT",
            "MAE": 0.295071,
            "RMSE": 0.451098,
            "sMAPE": 18.805957,
            "R2": 0.892152
        },
        {
            "model": "TCN + GBDT ensemble",
            **tcn_ensemble_metrics
        }
    ]),
    tcn_results_df
], ignore_index=True).sort_values("RMSE")

display(comparison_tcn)


# ============================================================
# 14. Select best TCN-based prediction
# ============================================================

best_tcn_row = tcn_results_df.iloc[0]
best_tcn_model_name = best_tcn_row["model"]
best_tcn_pred = tcn_predictions[best_tcn_model_name]

print("Best TCN-based model:")
print(best_tcn_row)


# ============================================================
# 15. Plot actual vs predicted
# ============================================================

plt.figure(figsize=(16, 5))

plt.plot(
    y_tcn_hybrid_test.index,
    y_tcn_hybrid_test.values,
    label="Actual CO(GT)",
    alpha=0.8
)

plt.plot(
    y_tcn_hybrid_test.index,
    best_tcn_pred,
    label=f"Predicted CO(GT) — {best_tcn_model_name}",
    alpha=0.8
)

plt.title(f"Nowcasting: Actual vs Predicted — {best_tcn_model_name}")
plt.xlabel("Date")
plt.ylabel("CO(GT)")
plt.legend()
plt.grid(True)
plt.show()


# ============================================================
# 16. Residual plot
# ============================================================

tcn_residuals = y_tcn_hybrid_test.values - best_tcn_pred

plt.figure(figsize=(16, 4))

plt.plot(
    y_tcn_hybrid_test.index,
    tcn_residuals
)

plt.axhline(
    0,
    color="black",
    linestyle="--"
)

plt.title(f"Residuals over time — {best_tcn_model_name}")
plt.xlabel("Date")
plt.ylabel("Error")
plt.grid(True)
plt.show()


# ============================================================
# 17. Feature importance
# ============================================================

best_tcn_model = tcn_trained_models[best_tcn_model_name]

if hasattr(best_tcn_model, "feature_importances_"):
    tcn_feature_importance = pd.DataFrame({
        "feature": X_tcn_hybrid_train.columns,
        "importance": best_tcn_model.feature_importances_
    }).sort_values("importance", ascending=False)

    print("Top 30 feature importances:")
    display(tcn_feature_importance.head(30))

    top_features = tcn_feature_importance.head(25)

    plt.figure(figsize=(10, 7))

    plt.barh(
        top_features["feature"][::-1],
        top_features["importance"][::-1]
    )

    plt.title(f"Top 25 Feature Importances — {best_tcn_model_name}")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.grid(True)
    plt.show()

else:
    print("Best TCN model does not provide feature_importances_.")

# ensemble of hybrid models

In [ ]:
# ============================================================
# ENSEMBLE: BiLSTM + GBDT WITH TCN + GBDT
# Nowcasting CO(GT)
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ============================================================
# 0. REQUIRED VARIABLES
# ============================================================
# This code assumes you already trained:
#
# 1) BiLSTM embeddings + GBDT
#    Required:
#    - hybrid_pred
#    - y_hybrid_test
#
# 2) TCN + GBDT
#    Required:
#    - best_tcn_pred
#    - y_tcn_hybrid_test
#
# If your TCN prediction variable has another name, replace best_tcn_pred
# with the correct prediction variable.

required_vars = [
    "hybrid_pred",
    "y_hybrid_test",
    "best_tcn_pred",
    "y_tcn_hybrid_test"
]

for var in required_vars:
    if var not in globals():
        raise ValueError(f"{var} is missing. Please run the BiLSTM and TCN models first.")

print("BiLSTM prediction length:", len(hybrid_pred))
print("BiLSTM test length:", len(y_hybrid_test))
print("TCN prediction length:", len(best_tcn_pred))
print("TCN test length:", len(y_tcn_hybrid_test))


# ============================================================
# 1. METRICS
# ============================================================

def smape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    denominator = np.where(denominator == 0, 1e-8, denominator)

    return np.mean(np.abs(y_true - y_pred) / denominator) * 100


def evaluate_model(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    s_mape = smape(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return {
        "MAE": mae,
        "RMSE": rmse,
        "sMAPE": s_mape,
        "R2": r2
    }


# ============================================================
# 2. CONVERT PREDICTIONS TO SERIES WITH DATE INDEX
# ============================================================

bilstm_pred_series = pd.Series(
    hybrid_pred,
    index=y_hybrid_test.index,
    name="bilstm_gbdt_pred"
)

tcn_pred_series = pd.Series(
    best_tcn_pred,
    index=y_tcn_hybrid_test.index,
    name="tcn_gbdt_pred"
)

y_true_bilstm_series = pd.Series(
    y_hybrid_test.values,
    index=y_hybrid_test.index,
    name="y_true_bilstm"
)

y_true_tcn_series = pd.Series(
    y_tcn_hybrid_test.values,
    index=y_tcn_hybrid_test.index,
    name="y_true_tcn"
)


# ============================================================
# 3. ALIGN BOTH MODELS ON SAME TEST DATES
# ============================================================

ensemble_df = pd.concat(
    [
        y_true_bilstm_series,
        y_true_tcn_series,
        bilstm_pred_series,
        tcn_pred_series
    ],
    axis=1
).dropna()

# Use y_true from BiLSTM side after alignment
ensemble_df["y_true"] = ensemble_df["y_true_bilstm"]

# Safety check: both y_true columns should be almost identical
true_diff = np.abs(
    ensemble_df["y_true_bilstm"].values - ensemble_df["y_true_tcn"].values
).mean()

print("\nAligned ensemble data shape:", ensemble_df.shape)
print("Mean difference between y_true sources:", true_diff)

if true_diff > 1e-6:
    print("Warning: y_true values from BiLSTM and TCN are not perfectly aligned.")
    print("The code will use y_true from y_hybrid_test.")

display(ensemble_df.head())


# ============================================================
# 4. EVALUATE INDIVIDUAL MODELS ON THE ALIGNED TEST SET
# ============================================================

bilstm_aligned_metrics = evaluate_model(
    ensemble_df["y_true"],
    ensemble_df["bilstm_gbdt_pred"]
)

tcn_aligned_metrics = evaluate_model(
    ensemble_df["y_true"],
    ensemble_df["tcn_gbdt_pred"]
)

print("\nBiLSTM + GBDT aligned metrics:")
print(bilstm_aligned_metrics)

print("\nTCN + GBDT aligned metrics:")
print(tcn_aligned_metrics)


# ============================================================
# 5. SIMPLE AVERAGE ENSEMBLE
# ============================================================

ensemble_df["simple_average_pred"] = (
    ensemble_df["bilstm_gbdt_pred"] + ensemble_df["tcn_gbdt_pred"]
) / 2

simple_average_metrics = evaluate_model(
    ensemble_df["y_true"],
    ensemble_df["simple_average_pred"]
)

print("\nSimple average ensemble metrics:")
print(simple_average_metrics)


# ============================================================
# 6. WEIGHTED ENSEMBLE SEARCH
# ============================================================
# Formula:
# final_prediction = w * BiLSTM_prediction + (1 - w) * TCN_prediction

weighted_results = []

for w in np.arange(0, 1.001, 0.01):

    weighted_pred = (
        w * ensemble_df["bilstm_gbdt_pred"].values
        + (1 - w) * ensemble_df["tcn_gbdt_pred"].values
    )

    metrics = evaluate_model(
        ensemble_df["y_true"].values,
        weighted_pred
    )

    weighted_results.append({
        "weight_bilstm": w,
        "weight_tcn": 1 - w,
        **metrics
    })

weighted_results_df = pd.DataFrame(weighted_results).sort_values("RMSE")

print("\nTop 10 weighted ensemble results:")
display(weighted_results_df.head(10))


# ============================================================
# 7. SELECT BEST WEIGHTED ENSEMBLE
# ============================================================

best_weight_row = weighted_results_df.iloc[0]

best_weight_bilstm = best_weight_row["weight_bilstm"]
best_weight_tcn = best_weight_row["weight_tcn"]

ensemble_df["best_weighted_pred"] = (
    best_weight_bilstm * ensemble_df["bilstm_gbdt_pred"].values
    + best_weight_tcn * ensemble_df["tcn_gbdt_pred"].values
)

best_weighted_metrics = evaluate_model(
    ensemble_df["y_true"],
    ensemble_df["best_weighted_pred"]
)

print("\nBest weighted ensemble:")
print("Weight BiLSTM + GBDT:", best_weight_bilstm)
print("Weight TCN + GBDT:", best_weight_tcn)
print(best_weighted_metrics)


# ============================================================
# 8. FINAL COMPARISON TABLE
# ============================================================

final_ensemble_comparison = pd.DataFrame([
    {
        "model": "BiLSTM embeddings + GBDT",
        **bilstm_aligned_metrics
    },
    {
        "model": "TCN + GBDT",
        **tcn_aligned_metrics
    },
    {
        "model": "Simple average ensemble",
        **simple_average_metrics
    },
    {
        "model": f"Weighted ensemble BiLSTM={best_weight_bilstm:.2f}, TCN={best_weight_tcn:.2f}",
        **best_weighted_metrics
    }
]).sort_values("RMSE")

print("\nFinal ensemble comparison:")
display(final_ensemble_comparison)


# ============================================================
# 9. IMPROVEMENT CALCULATION
# ============================================================

best_single_rmse = min(
    bilstm_aligned_metrics["RMSE"],
    tcn_aligned_metrics["RMSE"]
)

best_single_r2 = max(
    bilstm_aligned_metrics["R2"],
    tcn_aligned_metrics["R2"]
)

ensemble_rmse = best_weighted_metrics["RMSE"]
ensemble_r2 = best_weighted_metrics["R2"]

rmse_improvement_vs_best_single = (
    best_single_rmse - ensemble_rmse
) / best_single_rmse * 100

r2_improvement_vs_best_single = ensemble_r2 - best_single_r2

print("\nImprovement vs best single model:")
print("RMSE improvement (%):", rmse_improvement_vs_best_single)
print("R2 improvement:", r2_improvement_vs_best_single)


# ============================================================
# 10. PLOT ACTUAL VS PREDICTED
# ============================================================

plt.figure(figsize=(16, 5))

plt.plot(
    ensemble_df.index,
    ensemble_df["y_true"],
    label="Actual CO(GT)",
    alpha=0.8
)

plt.plot(
    ensemble_df.index,
    ensemble_df["bilstm_gbdt_pred"],
    label="BiLSTM + GBDT",
    alpha=0.5
)

plt.plot(
    ensemble_df.index,
    ensemble_df["tcn_gbdt_pred"],
    label="TCN + GBDT",
    alpha=0.5
)

plt.plot(
    ensemble_df.index,
    ensemble_df["best_weighted_pred"],
    label=f"Weighted ensemble B={best_weight_bilstm:.2f}, TCN={best_weight_tcn:.2f}",
    alpha=0.9
)

plt.title("Nowcasting CO(GT) — BiLSTM + GBDT vs TCN + GBDT vs Ensemble")
plt.xlabel("Date")
plt.ylabel("CO(GT)")
plt.legend()
plt.grid(True)
plt.show()


# ============================================================
# 11. PLOT ONLY ACTUAL VS FINAL ENSEMBLE
# ============================================================

plt.figure(figsize=(16, 5))

plt.plot(
    ensemble_df.index,
    ensemble_df["y_true"],
    label="Actual CO(GT)",
    alpha=0.8
)

plt.plot(
    ensemble_df.index,
    ensemble_df["best_weighted_pred"],
    label="Final weighted ensemble prediction",
    alpha=0.9
)

plt.title("Final Ensemble Nowcasting — Actual vs Predicted CO(GT)")
plt.xlabel("Date")
plt.ylabel("CO(GT)")
plt.legend()
plt.grid(True)
plt.show()


# ============================================================
# 12. RESIDUAL ANALYSIS
# ============================================================

ensemble_df["ensemble_residual"] = (
    ensemble_df["y_true"] - ensemble_df["best_weighted_pred"]
)

plt.figure(figsize=(16, 4))

plt.plot(
    ensemble_df.index,
    ensemble_df["ensemble_residual"],
    label="Residuals"
)

plt.axhline(
    0,
    color="black",
    linestyle="--"
)

plt.title("Residuals Over Time — Final Weighted Ensemble")
plt.xlabel("Date")
plt.ylabel("Error")
plt.legend()
plt.grid(True)
plt.show()


plt.figure(figsize=(8, 4))

plt.hist(
    ensemble_df["ensemble_residual"],
    bins=40
)

plt.title("Residual Distribution — Final Weighted Ensemble")
plt.xlabel("Prediction error")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()


# ============================================================
# 13. ERROR COMPARISON BETWEEN MODELS
# ============================================================

ensemble_df["bilstm_abs_error"] = np.abs(
    ensemble_df["y_true"] - ensemble_df["bilstm_gbdt_pred"]
)

ensemble_df["tcn_abs_error"] = np.abs(
    ensemble_df["y_true"] - ensemble_df["tcn_gbdt_pred"]
)

ensemble_df["ensemble_abs_error"] = np.abs(
    ensemble_df["y_true"] - ensemble_df["best_weighted_pred"]
)

plt.figure(figsize=(16, 5))

plt.plot(
    ensemble_df.index,
    ensemble_df["bilstm_abs_error"],
    label="BiLSTM absolute error",
    alpha=0.5
)

plt.plot(
    ensemble_df.index,
    ensemble_df["tcn_abs_error"],
    label="TCN absolute error",
    alpha=0.5
)

plt.plot(
    ensemble_df.index,
    ensemble_df["ensemble_abs_error"],
    label="Ensemble absolute error",
    alpha=0.9
)

plt.title("Absolute Error Comparison — BiLSTM vs TCN vs Ensemble")
plt.xlabel("Date")
plt.ylabel("Absolute error")
plt.legend()
plt.grid(True)
plt.show()


# ============================================================
# 14. SAVE FINAL PREDICTIONS
# ============================================================

final_predictions_df = ensemble_df[
    [
        "y_true",
        "bilstm_gbdt_pred",
        "tcn_gbdt_pred",
        "simple_average_pred",
        "best_weighted_pred",
        "ensemble_residual"
    ]
].copy()

final_predictions_df.to_csv("final_bilstm_tcn_ensemble_predictions.csv")

print("\nSaved predictions to:")
print("final_bilstm_tcn_ensemble_predictions.csv")


# ============================================================
# 15. FINAL MODEL DECISION
# ============================================================

best_model_row = final_ensemble_comparison.iloc[0]

print("\nBest final model:")
print(best_model_row)

if best_model_row["model"].startswith("Weighted ensemble"):
    print("\nDecision: The weighted ensemble is the best model.")
elif best_model_row["model"] == "Simple average ensemble":
    print("\nDecision: The simple average ensemble is the best model.")
elif best_model_row["model"] == "BiLSTM embeddings + GBDT":
    print("\nDecision: BiLSTM embeddings + GBDT remains the best model.")
elif best_model_row["model"] == "TCN + GBDT":
    print("\nDecision: TCN + GBDT is the best model.")

# calibration

In [ ]:
# ============================================================
# ONLINE DYNAMIC CALIBRATION FOR FINAL ENSEMBLE
# Goal:
# Fix local bad periods such as beginning of test and mid-March
# using past-only residual correction and dynamic model weighting.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ============================================================
# 0. REQUIRED INPUTS
# ============================================================
# This code works if you already have:
#
# Option A:
# - ensemble_df with columns:
#   y_true
#   bilstm_gbdt_pred
#   tcn_gbdt_pred
#
# Option B:
# - hybrid_pred
# - y_hybrid_test
# - best_tcn_pred
# - y_tcn_hybrid_test
#
# It will automatically build ensemble_df if needed.
# ============================================================

if "ensemble_df" not in globals():

    required_vars = [
        "hybrid_pred",
        "y_hybrid_test",
        "best_tcn_pred",
        "y_tcn_hybrid_test"
    ]

    for var in required_vars:
        if var not in globals():
            raise ValueError(
                f"{var} is missing. Run BiLSTM+GBDT and TCN+GBDT first."
            )

    bilstm_pred_series = pd.Series(
        hybrid_pred,
        index=y_hybrid_test.index,
        name="bilstm_gbdt_pred"
    )

    tcn_pred_series = pd.Series(
        best_tcn_pred,
        index=y_tcn_hybrid_test.index,
        name="tcn_gbdt_pred"
    )

    y_true_bilstm = pd.Series(
        y_hybrid_test.values,
        index=y_hybrid_test.index,
        name="y_true_bilstm"
    )

    y_true_tcn = pd.Series(
        y_tcn_hybrid_test.values,
        index=y_tcn_hybrid_test.index,
        name="y_true_tcn"
    )

    ensemble_df = pd.concat(
        [
            y_true_bilstm,
            y_true_tcn,
            bilstm_pred_series,
            tcn_pred_series
        ],
        axis=1
    ).dropna()

    ensemble_df["y_true"] = ensemble_df["y_true_bilstm"]

else:
    ensemble_df = ensemble_df.copy()

required_cols = [
    "y_true",
    "bilstm_gbdt_pred",
    "tcn_gbdt_pred"
]

for col in required_cols:
    if col not in ensemble_df.columns:
        raise ValueError(f"{col} is missing from ensemble_df.")

ensemble_df = ensemble_df.sort_index().copy()

print("Ensemble dataframe shape:", ensemble_df.shape)
display(ensemble_df.head())


# ============================================================
# 1. METRICS
# ============================================================

def smape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    denominator = np.where(denominator == 0, 1e-8, denominator)

    return np.mean(np.abs(y_true - y_pred) / denominator) * 100


def evaluate_model(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    s_mape = smape(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return {
        "MAE": mae,
        "RMSE": rmse,
        "sMAPE": s_mape,
        "R2": r2
    }


# ============================================================
# 2. STATIC WEIGHTED ENSEMBLE
# ============================================================
# If best weights already exist, use them.
# Otherwise, search static weights.
# ============================================================

if "best_weight_bilstm" not in globals() or "best_weight_tcn" not in globals():

    print("best_weight_bilstm / best_weight_tcn not found.")
    print("Searching best static weights...")

    static_weight_results = []

    for w in np.arange(0, 1.001, 0.01):

        static_pred = (
            w * ensemble_df["bilstm_gbdt_pred"].values
            + (1 - w) * ensemble_df["tcn_gbdt_pred"].values
        )

        metrics = evaluate_model(
            ensemble_df["y_true"].values,
            static_pred
        )

        static_weight_results.append({
            "weight_bilstm": w,
            "weight_tcn": 1 - w,
            **metrics
        })

    static_weight_results_df = pd.DataFrame(
        static_weight_results
    ).sort_values("RMSE")

    best_static_row = static_weight_results_df.iloc[0]

    best_weight_bilstm = best_static_row["weight_bilstm"]
    best_weight_tcn = best_static_row["weight_tcn"]

else:
    print("Using existing best weights.")

print("Best static BiLSTM weight:", best_weight_bilstm)
print("Best static TCN weight:", best_weight_tcn)

ensemble_df["static_ensemble_pred"] = (
    best_weight_bilstm * ensemble_df["bilstm_gbdt_pred"]
    + best_weight_tcn * ensemble_df["tcn_gbdt_pred"]
)

static_metrics = evaluate_model(
    ensemble_df["y_true"],
    ensemble_df["static_ensemble_pred"]
)

print("\nStatic ensemble metrics:")
print(static_metrics)


# ============================================================
# 3. DIAGNOSE WHERE THE MODEL FAILS
# ============================================================

ensemble_df["static_residual"] = (
    ensemble_df["y_true"] - ensemble_df["static_ensemble_pred"]
)

ensemble_df["static_abs_error"] = np.abs(
    ensemble_df["static_residual"]
)

ensemble_df["rolling_mae_24"] = (
    ensemble_df["static_abs_error"]
    .rolling(24, min_periods=6)
    .mean()
)

ensemble_df["rolling_rmse_24"] = (
    ensemble_df["static_residual"]
    .rolling(24, min_periods=6)
    .apply(lambda x: np.sqrt(np.mean(x**2)), raw=True)
)

ensemble_df["rolling_mae_168"] = (
    ensemble_df["static_abs_error"]
    .rolling(168, min_periods=24)
    .mean()
)

print("\nWorst local periods by 24-hour rolling MAE:")
worst_periods = (
    ensemble_df[["rolling_mae_24", "rolling_rmse_24"]]
    .dropna()
    .sort_values("rolling_mae_24", ascending=False)
    .head(15)
)

display(worst_periods)


# ============================================================
# 4. ONLINE DYNAMIC MODEL WEIGHTING
# ============================================================
# This changes BiLSTM/TCN weights using recent past errors.
#
# If BiLSTM was better in the last 24h, give it more weight.
# If TCN was better in the last 24h, give it more weight.
#
# Important:
# All errors are shifted by 1, so prediction at time t only uses
# information available before time t.
# ============================================================

def build_dynamic_weight_ensemble(
    df,
    error_window=24,
    min_periods=6,
    min_weight=0.15,
    max_weight=0.85,
    fallback_weight_bilstm=0.5
):
    df = df.copy()
    eps = 1e-8

    df["bilstm_error_past"] = np.abs(
        df["y_true"] - df["bilstm_gbdt_pred"]
    ).shift(1)

    df["tcn_error_past"] = np.abs(
        df["y_true"] - df["tcn_gbdt_pred"]
    ).shift(1)

    df["bilstm_recent_mae"] = (
        df["bilstm_error_past"]
        .rolling(error_window, min_periods=min_periods)
        .mean()
    )

    df["tcn_recent_mae"] = (
        df["tcn_error_past"]
        .rolling(error_window, min_periods=min_periods)
        .mean()
    )

    # Inverse-error weights
    inv_bilstm = 1 / (df["bilstm_recent_mae"] + eps)
    inv_tcn = 1 / (df["tcn_recent_mae"] + eps)

    df["dynamic_weight_bilstm"] = inv_bilstm / (inv_bilstm + inv_tcn)

    # For early periods with no enough history, use static/fallback weight
    df["dynamic_weight_bilstm"] = df["dynamic_weight_bilstm"].fillna(
        fallback_weight_bilstm
    )

    # Avoid extreme switching
    df["dynamic_weight_bilstm"] = df["dynamic_weight_bilstm"].clip(
        min_weight,
        max_weight
    )

    df["dynamic_weight_tcn"] = 1 - df["dynamic_weight_bilstm"]

    df["dynamic_ensemble_pred"] = (
        df["dynamic_weight_bilstm"] * df["bilstm_gbdt_pred"]
        + df["dynamic_weight_tcn"] * df["tcn_gbdt_pred"]
    )

    return df


dynamic_candidates = []

for error_window in [6, 12, 24, 48, 72, 168]:
    temp_df = build_dynamic_weight_ensemble(
        ensemble_df,
        error_window=error_window,
        min_periods=max(3, error_window // 4),
        min_weight=0.10,
        max_weight=0.90,
        fallback_weight_bilstm=best_weight_bilstm
    )

    metrics = evaluate_model(
        temp_df["y_true"],
        temp_df["dynamic_ensemble_pred"]
    )

    dynamic_candidates.append({
        "method": "dynamic_weighting",
        "error_window": error_window,
        **metrics
    })

dynamic_candidates_df = pd.DataFrame(
    dynamic_candidates
).sort_values("RMSE")

print("\nDynamic weighting candidates:")
display(dynamic_candidates_df)

best_dynamic_window = int(dynamic_candidates_df.iloc[0]["error_window"])

dynamic_df = build_dynamic_weight_ensemble(
    ensemble_df,
    error_window=best_dynamic_window,
    min_periods=max(3, best_dynamic_window // 4),
    min_weight=0.10,
    max_weight=0.90,
    fallback_weight_bilstm=best_weight_bilstm
)

dynamic_metrics = evaluate_model(
    dynamic_df["y_true"],
    dynamic_df["dynamic_ensemble_pred"]
)

print("\nBest dynamic weighting window:", best_dynamic_window)
print("Dynamic ensemble metrics:")
print(dynamic_metrics)


# ============================================================
# 5. ONLINE RESIDUAL BIAS CORRECTION
# ============================================================
# The model struggles in local regimes.
# So we correct using past residuals only:
#
# corrected_pred(t) = prediction(t) + alpha * EWM_past_residual(t)
#
# The residual is shifted by 1 to avoid using the current true value.
# ============================================================

def apply_online_bias_correction(
    df,
    pred_col,
    span=24,
    alpha=0.5,
    min_periods=3,
    output_col="corrected_pred"
):
    df = df.copy()

    residual_col = pred_col + "_residual"

    df[residual_col] = df["y_true"] - df[pred_col]

    df["past_residual_ewm"] = (
        df[residual_col]
        .shift(1)
        .ewm(span=span, adjust=False, min_periods=min_periods)
        .mean()
        .fillna(0)
    )

    df[output_col] = df[pred_col] + alpha * df["past_residual_ewm"]

    return df


bias_candidates = []

# We test correction on static ensemble
for span in [3, 6, 12, 24, 48, 72, 168]:
    for alpha in [0.25, 0.50, 0.75, 1.00]:

        temp_df = apply_online_bias_correction(
            ensemble_df,
            pred_col="static_ensemble_pred",
            span=span,
            alpha=alpha,
            min_periods=3,
            output_col="static_bias_corrected_pred"
        )

        metrics = evaluate_model(
            temp_df["y_true"],
            temp_df["static_bias_corrected_pred"]
        )

        bias_candidates.append({
            "base": "static_ensemble",
            "span": span,
            "alpha": alpha,
            **metrics
        })

# We test correction on dynamic ensemble
for span in [3, 6, 12, 24, 48, 72, 168]:
    for alpha in [0.25, 0.50, 0.75, 1.00]:

        temp_df = apply_online_bias_correction(
            dynamic_df,
            pred_col="dynamic_ensemble_pred",
            span=span,
            alpha=alpha,
            min_periods=3,
            output_col="dynamic_bias_corrected_pred"
        )

        metrics = evaluate_model(
            temp_df["y_true"],
            temp_df["dynamic_bias_corrected_pred"]
        )

        bias_candidates.append({
            "base": "dynamic_ensemble",
            "span": span,
            "alpha": alpha,
            **metrics
        })


bias_candidates_df = pd.DataFrame(
    bias_candidates
).sort_values("RMSE")

print("\nTop online bias correction candidates:")
display(bias_candidates_df.head(15))


# ============================================================
# 6. SELECT BEST ONLINE-CALIBRATED MODEL
# ============================================================

best_bias_row = bias_candidates_df.iloc[0]

best_base = best_bias_row["base"]
best_span = int(best_bias_row["span"])
best_alpha = float(best_bias_row["alpha"])

print("\nBest online calibration setup:")
print(best_bias_row)

if best_base == "static_ensemble":

    final_online_df = apply_online_bias_correction(
        ensemble_df,
        pred_col="static_ensemble_pred",
        span=best_span,
        alpha=best_alpha,
        min_periods=3,
        output_col="online_calibrated_pred"
    )

elif best_base == "dynamic_ensemble":

    final_online_df = apply_online_bias_correction(
        dynamic_df,
        pred_col="dynamic_ensemble_pred",
        span=best_span,
        alpha=best_alpha,
        min_periods=3,
        output_col="online_calibrated_pred"
    )

else:
    raise ValueError("Unknown best base model.")


online_calibrated_metrics = evaluate_model(
    final_online_df["y_true"],
    final_online_df["online_calibrated_pred"]
)

print("\nFinal online-calibrated metrics:")
print(online_calibrated_metrics)


# ============================================================
# 7. FINAL COMPARISON TABLE
# ============================================================

bilstm_metrics = evaluate_model(
    ensemble_df["y_true"],
    ensemble_df["bilstm_gbdt_pred"]
)

tcn_metrics = evaluate_model(
    ensemble_df["y_true"],
    ensemble_df["tcn_gbdt_pred"]
)

final_comparison_online = pd.DataFrame([
    {
        "model": "BiLSTM + GBDT",
        **bilstm_metrics
    },
    {
        "model": "TCN + GBDT",
        **tcn_metrics
    },
    {
        "model": "Static weighted ensemble",
        **static_metrics
    },
    {
        "model": f"Dynamic weighted ensemble window={best_dynamic_window}",
        **dynamic_metrics
    },
    {
        "model": f"Online calibrated ensemble base={best_base}, span={best_span}, alpha={best_alpha}",
        **online_calibrated_metrics
    }
]).sort_values("RMSE")

print("\nFinal comparison:")
display(final_comparison_online)


# ============================================================
# 8. PLOTS: FULL PERIOD
# ============================================================

plt.figure(figsize=(16, 5))

plt.plot(
    final_online_df.index,
    final_online_df["y_true"],
    label="Actual CO(GT)",
    alpha=0.8
)

plt.plot(
    final_online_df.index,
    final_online_df["static_ensemble_pred"],
    label="Original static ensemble",
    alpha=0.55
)

plt.plot(
    final_online_df.index,
    final_online_df["online_calibrated_pred"],
    label="Online calibrated ensemble",
    alpha=0.9
)

plt.title("CO(GT) Nowcasting — Original vs Online Calibrated Ensemble")
plt.xlabel("Date")
plt.ylabel("CO(GT)")
plt.legend()
plt.grid(True)
plt.show()


# ============================================================
# 9. ZOOM: BEGINNING OF TEST
# ============================================================

begin_start = final_online_df.index.min()
begin_end = begin_start + pd.Timedelta(days=10)

zoom_begin = final_online_df.loc[
    (final_online_df.index >= begin_start)
    & (final_online_df.index <= begin_end)
].copy()

if len(zoom_begin) > 0:

    plt.figure(figsize=(16, 5))

    plt.plot(
        zoom_begin.index,
        zoom_begin["y_true"],
        label="Actual CO(GT)",
        alpha=0.8
    )

    plt.plot(
        zoom_begin.index,
        zoom_begin["static_ensemble_pred"],
        label="Original static ensemble",
        alpha=0.55
    )

    plt.plot(
        zoom_begin.index,
        zoom_begin["online_calibrated_pred"],
        label="Online calibrated ensemble",
        alpha=0.9
    )

    plt.title("Zoom — Beginning of Test Period")
    plt.xlabel("Date")
    plt.ylabel("CO(GT)")
    plt.legend()
    plt.grid(True)
    plt.show()


# ============================================================
# 10. ZOOM: MID-MARCH
# ============================================================

march_start = pd.Timestamp("2005-03-10")
march_end = pd.Timestamp("2005-03-23")

zoom_march = final_online_df.loc[
    (final_online_df.index >= march_start)
    & (final_online_df.index <= march_end)
].copy()

if len(zoom_march) > 0:

    plt.figure(figsize=(16, 5))

    plt.plot(
        zoom_march.index,
        zoom_march["y_true"],
        label="Actual CO(GT)",
        alpha=0.8
    )

    plt.plot(
        zoom_march.index,
        zoom_march["static_ensemble_pred"],
        label="Original static ensemble",
        alpha=0.55
    )

    plt.plot(
        zoom_march.index,
        zoom_march["online_calibrated_pred"],
        label="Online calibrated ensemble",
        alpha=0.9
    )

    plt.title("Zoom — Mid-March Problem Period")
    plt.xlabel("Date")
    plt.ylabel("CO(GT)")
    plt.legend()
    plt.grid(True)
    plt.show()

else:
    print("Mid-March zoom period not found in index.")


# ============================================================
# 11. ROLLING ERROR COMPARISON
# ============================================================

final_online_df["original_abs_error"] = np.abs(
    final_online_df["y_true"] - final_online_df["static_ensemble_pred"]
)

final_online_df["calibrated_abs_error"] = np.abs(
    final_online_df["y_true"] - final_online_df["online_calibrated_pred"]
)

final_online_df["original_rolling_mae_24"] = (
    final_online_df["original_abs_error"]
    .rolling(24, min_periods=6)
    .mean()
)

final_online_df["calibrated_rolling_mae_24"] = (
    final_online_df["calibrated_abs_error"]
    .rolling(24, min_periods=6)
    .mean()
)

plt.figure(figsize=(16, 5))

plt.plot(
    final_online_df.index,
    final_online_df["original_rolling_mae_24"],
    label="Original rolling MAE 24h",
    alpha=0.8
)

plt.plot(
    final_online_df.index,
    final_online_df["calibrated_rolling_mae_24"],
    label="Calibrated rolling MAE 24h",
    alpha=0.8
)

plt.title("Rolling 24h MAE — Original vs Online Calibrated")
plt.xlabel("Date")
plt.ylabel("Rolling MAE")
plt.legend()
plt.grid(True)
plt.show()


# ============================================================
# 12. RESIDUAL ANALYSIS
# ============================================================

final_online_df["online_residual"] = (
    final_online_df["y_true"] - final_online_df["online_calibrated_pred"]
)

plt.figure(figsize=(16, 4))

plt.plot(
    final_online_df.index,
    final_online_df["online_residual"],
    alpha=0.8
)

plt.axhline(
    0,
    color="black",
    linestyle="--"
)

plt.title("Residuals Over Time — Online Calibrated Ensemble")
plt.xlabel("Date")
plt.ylabel("Error")
plt.grid(True)
plt.show()


plt.figure(figsize=(8, 4))

plt.hist(
    final_online_df["online_residual"],
    bins=40
)

plt.title("Residual Distribution — Online Calibrated Ensemble")
plt.xlabel("Prediction error")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()


# ============================================================
# 13. SAVE RESULTS
# ============================================================

final_online_predictions = final_online_df[
    [
        "y_true",
        "bilstm_gbdt_pred",
        "tcn_gbdt_pred",
        "static_ensemble_pred",
        "online_calibrated_pred",
        "static_residual",
        "online_residual",
        "original_abs_error",
        "calibrated_abs_error"
    ]
].copy()

final_online_predictions.to_csv(
    "online_calibrated_ensemble_predictions.csv"
)

final_comparison_online.to_csv(
    "online_calibration_comparison.csv",
    index=False
)

bias_candidates_df.to_csv(
    "online_bias_correction_grid_results.csv",
    index=False
)

dynamic_candidates_df.to_csv(
    "dynamic_weighting_grid_results.csv",
    index=False
)

print("\nSaved files:")
print("online_calibrated_ensemble_predictions.csv")
print("online_calibration_comparison.csv")
print("online_bias_correction_grid_results.csv")
print("dynamic_weighting_grid_results.csv")


# ============================================================
# 14. FINAL STATEMENT
# ============================================================

best_row = final_comparison_online.iloc[0]

print("\nBEST MODEL AFTER ONLINE CALIBRATION")
print("===================================")
print(best_row)

print("\nImportant note:")
print(
    "Online calibration uses past observed errors only. "
    "It is valid if, in deployment, recent observed CO(GT) values are available. "
    "If CO(GT) is missing for a long continuous block, this correction becomes less reliable."
)

# stability


In [ ]:
# ============================================================
# MODEL STABILITY ANALYSIS
# Error by hour, day, rolling periods, residual stability
# For final ensemble nowcasting model
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ============================================================
# 0. REQUIRED DATA
# ============================================================
# This code expects one of these situations:
#
# Situation A:
# ensemble_df already exists and contains:
# - y_true
# - final_ensemble_pred
#
# Situation B:
# ensemble_df contains:
# - y_true
# - best_weighted_pred
#
# Situation C:
# ensemble_df contains:
# - y_true
# - bilstm_gbdt_pred
# - tcn_gbdt_pred
# and best_weight_bilstm / best_weight_tcn exist
#
# The code will automatically detect the best available prediction column.
# ============================================================

if "ensemble_df" not in globals():
    raise ValueError("ensemble_df is missing. Run the ensemble code first.")

df_stability = ensemble_df.copy().sort_index()

if "y_true" not in df_stability.columns:
    raise ValueError("y_true column is missing from ensemble_df.")

# ------------------------------------------------------------
# Detect prediction column
# ------------------------------------------------------------

if "final_ensemble_pred" in df_stability.columns:
    pred_col = "final_ensemble_pred"

elif "best_weighted_pred" in df_stability.columns:
    pred_col = "best_weighted_pred"

elif "online_calibrated_pred" in df_stability.columns:
    pred_col = "online_calibrated_pred"

elif (
    "bilstm_gbdt_pred" in df_stability.columns
    and "tcn_gbdt_pred" in df_stability.columns
    and "best_weight_bilstm" in globals()
    and "best_weight_tcn" in globals()
):
    pred_col = "final_ensemble_pred"

    df_stability[pred_col] = (
        best_weight_bilstm * df_stability["bilstm_gbdt_pred"]
        + best_weight_tcn * df_stability["tcn_gbdt_pred"]
    )

else:
    raise ValueError(
        "No valid prediction column found. Need final_ensemble_pred, "
        "best_weighted_pred, online_calibrated_pred, or base predictions with weights."
    )

print("Using prediction column:", pred_col)
print("Data shape:", df_stability.shape)
display(df_stability.head())


# ============================================================
# 1. METRIC FUNCTIONS
# ============================================================

def smape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    denominator = np.where(denominator == 0, 1e-8, denominator)

    return np.mean(np.abs(y_true - y_pred) / denominator) * 100


def safe_r2(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    if len(y_true) < 3:
        return np.nan

    if np.std(y_true) == 0:
        return np.nan

    return r2_score(y_true, y_pred)


def evaluate_model(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    s_mape = smape(y_true, y_pred)
    r2 = safe_r2(y_true, y_pred)

    bias = np.mean(y_true - y_pred)

    return {
        "MAE": mae,
        "RMSE": rmse,
        "sMAPE": s_mape,
        "R2": r2,
        "Bias": bias
    }


# ============================================================
# 2. BASIC ERRORS
# ============================================================

df_stability["error"] = df_stability["y_true"] - df_stability[pred_col]
df_stability["abs_error"] = np.abs(df_stability["error"])
df_stability["squared_error"] = df_stability["error"] ** 2

df_stability["hour"] = df_stability.index.hour
df_stability["dayofweek"] = df_stability.index.dayofweek
df_stability["is_weekend"] = (df_stability["dayofweek"] >= 5).astype(int)

day_names = {
    0: "Monday",
    1: "Tuesday",
    2: "Wednesday",
    3: "Thursday",
    4: "Friday",
    5: "Saturday",
    6: "Sunday"
}

df_stability["day_name"] = df_stability["dayofweek"].map(day_names)

overall_metrics = evaluate_model(
    df_stability["y_true"],
    df_stability[pred_col]
)

print("\nOVERALL MODEL PERFORMANCE")
print("=========================")
print(overall_metrics)


# ============================================================
# 3. ACTUAL VS PREDICTED PLOT
# ============================================================

plt.figure(figsize=(16, 5))

plt.plot(
    df_stability.index,
    df_stability["y_true"],
    label="Actual CO(GT)",
    alpha=0.8
)

plt.plot(
    df_stability.index,
    df_stability[pred_col],
    label="Predicted CO(GT)",
    alpha=0.85
)

plt.title("Final Model — Actual vs Predicted CO(GT)")
plt.xlabel("Date")
plt.ylabel("CO(GT)")
plt.legend()
plt.grid(True)
plt.show()


# ============================================================
# 4. RESIDUAL PLOT
# ============================================================

plt.figure(figsize=(16, 4))

plt.plot(
    df_stability.index,
    df_stability["error"],
    label="Residual = actual - predicted",
    alpha=0.8
)

plt.axhline(
    0,
    color="black",
    linestyle="--"
)

plt.title("Residuals Over Time")
plt.xlabel("Date")
plt.ylabel("Error")
plt.legend()
plt.grid(True)
plt.show()


# ============================================================
# 5. ROLLING STABILITY ANALYSIS
# ============================================================
# Rolling MAE and RMSE show if the model becomes unstable locally.
# ============================================================

rolling_windows = [24, 48, 72, 168]

for window in rolling_windows:
    df_stability[f"rolling_mae_{window}h"] = (
        df_stability["abs_error"]
        .rolling(window, min_periods=max(6, window // 4))
        .mean()
    )

    df_stability[f"rolling_rmse_{window}h"] = (
        df_stability["squared_error"]
        .rolling(window, min_periods=max(6, window // 4))
        .mean()
        .apply(np.sqrt)
    )

    df_stability[f"rolling_bias_{window}h"] = (
        df_stability["error"]
        .rolling(window, min_periods=max(6, window // 4))
        .mean()
    )


# ------------------------------------------------------------
# Plot rolling MAE
# ------------------------------------------------------------

plt.figure(figsize=(16, 5))

for window in rolling_windows:
    plt.plot(
        df_stability.index,
        df_stability[f"rolling_mae_{window}h"],
        label=f"Rolling MAE {window}h",
        alpha=0.8
    )

plt.axhline(
    overall_metrics["MAE"],
    color="black",
    linestyle="--",
    label="Overall MAE"
)

plt.title("Rolling MAE — Model Stability Over Time")
plt.xlabel("Date")
plt.ylabel("MAE")
plt.legend()
plt.grid(True)
plt.show()


# ------------------------------------------------------------
# Plot rolling RMSE
# ------------------------------------------------------------

plt.figure(figsize=(16, 5))

for window in rolling_windows:
    plt.plot(
        df_stability.index,
        df_stability[f"rolling_rmse_{window}h"],
        label=f"Rolling RMSE {window}h",
        alpha=0.8
    )

plt.axhline(
    overall_metrics["RMSE"],
    color="black",
    linestyle="--",
    label="Overall RMSE"
)

plt.title("Rolling RMSE — Model Stability Over Time")
plt.xlabel("Date")
plt.ylabel("RMSE")
plt.legend()
plt.grid(True)
plt.show()


# ------------------------------------------------------------
# Plot rolling bias
# ------------------------------------------------------------

plt.figure(figsize=(16, 5))

for window in [24, 72, 168]:
    plt.plot(
        df_stability.index,
        df_stability[f"rolling_bias_{window}h"],
        label=f"Rolling bias {window}h",
        alpha=0.8
    )

plt.axhline(
    0,
    color="black",
    linestyle="--",
    label="Zero bias"
)

plt.title("Rolling Bias — Positive = Underprediction, Negative = Overprediction")
plt.xlabel("Date")
plt.ylabel("Bias")
plt.legend()
plt.grid(True)
plt.show()


# ============================================================
# 6. WORST LOCAL PERIODS
# ============================================================

worst_24h_periods = (
    df_stability[
        [
            "y_true",
            pred_col,
            "error",
            "abs_error",
            "rolling_mae_24h",
            "rolling_rmse_24h",
            "rolling_bias_24h"
        ]
    ]
    .dropna()
    .sort_values("rolling_mae_24h", ascending=False)
    .head(20)
)

print("\nWORST LOCAL PERIODS BY 24H ROLLING MAE")
print("======================================")
display(worst_24h_periods)


# ============================================================
# 7. ERROR BY HOUR OF DAY
# ============================================================

error_by_hour = (
    df_stability
    .groupby("hour")
    .apply(
        lambda g: pd.Series({
            "count": len(g),
            "MAE": mean_absolute_error(g["y_true"], g[pred_col]),
            "RMSE": np.sqrt(mean_squared_error(g["y_true"], g[pred_col])),
            "R2": safe_r2(g["y_true"], g[pred_col]),
            "Bias": np.mean(g["y_true"] - g[pred_col]),
            "Mean_actual": g["y_true"].mean(),
            "Mean_pred": g[pred_col].mean()
        })
    )
    .reset_index()
)

print("\nERROR BY HOUR")
print("=============")
display(error_by_hour)


# ------------------------------------------------------------
# Plot MAE by hour
# ------------------------------------------------------------

plt.figure(figsize=(12, 4))

plt.bar(
    error_by_hour["hour"],
    error_by_hour["MAE"]
)

plt.axhline(
    overall_metrics["MAE"],
    color="black",
    linestyle="--",
    label="Overall MAE"
)

plt.title("MAE by Hour of Day")
plt.xlabel("Hour")
plt.ylabel("MAE")
plt.xticks(range(0, 24))
plt.legend()
plt.grid(True)
plt.show()


# ------------------------------------------------------------
# Plot RMSE by hour
# ------------------------------------------------------------

plt.figure(figsize=(12, 4))

plt.bar(
    error_by_hour["hour"],
    error_by_hour["RMSE"]
)

plt.axhline(
    overall_metrics["RMSE"],
    color="black",
    linestyle="--",
    label="Overall RMSE"
)

plt.title("RMSE by Hour of Day")
plt.xlabel("Hour")
plt.ylabel("RMSE")
plt.xticks(range(0, 24))
plt.legend()
plt.grid(True)
plt.show()


# ------------------------------------------------------------
# Plot bias by hour
# ------------------------------------------------------------

plt.figure(figsize=(12, 4))

plt.bar(
    error_by_hour["hour"],
    error_by_hour["Bias"]
)

plt.axhline(
    0,
    color="black",
    linestyle="--"
)

plt.title("Bias by Hour — Positive = Underprediction, Negative = Overprediction")
plt.xlabel("Hour")
plt.ylabel("Bias")
plt.xticks(range(0, 24))
plt.grid(True)
plt.show()


# ============================================================
# 8. ERROR BY DAY OF WEEK
# ============================================================

error_by_day = (
    df_stability
    .groupby(["dayofweek", "day_name"])
    .apply(
        lambda g: pd.Series({
            "count": len(g),
            "MAE": mean_absolute_error(g["y_true"], g[pred_col]),
            "RMSE": np.sqrt(mean_squared_error(g["y_true"], g[pred_col])),
            "R2": safe_r2(g["y_true"], g[pred_col]),
            "Bias": np.mean(g["y_true"] - g[pred_col]),
            "Mean_actual": g["y_true"].mean(),
            "Mean_pred": g[pred_col].mean()
        })
    )
    .reset_index()
    .sort_values("dayofweek")
)

print("\nERROR BY DAY OF WEEK")
print("====================")
display(error_by_day)


# ------------------------------------------------------------
# Plot MAE by day
# ------------------------------------------------------------

plt.figure(figsize=(10, 4))

plt.bar(
    error_by_day["day_name"],
    error_by_day["MAE"]
)

plt.axhline(
    overall_metrics["MAE"],
    color="black",
    linestyle="--",
    label="Overall MAE"
)

plt.title("MAE by Day of Week")
plt.xlabel("Day")
plt.ylabel("MAE")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)
plt.show()


# ------------------------------------------------------------
# Plot RMSE by day
# ------------------------------------------------------------

plt.figure(figsize=(10, 4))

plt.bar(
    error_by_day["day_name"],
    error_by_day["RMSE"]
)

plt.axhline(
    overall_metrics["RMSE"],
    color="black",
    linestyle="--",
    label="Overall RMSE"
)

plt.title("RMSE by Day of Week")
plt.xlabel("Day")
plt.ylabel("RMSE")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)
plt.show()


# ------------------------------------------------------------
# Plot bias by day
# ------------------------------------------------------------

plt.figure(figsize=(10, 4))

plt.bar(
    error_by_day["day_name"],
    error_by_day["Bias"]
)

plt.axhline(
    0,
    color="black",
    linestyle="--"
)

plt.title("Bias by Day — Positive = Underprediction, Negative = Overprediction")
plt.xlabel("Day")
plt.ylabel("Bias")
plt.xticks(rotation=45)
plt.grid(True)
plt.show()


# ============================================================
# 9. ERROR HEATMAP: HOUR × DAY OF WEEK
# ============================================================

heatmap_mae = (
    df_stability
    .groupby(["dayofweek", "hour"])["abs_error"]
    .mean()
    .unstack()
)

heatmap_rmse = (
    df_stability
    .groupby(["dayofweek", "hour"])["squared_error"]
    .mean()
    .apply(np.sqrt)
    .unstack()
)

heatmap_bias = (
    df_stability
    .groupby(["dayofweek", "hour"])["error"]
    .mean()
    .unstack()
)

heatmap_mae.index = [day_names[i] for i in heatmap_mae.index]
heatmap_rmse.index = [day_names[i] for i in heatmap_rmse.index]
heatmap_bias.index = [day_names[i] for i in heatmap_bias.index]


# ------------------------------------------------------------
# MAE heatmap
# ------------------------------------------------------------

plt.figure(figsize=(14, 5))

plt.imshow(
    heatmap_mae,
    aspect="auto"
)

plt.colorbar(label="MAE")
plt.title("MAE Heatmap — Day of Week × Hour")
plt.xlabel("Hour")
plt.ylabel("Day of Week")
plt.xticks(range(24), range(24))
plt.yticks(range(len(heatmap_mae.index)), heatmap_mae.index)
plt.show()


# ------------------------------------------------------------
# RMSE heatmap
# ------------------------------------------------------------

plt.figure(figsize=(14, 5))

plt.imshow(
    heatmap_rmse,
    aspect="auto"
)

plt.colorbar(label="RMSE")
plt.title("RMSE Heatmap — Day of Week × Hour")
plt.xlabel("Hour")
plt.ylabel("Day of Week")
plt.xticks(range(24), range(24))
plt.yticks(range(len(heatmap_rmse.index)), heatmap_rmse.index)
plt.show()


# ------------------------------------------------------------
# Bias heatmap
# ------------------------------------------------------------

plt.figure(figsize=(14, 5))

plt.imshow(
    heatmap_bias,
    aspect="auto"
)

plt.colorbar(label="Bias")
plt.title("Bias Heatmap — Positive = Underprediction, Negative = Overprediction")
plt.xlabel("Hour")
plt.ylabel("Day of Week")
plt.xticks(range(24), range(24))
plt.yticks(range(len(heatmap_bias.index)), heatmap_bias.index)
plt.show()


# ============================================================
# 10. DAILY AND WEEKLY STABILITY
# ============================================================

def grouped_metrics(g):
    return pd.Series({
        "count": len(g),
        "MAE": mean_absolute_error(g["y_true"], g[pred_col]),
        "RMSE": np.sqrt(mean_squared_error(g["y_true"], g[pred_col])),
        "R2": safe_r2(g["y_true"], g[pred_col]),
        "Bias": np.mean(g["y_true"] - g[pred_col]),
        "Mean_actual": g["y_true"].mean(),
        "Mean_pred": g[pred_col].mean()
    })


daily_metrics = (
    df_stability
    .resample("D")
    .apply(lambda g: grouped_metrics(g) if len(g) > 0 else np.nan)
)

weekly_metrics = (
    df_stability
    .resample("W")
    .apply(lambda g: grouped_metrics(g) if len(g) > 0 else np.nan)
)

print("\nDAILY STABILITY METRICS")
print("=======================")
display(daily_metrics)

print("\nWEEKLY STABILITY METRICS")
print("========================")
display(weekly_metrics)


# ------------------------------------------------------------
# Daily MAE / RMSE
# ------------------------------------------------------------

plt.figure(figsize=(16, 5))

plt.plot(
    daily_metrics.index,
    daily_metrics["MAE"],
    marker="o",
    label="Daily MAE"
)

plt.axhline(
    overall_metrics["MAE"],
    color="black",
    linestyle="--",
    label="Overall MAE"
)

plt.title("Daily MAE Stability")
plt.xlabel("Date")
plt.ylabel("MAE")
plt.legend()
plt.grid(True)
plt.show()


plt.figure(figsize=(16, 5))

plt.plot(
    daily_metrics.index,
    daily_metrics["RMSE"],
    marker="o",
    label="Daily RMSE"
)

plt.axhline(
    overall_metrics["RMSE"],
    color="black",
    linestyle="--",
    label="Overall RMSE"
)

plt.title("Daily RMSE Stability")
plt.xlabel("Date")
plt.ylabel("RMSE")
plt.legend()
plt.grid(True)
plt.show()


# ------------------------------------------------------------
# Weekly MAE / RMSE
# ------------------------------------------------------------

plt.figure(figsize=(14, 5))

plt.plot(
    weekly_metrics.index,
    weekly_metrics["MAE"],
    marker="o",
    label="Weekly MAE"
)

plt.axhline(
    overall_metrics["MAE"],
    color="black",
    linestyle="--",
    label="Overall MAE"
)

plt.title("Weekly MAE Stability")
plt.xlabel("Week")
plt.ylabel("MAE")
plt.legend()
plt.grid(True)
plt.show()


plt.figure(figsize=(14, 5))

plt.plot(
    weekly_metrics.index,
    weekly_metrics["RMSE"],
    marker="o",
    label="Weekly RMSE"
)

plt.axhline(
    overall_metrics["RMSE"],
    color="black",
    linestyle="--",
    label="Overall RMSE"
)

plt.title("Weekly RMSE Stability")
plt.xlabel("Week")
plt.ylabel("RMSE")
plt.legend()
plt.grid(True)
plt.show()


# ============================================================
# 11. RESIDUAL AUTOCORRELATION CHECK
# ============================================================
# If residual autocorrelation is high, the model still misses temporal structure.
# ============================================================

max_lag = 48

acf_values = []

for lag in range(1, max_lag + 1):
    acf = df_stability["error"].autocorr(lag=lag)
    acf_values.append({
        "lag": lag,
        "residual_autocorrelation": acf
    })

acf_df = pd.DataFrame(acf_values)

print("\nRESIDUAL AUTOCORRELATION")
print("========================")
display(acf_df.head(20))

plt.figure(figsize=(12, 4))

plt.bar(
    acf_df["lag"],
    acf_df["residual_autocorrelation"]
)

plt.axhline(
    0,
    color="black",
    linestyle="--"
)

plt.title("Residual Autocorrelation")
plt.xlabel("Lag")
plt.ylabel("Autocorrelation")
plt.grid(True)
plt.show()


# ============================================================
# 12. STABILITY SUMMARY
# ============================================================

daily_mae_mean = daily_metrics["MAE"].mean()
daily_mae_std = daily_metrics["MAE"].std()
daily_mae_cv = daily_mae_std / daily_mae_mean

weekly_mae_mean = weekly_metrics["MAE"].mean()
weekly_mae_std = weekly_metrics["MAE"].std()
weekly_mae_cv = weekly_mae_std / weekly_mae_mean

max_rolling_mae_24 = df_stability["rolling_mae_24h"].max()
mean_rolling_mae_24 = df_stability["rolling_mae_24h"].mean()

max_rolling_rmse_24 = df_stability["rolling_rmse_24h"].max()
mean_rolling_rmse_24 = df_stability["rolling_rmse_24h"].mean()

stability_summary = pd.DataFrame([
    {
        "metric": "Overall MAE",
        "value": overall_metrics["MAE"]
    },
    {
        "metric": "Overall RMSE",
        "value": overall_metrics["RMSE"]
    },
    {
        "metric": "Overall R2",
        "value": overall_metrics["R2"]
    },
    {
        "metric": "Daily MAE coefficient of variation",
        "value": daily_mae_cv
    },
    {
        "metric": "Weekly MAE coefficient of variation",
        "value": weekly_mae_cv
    },
    {
        "metric": "Max 24h rolling MAE",
        "value": max_rolling_mae_24
    },
    {
        "metric": "Mean 24h rolling MAE",
        "value": mean_rolling_mae_24
    },
    {
        "metric": "Max 24h rolling RMSE",
        "value": max_rolling_rmse_24
    },
    {
        "metric": "Mean 24h rolling RMSE",
        "value": mean_rolling_rmse_24
    }
])

print("\nSTABILITY SUMMARY")
print("=================")
display(stability_summary)


# ============================================================
# 13. AUTOMATIC INTERPRETATION HELP
# ============================================================

print("\nINTERPRETATION GUIDE")
print("====================")

print(f"Overall MAE: {overall_metrics['MAE']:.4f}")
print(f"Overall RMSE: {overall_metrics['RMSE']:.4f}")
print(f"Overall R2: {overall_metrics['R2']:.4f}")
print(f"Daily MAE CV: {daily_mae_cv:.4f}")
print(f"Weekly MAE CV: {weekly_mae_cv:.4f}")
print(f"Max 24h rolling MAE: {max_rolling_mae_24:.4f}")
print(f"Mean 24h rolling MAE: {mean_rolling_mae_24:.4f}")

if daily_mae_cv < 0.30:
    print("\nDaily stability: Good. Daily errors are relatively stable.")
elif daily_mae_cv < 0.60:
    print("\nDaily stability: Medium. Some days are harder than others.")
else:
    print("\nDaily stability: Weak. The model has strong local instability.")

if max_rolling_mae_24 > 2 * overall_metrics["MAE"]:
    print(
        "Local regime warning: Some 24h periods have MAE more than twice "
        "the overall MAE. Investigate worst_24h_periods."
    )
else:
    print(
        "Local regime check: No extreme 24h instability based on MAE threshold."
    )

top_hour = error_by_hour.sort_values("MAE", ascending=False).iloc[0]
top_day = error_by_day.sort_values("MAE", ascending=False).iloc[0]

print(
    f"\nHardest hour: {int(top_hour['hour'])}:00 "
    f"with MAE = {top_hour['MAE']:.4f}"
)

print(
    f"Hardest day: {top_day['day_name']} "
    f"with MAE = {top_day['MAE']:.4f}"
)


# ============================================================
# 14. SAVE TABLES
# ============================================================

df_stability.to_csv("model_stability_full_results.csv")
error_by_hour.to_csv("error_by_hour.csv", index=False)
error_by_day.to_csv("error_by_day.csv", index=False)
daily_metrics.to_csv("daily_stability_metrics.csv")
weekly_metrics.to_csv("weekly_stability_metrics.csv")
worst_24h_periods.to_csv("worst_24h_periods.csv")
acf_df.to_csv("residual_autocorrelation.csv", index=False)
stability_summary.to_csv("stability_summary.csv", index=False)

print("\nSaved files:")
print("model_stability_full_results.csv")
print("error_by_hour.csv")
print("error_by_day.csv")
print("daily_stability_metrics.csv")
print("weekly_stability_metrics.csv")
print("worst_24h_periods.csv")
print("residual_autocorrelation.csv")
print("stability_summary.csv")

In [ ]:
# ============================================================
# STABILITY ANALYSIS FOR CALIBRATED MODEL
# Model with R² around 0.959
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ============================================================
# 0. CHOOSE DATAFRAME AND PREDICTION COLUMN
# ============================================================
# This code works if you have one of:
#
# 1) final_online_df
#    with columns:
#    - y_true
#    - online_calibrated_pred
#
# 2) ensemble_df
#    with columns:
#    - y_true
#    - online_calibrated_pred
#
# 3) another dataframe with y_true and calibrated predictions
#
# If the automatic detection fails, manually set:
# df_calibrated = your_dataframe.copy()
# pred_col = "your_prediction_column"
# ============================================================

if "final_online_df" in globals():
    df_calibrated = final_online_df.copy()
    print("Using final_online_df")

elif "ensemble_df" in globals():
    df_calibrated = ensemble_df.copy()
    print("Using ensemble_df")

else:
    raise ValueError(
        "No dataframe found. You need final_online_df or ensemble_df."
    )

df_calibrated = df_calibrated.sort_index().copy()

if "y_true" not in df_calibrated.columns:
    raise ValueError("y_true column is missing.")

# ------------------------------------------------------------
# Automatic prediction column detection
# ------------------------------------------------------------

possible_pred_cols = [
    "online_calibrated_pred",
    "calibrated_pred",
    "dynamic_bias_corrected_pred",
    "static_bias_corrected_pred",
    "final_calibrated_pred",
    "best_calibrated_pred"
]

pred_col = None

for col in possible_pred_cols:
    if col in df_calibrated.columns:
        pred_col = col
        break

if pred_col is None:
    print("Available columns:")
    print(df_calibrated.columns.tolist())
    raise ValueError(
        "No calibrated prediction column found. "
        "Set pred_col manually to your calibrated prediction column."
    )

print("Using prediction column:", pred_col)
print("Data shape:", df_calibrated.shape)

display(df_calibrated.head())


# ============================================================
# 1. METRIC FUNCTIONS
# ============================================================

def smape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    denominator = np.where(denominator == 0, 1e-8, denominator)

    return np.mean(np.abs(y_true - y_pred) / denominator) * 100


def safe_r2(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    if len(y_true) < 3:
        return np.nan

    if np.std(y_true) == 0:
        return np.nan

    return r2_score(y_true, y_pred)


def evaluate_model(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    s_mape = smape(y_true, y_pred)
    r2 = safe_r2(y_true, y_pred)
    bias = np.mean(y_true - y_pred)

    return {
        "MAE": mae,
        "RMSE": rmse,
        "sMAPE": s_mape,
        "R2": r2,
        "Bias": bias
    }


# ============================================================
# 2. BUILD ERROR COLUMNS
# ============================================================

df_calibrated["error"] = df_calibrated["y_true"] - df_calibrated[pred_col]
df_calibrated["abs_error"] = np.abs(df_calibrated["error"])
df_calibrated["squared_error"] = df_calibrated["error"] ** 2

df_calibrated["hour"] = df_calibrated.index.hour
df_calibrated["dayofweek"] = df_calibrated.index.dayofweek
df_calibrated["is_weekend"] = (df_calibrated["dayofweek"] >= 5).astype(int)

day_names = {
    0: "Monday",
    1: "Tuesday",
    2: "Wednesday",
    3: "Thursday",
    4: "Friday",
    5: "Saturday",
    6: "Sunday"
}

df_calibrated["day_name"] = df_calibrated["dayofweek"].map(day_names)

overall_metrics = evaluate_model(
    df_calibrated["y_true"],
    df_calibrated[pred_col]
)

print("\nOVERALL CALIBRATED MODEL PERFORMANCE")
print("====================================")
print(overall_metrics)


# ============================================================
# 3. ACTUAL VS PREDICTED
# ============================================================

plt.figure(figsize=(16, 5))

plt.plot(
    df_calibrated.index,
    df_calibrated["y_true"],
    label="Actual CO(GT)",
    alpha=0.8
)

plt.plot(
    df_calibrated.index,
    df_calibrated[pred_col],
    label="Calibrated prediction",
    alpha=0.85
)

plt.title("Calibrated Model — Actual vs Predicted CO(GT)")
plt.xlabel("Date")
plt.ylabel("CO(GT)")
plt.legend()
plt.grid(True)
plt.show()


# ============================================================
# 4. RESIDUAL PLOT
# ============================================================

plt.figure(figsize=(16, 4))

plt.plot(
    df_calibrated.index,
    df_calibrated["error"],
    label="Residual = actual - predicted",
    alpha=0.8
)

plt.axhline(
    0,
    color="black",
    linestyle="--"
)

plt.title("Residuals Over Time — Calibrated Model")
plt.xlabel("Date")
plt.ylabel("Error")
plt.legend()
plt.grid(True)
plt.show()


# ============================================================
# 5. ROLLING STABILITY
# ============================================================

rolling_windows = [24, 48, 72, 168]

for window in rolling_windows:

    df_calibrated[f"rolling_mae_{window}h"] = (
        df_calibrated["abs_error"]
        .rolling(window, min_periods=max(6, window // 4))
        .mean()
    )

    df_calibrated[f"rolling_rmse_{window}h"] = (
        df_calibrated["squared_error"]
        .rolling(window, min_periods=max(6, window // 4))
        .mean()
        .apply(np.sqrt)
    )

    df_calibrated[f"rolling_bias_{window}h"] = (
        df_calibrated["error"]
        .rolling(window, min_periods=max(6, window // 4))
        .mean()
    )


# ------------------------------------------------------------
# Rolling MAE
# ------------------------------------------------------------

plt.figure(figsize=(16, 5))

for window in rolling_windows:
    plt.plot(
        df_calibrated.index,
        df_calibrated[f"rolling_mae_{window}h"],
        label=f"Rolling MAE {window}h",
        alpha=0.8
    )

plt.axhline(
    overall_metrics["MAE"],
    color="black",
    linestyle="--",
    label="Overall MAE"
)

plt.title("Rolling MAE — Calibrated Model Stability")
plt.xlabel("Date")
plt.ylabel("MAE")
plt.legend()
plt.grid(True)
plt.show()


# ------------------------------------------------------------
# Rolling RMSE
# ------------------------------------------------------------

plt.figure(figsize=(16, 5))

for window in rolling_windows:
    plt.plot(
        df_calibrated.index,
        df_calibrated[f"rolling_rmse_{window}h"],
        label=f"Rolling RMSE {window}h",
        alpha=0.8
    )

plt.axhline(
    overall_metrics["RMSE"],
    color="black",
    linestyle="--",
    label="Overall RMSE"
)

plt.title("Rolling RMSE — Calibrated Model Stability")
plt.xlabel("Date")
plt.ylabel("RMSE")
plt.legend()
plt.grid(True)
plt.show()


# ------------------------------------------------------------
# Rolling bias
# ------------------------------------------------------------

plt.figure(figsize=(16, 5))

for window in [24, 72, 168]:
    plt.plot(
        df_calibrated.index,
        df_calibrated[f"rolling_bias_{window}h"],
        label=f"Rolling bias {window}h",
        alpha=0.8
    )

plt.axhline(
    0,
    color="black",
    linestyle="--",
    label="Zero bias"
)

plt.title("Rolling Bias — Calibrated Model")
plt.xlabel("Date")
plt.ylabel("Bias")
plt.legend()
plt.grid(True)
plt.show()


# ============================================================
# 6. WORST LOCAL PERIODS
# ============================================================

worst_24h_periods = (
    df_calibrated[
        [
            "y_true",
            pred_col,
            "error",
            "abs_error",
            "rolling_mae_24h",
            "rolling_rmse_24h",
            "rolling_bias_24h"
        ]
    ]
    .dropna()
    .sort_values("rolling_mae_24h", ascending=False)
    .head(20)
)

print("\nWORST LOCAL PERIODS BY 24H ROLLING MAE")
print("======================================")
display(worst_24h_periods)


# ============================================================
# 7. ERROR BY HOUR
# ============================================================

error_by_hour = (
    df_calibrated
    .groupby("hour")
    .apply(
        lambda g: pd.Series({
            "count": len(g),
            "MAE": mean_absolute_error(g["y_true"], g[pred_col]),
            "RMSE": np.sqrt(mean_squared_error(g["y_true"], g[pred_col])),
            "R2": safe_r2(g["y_true"], g[pred_col]),
            "Bias": np.mean(g["y_true"] - g[pred_col]),
            "Mean_actual": g["y_true"].mean(),
            "Mean_pred": g[pred_col].mean()
        })
    )
    .reset_index()
)

print("\nERROR BY HOUR — CALIBRATED MODEL")
print("================================")
display(error_by_hour)


plt.figure(figsize=(12, 4))

plt.bar(
    error_by_hour["hour"],
    error_by_hour["MAE"]
)

plt.axhline(
    overall_metrics["MAE"],
    color="black",
    linestyle="--",
    label="Overall MAE"
)

plt.title("MAE by Hour — Calibrated Model")
plt.xlabel("Hour")
plt.ylabel("MAE")
plt.xticks(range(0, 24))
plt.legend()
plt.grid(True)
plt.show()


plt.figure(figsize=(12, 4))

plt.bar(
    error_by_hour["hour"],
    error_by_hour["RMSE"]
)

plt.axhline(
    overall_metrics["RMSE"],
    color="black",
    linestyle="--",
    label="Overall RMSE"
)

plt.title("RMSE by Hour — Calibrated Model")
plt.xlabel("Hour")
plt.ylabel("RMSE")
plt.xticks(range(0, 24))
plt.legend()
plt.grid(True)
plt.show()


plt.figure(figsize=(12, 4))

plt.bar(
    error_by_hour["hour"],
    error_by_hour["Bias"]
)

plt.axhline(
    0,
    color="black",
    linestyle="--"
)

plt.title("Bias by Hour — Calibrated Model")
plt.xlabel("Hour")
plt.ylabel("Bias")
plt.xticks(range(0, 24))
plt.grid(True)
plt.show()


# ============================================================
# 8. ERROR BY DAY OF WEEK
# ============================================================

error_by_day = (
    df_calibrated
    .groupby(["dayofweek", "day_name"])
    .apply(
        lambda g: pd.Series({
            "count": len(g),
            "MAE": mean_absolute_error(g["y_true"], g[pred_col]),
            "RMSE": np.sqrt(mean_squared_error(g["y_true"], g[pred_col])),
            "R2": safe_r2(g["y_true"], g[pred_col]),
            "Bias": np.mean(g["y_true"] - g[pred_col]),
            "Mean_actual": g["y_true"].mean(),
            "Mean_pred": g[pred_col].mean()
        })
    )
    .reset_index()
    .sort_values("dayofweek")
)

print("\nERROR BY DAY OF WEEK — CALIBRATED MODEL")
print("=======================================")
display(error_by_day)


plt.figure(figsize=(10, 4))

plt.bar(
    error_by_day["day_name"],
    error_by_day["MAE"]
)

plt.axhline(
    overall_metrics["MAE"],
    color="black",
    linestyle="--",
    label="Overall MAE"
)

plt.title("MAE by Day of Week — Calibrated Model")
plt.xlabel("Day")
plt.ylabel("MAE")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)
plt.show()


plt.figure(figsize=(10, 4))

plt.bar(
    error_by_day["day_name"],
    error_by_day["RMSE"]
)

plt.axhline(
    overall_metrics["RMSE"],
    color="black",
    linestyle="--",
    label="Overall RMSE"
)

plt.title("RMSE by Day of Week — Calibrated Model")
plt.xlabel("Day")
plt.ylabel("RMSE")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)
plt.show()


plt.figure(figsize=(10, 4))

plt.bar(
    error_by_day["day_name"],
    error_by_day["Bias"]
)

plt.axhline(
    0,
    color="black",
    linestyle="--"
)

plt.title("Bias by Day — Calibrated Model")
plt.xlabel("Day")
plt.ylabel("Bias")
plt.xticks(rotation=45)
plt.grid(True)
plt.show()


# ============================================================
# 9. ERROR HEATMAP: DAY × HOUR
# ============================================================

heatmap_mae = (
    df_calibrated
    .groupby(["dayofweek", "hour"])["abs_error"]
    .mean()
    .unstack()
)

heatmap_rmse = (
    df_calibrated
    .groupby(["dayofweek", "hour"])["squared_error"]
    .mean()
    .apply(np.sqrt)
    .unstack()
)

heatmap_bias = (
    df_calibrated
    .groupby(["dayofweek", "hour"])["error"]
    .mean()
    .unstack()
)

heatmap_mae.index = [day_names[i] for i in heatmap_mae.index]
heatmap_rmse.index = [day_names[i] for i in heatmap_rmse.index]
heatmap_bias.index = [day_names[i] for i in heatmap_bias.index]


plt.figure(figsize=(14, 5))

plt.imshow(
    heatmap_mae,
    aspect="auto"
)

plt.colorbar(label="MAE")
plt.title("MAE Heatmap — Calibrated Model")
plt.xlabel("Hour")
plt.ylabel("Day of Week")
plt.xticks(range(24), range(24))
plt.yticks(range(len(heatmap_mae.index)), heatmap_mae.index)
plt.show()


plt.figure(figsize=(14, 5))

plt.imshow(
    heatmap_rmse,
    aspect="auto"
)

plt.colorbar(label="RMSE")
plt.title("RMSE Heatmap — Calibrated Model")
plt.xlabel("Hour")
plt.ylabel("Day of Week")
plt.xticks(range(24), range(24))
plt.yticks(range(len(heatmap_rmse.index)), heatmap_rmse.index)
plt.show()


plt.figure(figsize=(14, 5))

plt.imshow(
    heatmap_bias,
    aspect="auto"
)

plt.colorbar(label="Bias")
plt.title("Bias Heatmap — Calibrated Model")
plt.xlabel("Hour")
plt.ylabel("Day of Week")
plt.xticks(range(24), range(24))
plt.yticks(range(len(heatmap_bias.index)), heatmap_bias.index)
plt.show()


# ============================================================
# 10. DAILY AND WEEKLY STABILITY
# ============================================================

def grouped_metrics(g):
    return pd.Series({
        "count": len(g),
        "MAE": mean_absolute_error(g["y_true"], g[pred_col]),
        "RMSE": np.sqrt(mean_squared_error(g["y_true"], g[pred_col])),
        "R2": safe_r2(g["y_true"], g[pred_col]),
        "Bias": np.mean(g["y_true"] - g[pred_col]),
        "Mean_actual": g["y_true"].mean(),
        "Mean_pred": g[pred_col].mean()
    })


daily_metrics = (
    df_calibrated
    .resample("D")
    .apply(lambda g: grouped_metrics(g) if len(g) > 0 else np.nan)
)

weekly_metrics = (
    df_calibrated
    .resample("W")
    .apply(lambda g: grouped_metrics(g) if len(g) > 0 else np.nan)
)

print("\nDAILY STABILITY METRICS — CALIBRATED MODEL")
print("==========================================")
display(daily_metrics)

print("\nWEEKLY STABILITY METRICS — CALIBRATED MODEL")
print("===========================================")
display(weekly_metrics)


plt.figure(figsize=(16, 5))

plt.plot(
    daily_metrics.index,
    daily_metrics["MAE"],
    marker="o",
    label="Daily MAE"
)

plt.axhline(
    overall_metrics["MAE"],
    color="black",
    linestyle="--",
    label="Overall MAE"
)

plt.title("Daily MAE Stability — Calibrated Model")
plt.xlabel("Date")
plt.ylabel("MAE")
plt.legend()
plt.grid(True)
plt.show()


plt.figure(figsize=(16, 5))

plt.plot(
    daily_metrics.index,
    daily_metrics["RMSE"],
    marker="o",
    label="Daily RMSE"
)

plt.axhline(
    overall_metrics["RMSE"],
    color="black",
    linestyle="--",
    label="Overall RMSE"
)

plt.title("Daily RMSE Stability — Calibrated Model")
plt.xlabel("Date")
plt.ylabel("RMSE")
plt.legend()
plt.grid(True)
plt.show()


plt.figure(figsize=(14, 5))

plt.plot(
    weekly_metrics.index,
    weekly_metrics["MAE"],
    marker="o",
    label="Weekly MAE"
)

plt.axhline(
    overall_metrics["MAE"],
    color="black",
    linestyle="--",
    label="Overall MAE"
)

plt.title("Weekly MAE Stability — Calibrated Model")
plt.xlabel("Week")
plt.ylabel("MAE")
plt.legend()
plt.grid(True)
plt.show()


plt.figure(figsize=(14, 5))

plt.plot(
    weekly_metrics.index,
    weekly_metrics["RMSE"],
    marker="o",
    label="Weekly RMSE"
)

plt.axhline(
    overall_metrics["RMSE"],
    color="black",
    linestyle="--",
    label="Overall RMSE"
)

plt.title("Weekly RMSE Stability — Calibrated Model")
plt.xlabel("Week")
plt.ylabel("RMSE")
plt.legend()
plt.grid(True)
plt.show()


# ============================================================
# 11. RESIDUAL AUTOCORRELATION
# ============================================================

max_lag = 48

acf_values = []

for lag in range(1, max_lag + 1):
    acf = df_calibrated["error"].autocorr(lag=lag)
    acf_values.append({
        "lag": lag,
        "residual_autocorrelation": acf
    })

acf_df = pd.DataFrame(acf_values)

print("\nRESIDUAL AUTOCORRELATION — CALIBRATED MODEL")
print("===========================================")
display(acf_df.head(20))


plt.figure(figsize=(12, 4))

plt.bar(
    acf_df["lag"],
    acf_df["residual_autocorrelation"]
)

plt.axhline(
    0,
    color="black",
    linestyle="--"
)

plt.title("Residual Autocorrelation — Calibrated Model")
plt.xlabel("Lag")
plt.ylabel("Autocorrelation")
plt.grid(True)
plt.show()


# ============================================================
# 12. STABILITY SUMMARY
# ============================================================

daily_mae_mean = daily_metrics["MAE"].mean()
daily_mae_std = daily_metrics["MAE"].std()
daily_mae_cv = daily_mae_std / daily_mae_mean

weekly_mae_mean = weekly_metrics["MAE"].mean()
weekly_mae_std = weekly_metrics["MAE"].std()
weekly_mae_cv = weekly_mae_std / weekly_mae_mean

max_rolling_mae_24 = df_calibrated["rolling_mae_24h"].max()
mean_rolling_mae_24 = df_calibrated["rolling_mae_24h"].mean()

max_rolling_rmse_24 = df_calibrated["rolling_rmse_24h"].max()
mean_rolling_rmse_24 = df_calibrated["rolling_rmse_24h"].mean()

stability_summary_calibrated = pd.DataFrame([
    {
        "metric": "Overall MAE",
        "value": overall_metrics["MAE"]
    },
    {
        "metric": "Overall RMSE",
        "value": overall_metrics["RMSE"]
    },
    {
        "metric": "Overall R2",
        "value": overall_metrics["R2"]
    },
    {
        "metric": "Daily MAE coefficient of variation",
        "value": daily_mae_cv
    },
    {
        "metric": "Weekly MAE coefficient of variation",
        "value": weekly_mae_cv
    },
    {
        "metric": "Max 24h rolling MAE",
        "value": max_rolling_mae_24
    },
    {
        "metric": "Mean 24h rolling MAE",
        "value": mean_rolling_mae_24
    },
    {
        "metric": "Max 24h rolling RMSE",
        "value": max_rolling_rmse_24
    },
    {
        "metric": "Mean 24h rolling RMSE",
        "value": mean_rolling_rmse_24
    }
])

print("\nSTABILITY SUMMARY — CALIBRATED MODEL")
print("====================================")
display(stability_summary_calibrated)


# ============================================================
# 13. AUTOMATIC INTERPRETATION
# ============================================================

print("\nINTERPRETATION GUIDE — CALIBRATED MODEL")
print("=======================================")

print(f"Overall MAE: {overall_metrics['MAE']:.4f}")
print(f"Overall RMSE: {overall_metrics['RMSE']:.4f}")
print(f"Overall R2: {overall_metrics['R2']:.4f}")
print(f"Daily MAE CV: {daily_mae_cv:.4f}")
print(f"Weekly MAE CV: {weekly_mae_cv:.4f}")
print(f"Max 24h rolling MAE: {max_rolling_mae_24:.4f}")
print(f"Mean 24h rolling MAE: {mean_rolling_mae_24:.4f}")

if daily_mae_cv < 0.30:
    print("\nDaily stability: Good. Daily errors are relatively stable.")
elif daily_mae_cv < 0.60:
    print("\nDaily stability: Medium. Some days are harder than others.")
else:
    print("\nDaily stability: Weak. The model still has strong local instability.")

if max_rolling_mae_24 > 2 * overall_metrics["MAE"]:
    print(
        "Local regime warning: Some 24h periods have MAE more than twice "
        "the overall MAE. Investigate worst_24h_periods."
    )
else:
    print(
        "Local regime check: No extreme 24h instability based on MAE threshold."
    )

top_hour = error_by_hour.sort_values("MAE", ascending=False).iloc[0]
top_day = error_by_day.sort_values("MAE", ascending=False).iloc[0]

print(
    f"\nHardest hour: {int(top_hour['hour'])}:00 "
    f"with MAE = {top_hour['MAE']:.4f}"
)

print(
    f"Hardest day: {top_day['day_name']} "
    f"with MAE = {top_day['MAE']:.4f}"
)


# ============================================================
# 14. COMPARE CALIBRATED VS NON-CALIBRATED IF AVAILABLE
# ============================================================

if "static_ensemble_pred" in df_calibrated.columns:

    static_metrics = evaluate_model(
        df_calibrated["y_true"],
        df_calibrated["static_ensemble_pred"]
    )

    comparison_calibration = pd.DataFrame([
        {
            "model": "Static ensemble",
            **static_metrics
        },
        {
            "model": "Calibrated model",
            **overall_metrics
        }
    ]).sort_values("RMSE")

    print("\nCALIBRATION COMPARISON")
    print("======================")
    display(comparison_calibration)

    df_calibrated["static_abs_error"] = np.abs(
        df_calibrated["y_true"] - df_calibrated["static_ensemble_pred"]
    )

    df_calibrated["calibrated_abs_error"] = df_calibrated["abs_error"]

    df_calibrated["static_rolling_mae_24h"] = (
        df_calibrated["static_abs_error"]
        .rolling(24, min_periods=6)
        .mean()
    )

    df_calibrated["calibrated_rolling_mae_24h"] = (
        df_calibrated["calibrated_abs_error"]
        .rolling(24, min_periods=6)
        .mean()
    )

    plt.figure(figsize=(16, 5))

    plt.plot(
        df_calibrated.index,
        df_calibrated["static_rolling_mae_24h"],
        label="Static ensemble rolling MAE 24h",
        alpha=0.8
    )

    plt.plot(
        df_calibrated.index,
        df_calibrated["calibrated_rolling_mae_24h"],
        label="Calibrated rolling MAE 24h",
        alpha=0.8
    )

    plt.title("Rolling MAE Comparison — Static vs Calibrated")
    plt.xlabel("Date")
    plt.ylabel("Rolling MAE")
    plt.legend()
    plt.grid(True)
    plt.show()


# ============================================================
# 15. SAVE OUTPUTS
# ============================================================

df_calibrated.to_csv("calibrated_model_stability_full_results.csv")
error_by_hour.to_csv("calibrated_error_by_hour.csv", index=False)
error_by_day.to_csv("calibrated_error_by_day.csv", index=False)
daily_metrics.to_csv("calibrated_daily_stability_metrics.csv")
weekly_metrics.to_csv("calibrated_weekly_stability_metrics.csv")
worst_24h_periods.to_csv("calibrated_worst_24h_periods.csv")
acf_df.to_csv("calibrated_residual_autocorrelation.csv", index=False)
stability_summary_calibrated.to_csv("calibrated_stability_summary.csv", index=False)

print("\nSaved files:")
print("calibrated_model_stability_full_results.csv")
print("calibrated_error_by_hour.csv")
print("calibrated_error_by_day.csv")
print("calibrated_daily_stability_metrics.csv")
print("calibrated_weekly_stability_metrics.csv")
print("calibrated_worst_24h_periods.csv")
print("calibrated_residual_autocorrelation.csv")
print("calibrated_stability_summary.csv")

In [ ]:
# ============================================================
# LOCAL CALIBRATION FOR THE PROBLEMATIC PERIOD
# Target period: 2005-02-11 to 2005-02-12
# Model: calibrated / ensemble prediction
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ============================================================
# 0. CHOOSE DATAFRAME AND PREDICTION COLUMN
# ============================================================
# This code expects one of:
#
# final_online_df with:
#   - y_true
#   - online_calibrated_pred
#
# OR ensemble_df with:
#   - y_true
#   - online_calibrated_pred / final_ensemble_pred / best_weighted_pred
#
# If your prediction column has a different name, set pred_col manually below.
# ============================================================

if "final_online_df" in globals():
    df_local = final_online_df.copy()
    print("Using final_online_df")

elif "ensemble_df" in globals():
    df_local = ensemble_df.copy()
    print("Using ensemble_df")

else:
    raise ValueError("No dataframe found. You need final_online_df or ensemble_df.")

df_local = df_local.sort_index().copy()

if "y_true" not in df_local.columns:
    raise ValueError("y_true column is missing.")

# ------------------------------------------------------------
# Automatic prediction column detection
# ------------------------------------------------------------

possible_pred_cols = [
    "online_calibrated_pred",
    "final_ensemble_pred",
    "best_weighted_pred",
    "static_ensemble_pred",
    "calibrated_pred"
]

pred_col = None

for col in possible_pred_cols:
    if col in df_local.columns:
        pred_col = col
        break

if pred_col is None:
    print("Available columns:")
    print(df_local.columns.tolist())
    raise ValueError("No prediction column found. Set pred_col manually.")

print("Using base prediction column:", pred_col)
print("Data shape:", df_local.shape)


# ============================================================
# 1. DEFINE TARGET LOCAL PERIOD
# ============================================================
# We use February 11–12, 2005 because your worst period was:
# 2005-02-11 09:00 → 2005-02-12 08:00
#
# If you really mean December 11–12, change dates below.
# ============================================================

event_start = pd.Timestamp("2005-02-11 00:00:00")
event_end = pd.Timestamp("2005-02-12 23:00:00")

df_local["is_target_event"] = (
    (df_local.index >= event_start)
    & (df_local.index <= event_end)
).astype(int)

target_event_df = df_local[df_local["is_target_event"] == 1].copy()

print("Target event start:", event_start)
print("Target event end:", event_end)
print("Target event observations:", len(target_event_df))

if len(target_event_df) == 0:
    raise ValueError("No observations found in the target event period. Check the dates.")


# ============================================================
# 2. METRICS
# ============================================================

def smape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    denominator = np.where(denominator == 0, 1e-8, denominator)

    return np.mean(np.abs(y_true - y_pred) / denominator) * 100


def safe_r2(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    if len(y_true) < 3:
        return np.nan

    if np.std(y_true) == 0:
        return np.nan

    return r2_score(y_true, y_pred)


def evaluate_model(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "sMAPE": smape(y_true, y_pred),
        "R2": safe_r2(y_true, y_pred),
        "Bias": np.mean(y_true - y_pred)
    }


# ============================================================
# 3. BASELINE ERRORS BEFORE LOCAL CALIBRATION
# ============================================================

df_local["base_pred"] = df_local[pred_col]
df_local["base_residual"] = df_local["y_true"] - df_local["base_pred"]
df_local["base_abs_error"] = np.abs(df_local["base_residual"])

base_full_metrics = evaluate_model(
    df_local["y_true"],
    df_local["base_pred"]
)

base_event_metrics = evaluate_model(
    target_event_df["y_true"],
    target_event_df[pred_col]
)

print("\nBASE MODEL FULL TEST METRICS")
print("============================")
print(base_full_metrics)

print("\nBASE MODEL TARGET EVENT METRICS")
print("===============================")
print(base_event_metrics)


# ============================================================
# 4. SAME-HOUR PAST RESIDUAL CORRECTION
# ============================================================
# For each timestamp t, use residuals from the same hour in previous days:
#
# correction(t) = average residual at:
# t-24h, t-48h, t-72h, ...
#
# This helps if the model has systematic hour-specific local bias.
# ============================================================

def compute_same_hour_past_residual(df, residual_col, lookback_days=7):
    corrections = []

    residual_series = df[residual_col]

    for current_time in df.index:
        past_values = []

        for d in range(1, lookback_days + 1):
            past_time = current_time - pd.Timedelta(days=d)

            if past_time in residual_series.index:
                past_values.append(residual_series.loc[past_time])

        if len(past_values) == 0:
            corrections.append(0.0)
        else:
            corrections.append(np.mean(past_values))

    return pd.Series(corrections, index=df.index)


# ============================================================
# 5. LOCAL CALIBRATION GRID
# ============================================================
# We test several correction styles:
#
# A) EWMA past residual correction
# B) rolling past residual correction
# C) same-hour past residual correction
# D) combined EWMA + same-hour correction
#
# The correction is applied ONLY during 2005-02-11 to 2005-02-12.
# Outside this period, prediction remains unchanged.
#
# Important:
# All residual corrections are shifted or based on past timestamps only.
# ============================================================

calibration_results = []

df_work = df_local.copy()

# ------------------------------------------------------------
# Past residual features
# ------------------------------------------------------------

for span in [2, 3, 6, 12, 24, 48]:
    df_work[f"ewm_residual_span_{span}"] = (
        df_work["base_residual"]
        .shift(1)
        .ewm(span=span, adjust=False, min_periods=2)
        .mean()
        .fillna(0)
    )

for window in [3, 6, 12, 24, 48]:
    df_work[f"rolling_residual_window_{window}"] = (
        df_work["base_residual"]
        .shift(1)
        .rolling(window, min_periods=2)
        .mean()
        .fillna(0)
    )

for lookback_days in [2, 3, 5, 7, 10, 14]:
    df_work[f"same_hour_residual_{lookback_days}d"] = compute_same_hour_past_residual(
        df_work,
        residual_col="base_residual",
        lookback_days=lookback_days
    )


# ============================================================
# 6. FUNCTION TO APPLY LOCAL CORRECTION
# ============================================================

def apply_local_correction(df, correction, alpha, output_col):
    df = df.copy()

    df[output_col] = df["base_pred"].copy()

    event_mask = df["is_target_event"] == 1

    df.loc[event_mask, output_col] = (
        df.loc[event_mask, "base_pred"]
        + alpha * correction.loc[event_mask]
    )

    return df


# ============================================================
# 7. TEST EWMA CORRECTION
# ============================================================

for span in [2, 3, 6, 12, 24, 48]:
    correction_col = f"ewm_residual_span_{span}"
    correction = df_work[correction_col]

    for alpha in [-0.50, -0.25, 0.25, 0.50, 0.75, 1.00, 1.25, 1.50]:

        temp = apply_local_correction(
            df_work,
            correction=correction,
            alpha=alpha,
            output_col="local_calibrated_pred"
        )

        full_metrics = evaluate_model(
            temp["y_true"],
            temp["local_calibrated_pred"]
        )

        event_metrics = evaluate_model(
            temp.loc[temp["is_target_event"] == 1, "y_true"],
            temp.loc[temp["is_target_event"] == 1, "local_calibrated_pred"]
        )

        calibration_results.append({
            "method": "EWMA",
            "param": f"span={span}",
            "alpha": alpha,
            "full_MAE": full_metrics["MAE"],
            "full_RMSE": full_metrics["RMSE"],
            "full_R2": full_metrics["R2"],
            "event_MAE": event_metrics["MAE"],
            "event_RMSE": event_metrics["RMSE"],
            "event_R2": event_metrics["R2"],
            "event_Bias": event_metrics["Bias"]
        })


# ============================================================
# 8. TEST ROLLING RESIDUAL CORRECTION
# ============================================================

for window in [3, 6, 12, 24, 48]:
    correction_col = f"rolling_residual_window_{window}"
    correction = df_work[correction_col]

    for alpha in [-0.50, -0.25, 0.25, 0.50, 0.75, 1.00, 1.25, 1.50]:

        temp = apply_local_correction(
            df_work,
            correction=correction,
            alpha=alpha,
            output_col="local_calibrated_pred"
        )

        full_metrics = evaluate_model(
            temp["y_true"],
            temp["local_calibrated_pred"]
        )

        event_metrics = evaluate_model(
            temp.loc[temp["is_target_event"] == 1, "y_true"],
            temp.loc[temp["is_target_event"] == 1, "local_calibrated_pred"]
        )

        calibration_results.append({
            "method": "Rolling",
            "param": f"window={window}",
            "alpha": alpha,
            "full_MAE": full_metrics["MAE"],
            "full_RMSE": full_metrics["RMSE"],
            "full_R2": full_metrics["R2"],
            "event_MAE": event_metrics["MAE"],
            "event_RMSE": event_metrics["RMSE"],
            "event_R2": event_metrics["R2"],
            "event_Bias": event_metrics["Bias"]
        })


# ============================================================
# 9. TEST SAME-HOUR RESIDUAL CORRECTION
# ============================================================

for lookback_days in [2, 3, 5, 7, 10, 14]:
    correction_col = f"same_hour_residual_{lookback_days}d"
    correction = df_work[correction_col]

    for alpha in [-0.50, -0.25, 0.25, 0.50, 0.75, 1.00, 1.25, 1.50]:

        temp = apply_local_correction(
            df_work,
            correction=correction,
            alpha=alpha,
            output_col="local_calibrated_pred"
        )

        full_metrics = evaluate_model(
            temp["y_true"],
            temp["local_calibrated_pred"]
        )

        event_metrics = evaluate_model(
            temp.loc[temp["is_target_event"] == 1, "y_true"],
            temp.loc[temp["is_target_event"] == 1, "local_calibrated_pred"]
        )

        calibration_results.append({
            "method": "Same-hour",
            "param": f"lookback_days={lookback_days}",
            "alpha": alpha,
            "full_MAE": full_metrics["MAE"],
            "full_RMSE": full_metrics["RMSE"],
            "full_R2": full_metrics["R2"],
            "event_MAE": event_metrics["MAE"],
            "event_RMSE": event_metrics["RMSE"],
            "event_R2": event_metrics["R2"],
            "event_Bias": event_metrics["Bias"]
        })


# ============================================================
# 10. TEST COMBINED EWMA + SAME-HOUR CORRECTION
# ============================================================

combined_results = []

for span in [3, 6, 12, 24]:
    ewm_col = f"ewm_residual_span_{span}"

    for lookback_days in [3, 5, 7, 10]:
        same_col = f"same_hour_residual_{lookback_days}d"

        for alpha_ewm in [0.25, 0.50, 0.75, 1.00]:
            for alpha_same in [0.25, 0.50, 0.75, 1.00]:

                correction = (
                    alpha_ewm * df_work[ewm_col]
                    + alpha_same * df_work[same_col]
                )

                temp = df_work.copy()
                temp["local_calibrated_pred"] = temp["base_pred"]

                event_mask = temp["is_target_event"] == 1

                temp.loc[event_mask, "local_calibrated_pred"] = (
                    temp.loc[event_mask, "base_pred"]
                    + correction.loc[event_mask]
                )

                full_metrics = evaluate_model(
                    temp["y_true"],
                    temp["local_calibrated_pred"]
                )

                event_metrics = evaluate_model(
                    temp.loc[event_mask, "y_true"],
                    temp.loc[event_mask, "local_calibrated_pred"]
                )

                calibration_results.append({
                    "method": "Combined_EWMA_same_hour",
                    "param": f"span={span}, lookback_days={lookback_days}, alpha_ewm={alpha_ewm}, alpha_same={alpha_same}",
                    "alpha": np.nan,
                    "full_MAE": full_metrics["MAE"],
                    "full_RMSE": full_metrics["RMSE"],
                    "full_R2": full_metrics["R2"],
                    "event_MAE": event_metrics["MAE"],
                    "event_RMSE": event_metrics["RMSE"],
                    "event_R2": event_metrics["R2"],
                    "event_Bias": event_metrics["Bias"]
                })


# ============================================================
# 11. SELECT BEST LOCAL CALIBRATION
# ============================================================
# We optimize primarily on event_RMSE because the goal is to fix this local period.
# But we also check the full metrics to make sure we did not damage the whole test.
# ============================================================

calibration_results_df = pd.DataFrame(calibration_results)

calibration_results_df = calibration_results_df.sort_values("event_RMSE")

print("\nTOP 20 LOCAL CALIBRATION CANDIDATES")
print("===================================")
display(calibration_results_df.head(20))

best_local_setup = calibration_results_df.iloc[0]

print("\nBEST LOCAL CALIBRATION SETUP")
print("============================")
print(best_local_setup)


# ============================================================
# 12. REBUILD BEST LOCAL CALIBRATED PREDICTION
# ============================================================

best_method = best_local_setup["method"]
best_param = best_local_setup["param"]
best_alpha = best_local_setup["alpha"]

df_final_local = df_work.copy()
df_final_local["local_calibrated_pred"] = df_final_local["base_pred"]
event_mask = df_final_local["is_target_event"] == 1

if best_method == "EWMA":

    span = int(best_param.replace("span=", ""))
    correction = df_final_local[f"ewm_residual_span_{span}"]

    df_final_local.loc[event_mask, "local_calibrated_pred"] = (
        df_final_local.loc[event_mask, "base_pred"]
        + best_alpha * correction.loc[event_mask]
    )

elif best_method == "Rolling":

    window = int(best_param.replace("window=", ""))
    correction = df_final_local[f"rolling_residual_window_{window}"]

    df_final_local.loc[event_mask, "local_calibrated_pred"] = (
        df_final_local.loc[event_mask, "base_pred"]
        + best_alpha * correction.loc[event_mask]
    )

elif best_method == "Same-hour":

    lookback_days = int(best_param.replace("lookback_days=", ""))
    correction = df_final_local[f"same_hour_residual_{lookback_days}d"]

    df_final_local.loc[event_mask, "local_calibrated_pred"] = (
        df_final_local.loc[event_mask, "base_pred"]
        + best_alpha * correction.loc[event_mask]
    )

elif best_method == "Combined_EWMA_same_hour":

    # Parse string like:
    # span=3, lookback_days=7, alpha_ewm=0.5, alpha_same=0.75
    parts = best_param.split(",")

    span = int(parts[0].split("=")[1].strip())
    lookback_days = int(parts[1].split("=")[1].strip())
    alpha_ewm = float(parts[2].split("=")[1].strip())
    alpha_same = float(parts[3].split("=")[1].strip())

    correction = (
        alpha_ewm * df_final_local[f"ewm_residual_span_{span}"]
        + alpha_same * df_final_local[f"same_hour_residual_{lookback_days}d"]
    )

    df_final_local.loc[event_mask, "local_calibrated_pred"] = (
        df_final_local.loc[event_mask, "base_pred"]
        + correction.loc[event_mask]
    )

else:
    raise ValueError("Unknown calibration method.")


# ============================================================
# 13. FINAL METRICS BEFORE VS AFTER
# ============================================================

base_full_metrics = evaluate_model(
    df_final_local["y_true"],
    df_final_local["base_pred"]
)

local_full_metrics = evaluate_model(
    df_final_local["y_true"],
    df_final_local["local_calibrated_pred"]
)

base_event_metrics = evaluate_model(
    df_final_local.loc[event_mask, "y_true"],
    df_final_local.loc[event_mask, "base_pred"]
)

local_event_metrics = evaluate_model(
    df_final_local.loc[event_mask, "y_true"],
    df_final_local.loc[event_mask, "local_calibrated_pred"]
)

comparison_local_calibration = pd.DataFrame([
    {
        "scope": "Full test",
        "model": "Before local calibration",
        **base_full_metrics
    },
    {
        "scope": "Full test",
        "model": "After local calibration",
        **local_full_metrics
    },
    {
        "scope": "Target event only",
        "model": "Before local calibration",
        **base_event_metrics
    },
    {
        "scope": "Target event only",
        "model": "After local calibration",
        **local_event_metrics
    }
])

print("\nBEFORE VS AFTER LOCAL CALIBRATION")
print("=================================")
display(comparison_local_calibration)


# ============================================================
# 14. ERROR IMPROVEMENT DURING EVENT
# ============================================================

event_rmse_improvement = (
    base_event_metrics["RMSE"] - local_event_metrics["RMSE"]
) / base_event_metrics["RMSE"] * 100

event_mae_improvement = (
    base_event_metrics["MAE"] - local_event_metrics["MAE"]
) / base_event_metrics["MAE"] * 100

full_rmse_improvement = (
    base_full_metrics["RMSE"] - local_full_metrics["RMSE"]
) / base_full_metrics["RMSE"] * 100

full_mae_improvement = (
    base_full_metrics["MAE"] - local_full_metrics["MAE"]
) / base_full_metrics["MAE"] * 100

print("\nIMPROVEMENT")
print("===========")
print("Event RMSE improvement (%):", event_rmse_improvement)
print("Event MAE improvement (%):", event_mae_improvement)
print("Full test RMSE improvement (%):", full_rmse_improvement)
print("Full test MAE improvement (%):", full_mae_improvement)


# ============================================================
# 15. PLOT TARGET EVENT BEFORE VS AFTER
# ============================================================

event_plot_df = df_final_local.loc[event_start:event_end].copy()

plt.figure(figsize=(16, 5))

plt.plot(
    event_plot_df.index,
    event_plot_df["y_true"],
    label="Actual CO(GT)",
    marker="o",
    alpha=0.85
)

plt.plot(
    event_plot_df.index,
    event_plot_df["base_pred"],
    label="Before local calibration",
    marker="o",
    alpha=0.65
)

plt.plot(
    event_plot_df.index,
    event_plot_df["local_calibrated_pred"],
    label="After local calibration",
    marker="o",
    alpha=0.9
)

plt.title("Target Period Calibration — 2005-02-11 to 2005-02-12")
plt.xlabel("Date")
plt.ylabel("CO(GT)")
plt.legend()
plt.grid(True)
plt.show()


# ============================================================
# 16. PLOT FULL TEST BEFORE VS AFTER
# ============================================================

plt.figure(figsize=(16, 5))

plt.plot(
    df_final_local.index,
    df_final_local["y_true"],
    label="Actual CO(GT)",
    alpha=0.75
)

plt.plot(
    df_final_local.index,
    df_final_local["base_pred"],
    label="Before local calibration",
    alpha=0.55
)

plt.plot(
    df_final_local.index,
    df_final_local["local_calibrated_pred"],
    label="After local calibration",
    alpha=0.85
)

plt.axvspan(
    event_start,
    event_end,
    alpha=0.2,
    label="Locally calibrated period"
)

plt.title("Full Test — Before vs After Local Event Calibration")
plt.xlabel("Date")
plt.ylabel("CO(GT)")
plt.legend()
plt.grid(True)
plt.show()


# ============================================================
# 17. RESIDUALS DURING TARGET EVENT
# ============================================================

event_plot_df["base_error"] = event_plot_df["y_true"] - event_plot_df["base_pred"]
event_plot_df["local_error"] = event_plot_df["y_true"] - event_plot_df["local_calibrated_pred"]

plt.figure(figsize=(16, 4))

plt.plot(
    event_plot_df.index,
    event_plot_df["base_error"],
    label="Before local calibration residual",
    marker="o",
    alpha=0.7
)

plt.plot(
    event_plot_df.index,
    event_plot_df["local_error"],
    label="After local calibration residual",
    marker="o",
    alpha=0.9
)

plt.axhline(
    0,
    color="black",
    linestyle="--"
)

plt.title("Residuals During Target Period — Before vs After")
plt.xlabel("Date")
plt.ylabel("Residual")
plt.legend()
plt.grid(True)
plt.show()


# ============================================================
# 18. ROW-BY-ROW EVENT COMPARISON
# ============================================================

event_comparison_rows = event_plot_df[
    [
        "y_true",
        "base_pred",
        "local_calibrated_pred",
        "base_error",
        "local_error"
    ]
].copy()

event_comparison_rows["base_abs_error"] = np.abs(event_comparison_rows["base_error"])
event_comparison_rows["local_abs_error"] = np.abs(event_comparison_rows["local_error"])
event_comparison_rows["abs_error_improvement"] = (
    event_comparison_rows["base_abs_error"]
    - event_comparison_rows["local_abs_error"]
)

print("\nEVENT ROW-BY-ROW COMPARISON")
print("===========================")
display(event_comparison_rows)


# ============================================================
# 19. SAVE RESULTS
# ============================================================

df_final_local.to_csv("locally_calibrated_2005_02_11_12_full_results.csv")
calibration_results_df.to_csv("local_calibration_grid_results_2005_02_11_12.csv", index=False)
comparison_local_calibration.to_csv("local_calibration_comparison_2005_02_11_12.csv", index=False)
event_comparison_rows.to_csv("local_calibration_event_rows_2005_02_11_12.csv")

print("\nSaved files:")
print("locally_calibrated_2005_02_11_12_full_results.csv")
print("local_calibration_grid_results_2005_02_11_12.csv")
print("local_calibration_comparison_2005_02_11_12.csv")
print("local_calibration_event_rows_2005_02_11_12.csv")


# ============================================================
# 20. IMPORTANT WARNING
# ============================================================

print("\nIMPORTANT WARNING")
print("=================")
print(
    "This is a targeted local calibration for a known problematic test period. "
    "It is useful as a diagnostic experiment. "
    "However, if the period was selected after looking at test errors, "
    "do not report the improved score as a fully unbiased final test score. "
    "Use it as a local regime correction experiment or operational calibration example."
)

In [ ]:
# ============================================================
# STABILITY TEST FOR LATEST CALIBRATED MODEL
# Works with:
# - local_calibrated_pred
# - online_calibrated_pred
# - final_ensemble_pred
# - best_weighted_pred
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ============================================================
# 0. SELECT DATAFRAME
# ============================================================

if "df_final_local" in globals():
    df_stability = df_final_local.copy()
    print("Using df_final_local")

elif "final_online_df" in globals():
    df_stability = final_online_df.copy()
    print("Using final_online_df")

elif "ensemble_df" in globals():
    df_stability = ensemble_df.copy()
    print("Using ensemble_df")

else:
    raise ValueError(
        "No dataframe found. You need df_final_local, final_online_df, or ensemble_df."
    )

df_stability = df_stability.sort_index().copy()

if "y_true" not in df_stability.columns:
    raise ValueError("y_true column is missing.")

# ============================================================
# 1. SELECT BEST PREDICTION COLUMN
# ============================================================

possible_pred_cols = [
    "local_calibrated_pred",
    "online_calibrated_pred",
    "final_ensemble_pred",
    "best_weighted_pred",
    "static_ensemble_pred"
]

pred_col = None

for col in possible_pred_cols:
    if col in df_stability.columns:
        pred_col = col
        break

if pred_col is None:
    print("Available columns:")
    print(df_stability.columns.tolist())
    raise ValueError("No valid prediction column found.")

print("Using prediction column:", pred_col)
print("Data shape:", df_stability.shape)

display(df_stability.head())


# ============================================================
# 2. METRIC FUNCTIONS
# ============================================================

def smape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    denominator = np.where(denominator == 0, 1e-8, denominator)

    return np.mean(np.abs(y_true - y_pred) / denominator) * 100


def safe_r2(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    if len(y_true) < 3:
        return np.nan

    if np.std(y_true) == 0:
        return np.nan

    return r2_score(y_true, y_pred)


def evaluate_model(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "sMAPE": smape(y_true, y_pred),
        "R2": safe_r2(y_true, y_pred),
        "Bias": np.mean(y_true - y_pred)
    }


# ============================================================
# 3. BASIC ERROR COLUMNS
# ============================================================

df_stability["error"] = df_stability["y_true"] - df_stability[pred_col]
df_stability["abs_error"] = np.abs(df_stability["error"])
df_stability["squared_error"] = df_stability["error"] ** 2

df_stability["hour"] = df_stability.index.hour
df_stability["dayofweek"] = df_stability.index.dayofweek
df_stability["is_weekend"] = (df_stability["dayofweek"] >= 5).astype(int)

day_names = {
    0: "Monday",
    1: "Tuesday",
    2: "Wednesday",
    3: "Thursday",
    4: "Friday",
    5: "Saturday",
    6: "Sunday"
}

df_stability["day_name"] = df_stability["dayofweek"].map(day_names)

overall_metrics = evaluate_model(
    df_stability["y_true"],
    df_stability[pred_col]
)

print("\nOVERALL STABILITY MODEL PERFORMANCE")
print("===================================")
print(overall_metrics)


# ============================================================
# 4. ACTUAL VS PREDICTED
# ============================================================

plt.figure(figsize=(16, 5))

plt.plot(
    df_stability.index,
    df_stability["y_true"],
    label="Actual CO(GT)",
    alpha=0.8
)

plt.plot(
    df_stability.index,
    df_stability[pred_col],
    label=f"Prediction: {pred_col}",
    alpha=0.85
)

plt.title(f"Actual vs Predicted — {pred_col}")
plt.xlabel("Date")
plt.ylabel("CO(GT)")
plt.legend()
plt.grid(True)
plt.show()


# ============================================================
# 5. RESIDUALS OVER TIME
# ============================================================

plt.figure(figsize=(16, 4))

plt.plot(
    df_stability.index,
    df_stability["error"],
    label="Residual = actual - predicted",
    alpha=0.8
)

plt.axhline(
    0,
    color="black",
    linestyle="--"
)

plt.title(f"Residuals Over Time — {pred_col}")
plt.xlabel("Date")
plt.ylabel("Error")
plt.legend()
plt.grid(True)
plt.show()


# ============================================================
# 6. ROLLING STABILITY
# ============================================================

rolling_windows = [24, 48, 72, 168]

for window in rolling_windows:

    min_periods = max(6, window // 4)

    df_stability[f"rolling_mae_{window}h"] = (
        df_stability["abs_error"]
        .rolling(window, min_periods=min_periods)
        .mean()
    )

    df_stability[f"rolling_rmse_{window}h"] = (
        df_stability["squared_error"]
        .rolling(window, min_periods=min_periods)
        .mean()
        .apply(np.sqrt)
    )

    df_stability[f"rolling_bias_{window}h"] = (
        df_stability["error"]
        .rolling(window, min_periods=min_periods)
        .mean()
    )


# ------------------------------------------------------------
# Rolling MAE
# ------------------------------------------------------------

plt.figure(figsize=(16, 5))

for window in rolling_windows:
    plt.plot(
        df_stability.index,
        df_stability[f"rolling_mae_{window}h"],
        label=f"Rolling MAE {window}h",
        alpha=0.8
    )

plt.axhline(
    overall_metrics["MAE"],
    color="black",
    linestyle="--",
    label="Overall MAE"
)

plt.title("Rolling MAE — Stability Over Time")
plt.xlabel("Date")
plt.ylabel("MAE")
plt.legend()
plt.grid(True)
plt.show()


# ------------------------------------------------------------
# Rolling RMSE
# ------------------------------------------------------------

plt.figure(figsize=(16, 5))

for window in rolling_windows:
    plt.plot(
        df_stability.index,
        df_stability[f"rolling_rmse_{window}h"],
        label=f"Rolling RMSE {window}h",
        alpha=0.8
    )

plt.axhline(
    overall_metrics["RMSE"],
    color="black",
    linestyle="--",
    label="Overall RMSE"
)

plt.title("Rolling RMSE — Stability Over Time")
plt.xlabel("Date")
plt.ylabel("RMSE")
plt.legend()
plt.grid(True)
plt.show()


# ------------------------------------------------------------
# Rolling bias
# ------------------------------------------------------------

plt.figure(figsize=(16, 5))

for window in [24, 72, 168]:
    plt.plot(
        df_stability.index,
        df_stability[f"rolling_bias_{window}h"],
        label=f"Rolling bias {window}h",
        alpha=0.8
    )

plt.axhline(
    0,
    color="black",
    linestyle="--",
    label="Zero bias"
)

plt.title("Rolling Bias — Positive = Underprediction, Negative = Overprediction")
plt.xlabel("Date")
plt.ylabel("Bias")
plt.legend()
plt.grid(True)
plt.show()


# ============================================================
# 7. WORST 24H LOCAL PERIODS
# ============================================================

worst_24h_periods = (
    df_stability[
        [
            "y_true",
            pred_col,
            "error",
            "abs_error",
            "rolling_mae_24h",
            "rolling_rmse_24h",
            "rolling_bias_24h"
        ]
    ]
    .dropna()
    .sort_values("rolling_mae_24h", ascending=False)
    .head(20)
)

print("\nWORST 24H LOCAL PERIODS")
print("=======================")
display(worst_24h_periods)


# ============================================================
# 8. ERROR BY HOUR
# ============================================================

error_by_hour = (
    df_stability
    .groupby("hour")
    .apply(
        lambda g: pd.Series({
            "count": len(g),
            "MAE": mean_absolute_error(g["y_true"], g[pred_col]),
            "RMSE": np.sqrt(mean_squared_error(g["y_true"], g[pred_col])),
            "R2": safe_r2(g["y_true"], g[pred_col]),
            "Bias": np.mean(g["y_true"] - g[pred_col]),
            "Mean_actual": g["y_true"].mean(),
            "Mean_pred": g[pred_col].mean()
        })
    )
    .reset_index()
)

print("\nERROR BY HOUR")
print("=============")
display(error_by_hour)


plt.figure(figsize=(12, 4))

plt.bar(
    error_by_hour["hour"],
    error_by_hour["MAE"]
)

plt.axhline(
    overall_metrics["MAE"],
    color="black",
    linestyle="--",
    label="Overall MAE"
)

plt.title("MAE by Hour")
plt.xlabel("Hour")
plt.ylabel("MAE")
plt.xticks(range(24))
plt.legend()
plt.grid(True)
plt.show()


plt.figure(figsize=(12, 4))

plt.bar(
    error_by_hour["hour"],
    error_by_hour["RMSE"]
)

plt.axhline(
    overall_metrics["RMSE"],
    color="black",
    linestyle="--",
    label="Overall RMSE"
)

plt.title("RMSE by Hour")
plt.xlabel("Hour")
plt.ylabel("RMSE")
plt.xticks(range(24))
plt.legend()
plt.grid(True)
plt.show()


plt.figure(figsize=(12, 4))

plt.bar(
    error_by_hour["hour"],
    error_by_hour["Bias"]
)

plt.axhline(
    0,
    color="black",
    linestyle="--"
)

plt.title("Bias by Hour — Positive = Underprediction, Negative = Overprediction")
plt.xlabel("Hour")
plt.ylabel("Bias")
plt.xticks(range(24))
plt.grid(True)
plt.show()


# ============================================================
# 9. ERROR BY DAY OF WEEK
# ============================================================

error_by_day = (
    df_stability
    .groupby(["dayofweek", "day_name"])
    .apply(
        lambda g: pd.Series({
            "count": len(g),
            "MAE": mean_absolute_error(g["y_true"], g[pred_col]),
            "RMSE": np.sqrt(mean_squared_error(g["y_true"], g[pred_col])),
            "R2": safe_r2(g["y_true"], g[pred_col]),
            "Bias": np.mean(g["y_true"] - g[pred_col]),
            "Mean_actual": g["y_true"].mean(),
            "Mean_pred": g[pred_col].mean()
        })
    )
    .reset_index()
    .sort_values("dayofweek")
)

print("\nERROR BY DAY OF WEEK")
print("====================")
display(error_by_day)


plt.figure(figsize=(10, 4))

plt.bar(
    error_by_day["day_name"],
    error_by_day["MAE"]
)

plt.axhline(
    overall_metrics["MAE"],
    color="black",
    linestyle="--",
    label="Overall MAE"
)

plt.title("MAE by Day of Week")
plt.xlabel("Day")
plt.ylabel("MAE")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)
plt.show()


plt.figure(figsize=(10, 4))

plt.bar(
    error_by_day["day_name"],
    error_by_day["RMSE"]
)

plt.axhline(
    overall_metrics["RMSE"],
    color="black",
    linestyle="--",
    label="Overall RMSE"
)

plt.title("RMSE by Day of Week")
plt.xlabel("Day")
plt.ylabel("RMSE")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)
plt.show()


plt.figure(figsize=(10, 4))

plt.bar(
    error_by_day["day_name"],
    error_by_day["Bias"]
)

plt.axhline(
    0,
    color="black",
    linestyle="--"
)

plt.title("Bias by Day — Positive = Underprediction, Negative = Overprediction")
plt.xlabel("Day")
plt.ylabel("Bias")
plt.xticks(rotation=45)
plt.grid(True)
plt.show()


# ============================================================
# 10. DAY × HOUR HEATMAPS
# ============================================================

heatmap_mae = (
    df_stability
    .groupby(["dayofweek", "hour"])["abs_error"]
    .mean()
    .unstack()
)

heatmap_rmse = (
    df_stability
    .groupby(["dayofweek", "hour"])["squared_error"]
    .mean()
    .apply(np.sqrt)
    .unstack()
)

heatmap_bias = (
    df_stability
    .groupby(["dayofweek", "hour"])["error"]
    .mean()
    .unstack()
)

heatmap_mae.index = [day_names[i] for i in heatmap_mae.index]
heatmap_rmse.index = [day_names[i] for i in heatmap_rmse.index]
heatmap_bias.index = [day_names[i] for i in heatmap_bias.index]


plt.figure(figsize=(14, 5))

plt.imshow(
    heatmap_mae,
    aspect="auto"
)

plt.colorbar(label="MAE")
plt.title("MAE Heatmap — Day of Week × Hour")
plt.xlabel("Hour")
plt.ylabel("Day of Week")
plt.xticks(range(24), range(24))
plt.yticks(range(len(heatmap_mae.index)), heatmap_mae.index)
plt.show()


plt.figure(figsize=(14, 5))

plt.imshow(
    heatmap_rmse,
    aspect="auto"
)

plt.colorbar(label="RMSE")
plt.title("RMSE Heatmap — Day of Week × Hour")
plt.xlabel("Hour")
plt.ylabel("Day of Week")
plt.xticks(range(24), range(24))
plt.yticks(range(len(heatmap_rmse.index)), heatmap_rmse.index)
plt.show()


plt.figure(figsize=(14, 5))

plt.imshow(
    heatmap_bias,
    aspect="auto"
)

plt.colorbar(label="Bias")
plt.title("Bias Heatmap — Positive = Underprediction, Negative = Overprediction")
plt.xlabel("Hour")
plt.ylabel("Day of Week")
plt.xticks(range(24), range(24))
plt.yticks(range(len(heatmap_bias.index)), heatmap_bias.index)
plt.show()


# ============================================================
# 11. DAILY AND WEEKLY STABILITY
# ============================================================

def grouped_metrics(g):
    return pd.Series({
        "count": len(g),
        "MAE": mean_absolute_error(g["y_true"], g[pred_col]),
        "RMSE": np.sqrt(mean_squared_error(g["y_true"], g[pred_col])),
        "R2": safe_r2(g["y_true"], g[pred_col]),
        "Bias": np.mean(g["y_true"] - g[pred_col]),
        "Mean_actual": g["y_true"].mean(),
        "Mean_pred": g[pred_col].mean()
    })


daily_metrics = (
    df_stability
    .resample("D")
    .apply(lambda g: grouped_metrics(g) if len(g) > 0 else np.nan)
)

weekly_metrics = (
    df_stability
    .resample("W")
    .apply(lambda g: grouped_metrics(g) if len(g) > 0 else np.nan)
)

print("\nDAILY STABILITY METRICS")
print("=======================")
display(daily_metrics)

print("\nWEEKLY STABILITY METRICS")
print("========================")
display(weekly_metrics)


plt.figure(figsize=(16, 5))

plt.plot(
    daily_metrics.index,
    daily_metrics["MAE"],
    marker="o",
    label="Daily MAE"
)

plt.axhline(
    overall_metrics["MAE"],
    color="black",
    linestyle="--",
    label="Overall MAE"
)

plt.title("Daily MAE Stability")
plt.xlabel("Date")
plt.ylabel("MAE")
plt.legend()
plt.grid(True)
plt.show()


plt.figure(figsize=(16, 5))

plt.plot(
    daily_metrics.index,
    daily_metrics["RMSE"],
    marker="o",
    label="Daily RMSE"
)

plt.axhline(
    overall_metrics["RMSE"],
    color="black",
    linestyle="--",
    label="Overall RMSE"
)

plt.title("Daily RMSE Stability")
plt.xlabel("Date")
plt.ylabel("RMSE")
plt.legend()
plt.grid(True)
plt.show()


plt.figure(figsize=(14, 5))

plt.plot(
    weekly_metrics.index,
    weekly_metrics["MAE"],
    marker="o",
    label="Weekly MAE"
)

plt.axhline(
    overall_metrics["MAE"],
    color="black",
    linestyle="--",
    label="Overall MAE"
)

plt.title("Weekly MAE Stability")
plt.xlabel("Week")
plt.ylabel("MAE")
plt.legend()
plt.grid(True)
plt.show()


plt.figure(figsize=(14, 5))

plt.plot(
    weekly_metrics.index,
    weekly_metrics["RMSE"],
    marker="o",
    label="Weekly RMSE"
)

plt.axhline(
    overall_metrics["RMSE"],
    color="black",
    linestyle="--",
    label="Overall RMSE"
)

plt.title("Weekly RMSE Stability")
plt.xlabel("Week")
plt.ylabel("RMSE")
plt.legend()
plt.grid(True)
plt.show()


# ============================================================
# 12. RESIDUAL AUTOCORRELATION
# ============================================================

max_lag = 48

acf_values = []

for lag in range(1, max_lag + 1):
    acf_values.append({
        "lag": lag,
        "residual_autocorrelation": df_stability["error"].autocorr(lag=lag)
    })

acf_df = pd.DataFrame(acf_values)

print("\nRESIDUAL AUTOCORRELATION")
print("========================")
display(acf_df.head(20))


plt.figure(figsize=(12, 4))

plt.bar(
    acf_df["lag"],
    acf_df["residual_autocorrelation"]
)

plt.axhline(
    0,
    color="black",
    linestyle="--"
)

plt.title("Residual Autocorrelation")
plt.xlabel("Lag")
plt.ylabel("Autocorrelation")
plt.grid(True)
plt.show()


# ============================================================
# 13. STABILITY SUMMARY
# ============================================================

daily_mae_mean = daily_metrics["MAE"].mean()
daily_mae_std = daily_metrics["MAE"].std()
daily_mae_cv = daily_mae_std / daily_mae_mean

weekly_mae_mean = weekly_metrics["MAE"].mean()
weekly_mae_std = weekly_metrics["MAE"].std()
weekly_mae_cv = weekly_mae_std / weekly_mae_mean

max_rolling_mae_24 = df_stability["rolling_mae_24h"].max()
mean_rolling_mae_24 = df_stability["rolling_mae_24h"].mean()

max_rolling_rmse_24 = df_stability["rolling_rmse_24h"].max()
mean_rolling_rmse_24 = df_stability["rolling_rmse_24h"].mean()

stability_summary = pd.DataFrame([
    {
        "metric": "Overall MAE",
        "value": overall_metrics["MAE"]
    },
    {
        "metric": "Overall RMSE",
        "value": overall_metrics["RMSE"]
    },
    {
        "metric": "Overall R2",
        "value": overall_metrics["R2"]
    },
    {
        "metric": "Overall Bias",
        "value": overall_metrics["Bias"]
    },
    {
        "metric": "Daily MAE coefficient of variation",
        "value": daily_mae_cv
    },
    {
        "metric": "Weekly MAE coefficient of variation",
        "value": weekly_mae_cv
    },
    {
        "metric": "Max 24h rolling MAE",
        "value": max_rolling_mae_24
    },
    {
        "metric": "Mean 24h rolling MAE",
        "value": mean_rolling_mae_24
    },
    {
        "metric": "Max 24h rolling RMSE",
        "value": max_rolling_rmse_24
    },
    {
        "metric": "Mean 24h rolling RMSE",
        "value": mean_rolling_rmse_24
    }
])

print("\nSTABILITY SUMMARY")
print("=================")
display(stability_summary)


# ============================================================
# 14. AUTOMATIC INTERPRETATION
# ============================================================

print("\nINTERPRETATION GUIDE")
print("====================")

print(f"Overall MAE: {overall_metrics['MAE']:.4f}")
print(f"Overall RMSE: {overall_metrics['RMSE']:.4f}")
print(f"Overall R2: {overall_metrics['R2']:.4f}")
print(f"Overall Bias: {overall_metrics['Bias']:.4f}")

print(f"Daily MAE CV: {daily_mae_cv:.4f}")
print(f"Weekly MAE CV: {weekly_mae_cv:.4f}")
print(f"Max 24h rolling MAE: {max_rolling_mae_24:.4f}")
print(f"Mean 24h rolling MAE: {mean_rolling_mae_24:.4f}")

if daily_mae_cv < 0.30:
    print("\nDaily stability: Good. Daily errors are relatively stable.")
elif daily_mae_cv < 0.60:
    print("\nDaily stability: Medium. Some days are harder than others.")
else:
    print("\nDaily stability: Weak. The model has strong local instability.")

if weekly_mae_cv < 0.30:
    print("Weekly stability: Good.")
elif weekly_mae_cv < 0.60:
    print("Weekly stability: Medium.")
else:
    print("Weekly stability: Weak.")

if max_rolling_mae_24 > 2 * overall_metrics["MAE"]:
    print(
        "Local regime warning: Some 24h periods have MAE more than twice "
        "the overall MAE. Investigate worst_24h_periods."
    )
else:
    print(
        "Local regime check: No extreme 24h instability based on MAE threshold."
    )

top_hour = error_by_hour.sort_values("MAE", ascending=False).iloc[0]
top_day = error_by_day.sort_values("MAE", ascending=False).iloc[0]

print(
    f"\nHardest hour: {int(top_hour['hour'])}:00 "
    f"with MAE = {top_hour['MAE']:.4f}"
)

print(
    f"Hardest day: {top_day['day_name']} "
    f"with MAE = {top_day['MAE']:.4f}"
)


# ============================================================
# 15. COMPARE BEFORE / AFTER LOCAL CALIBRATION IF AVAILABLE
# ============================================================

if "base_pred" in df_stability.columns and pred_col == "local_calibrated_pred":

    base_metrics = evaluate_model(
        df_stability["y_true"],
        df_stability["base_pred"]
    )

    local_metrics = evaluate_model(
        df_stability["y_true"],
        df_stability["local_calibrated_pred"]
    )

    local_comparison = pd.DataFrame([
        {
            "model": "Before local calibration",
            **base_metrics
        },
        {
            "model": "After local calibration",
            **local_metrics
        }
    ]).sort_values("RMSE")

    print("\nBEFORE VS AFTER LOCAL CALIBRATION")
    print("=================================")
    display(local_comparison)


# ============================================================
# 16. SAVE RESULTS
# ============================================================

df_stability.to_csv("latest_model_stability_full_results.csv")
error_by_hour.to_csv("latest_error_by_hour.csv", index=False)
error_by_day.to_csv("latest_error_by_day.csv", index=False)
daily_metrics.to_csv("latest_daily_stability_metrics.csv")
weekly_metrics.to_csv("latest_weekly_stability_metrics.csv")
worst_24h_periods.to_csv("latest_worst_24h_periods.csv")
acf_df.to_csv("latest_residual_autocorrelation.csv", index=False)
stability_summary.to_csv("latest_stability_summary.csv", index=False)

print("\nSaved files:")
print("latest_model_stability_full_results.csv")
print("latest_error_by_hour.csv")
print("latest_error_by_day.csv")
print("latest_daily_stability_metrics.csv")
print("latest_weekly_stability_metrics.csv")
print("latest_worst_24h_periods.csv")
print("latest_residual_autocorrelation.csv")
print("latest_stability_summary.csv")

# final leakage check

In [ ]:
[col for col in X_hybrid_train.columns if col == "CO(GT)"]